In [ ]:
# '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Bibliotecas ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
from dash import html, dcc, dash_table, Dash, Input, Output, State, no_update       # Framework para criar dashboards interativos...
from matplotlib.colors import ListedColormap                                        # Importando as cores dos gráficos...
import matplotlib.patheffects as patheffects                                        # Alerta visual do "Termômetro-gauge" (efeitos de contorno/sombra em textos/linhas)...
from plotly.subplots import make_subplots                                           # Cria figuras Plotly com múltiplos subplots (linhas/colunas, eixos compartilhados, etc.)...
from dash.exceptions import PreventUpdate                                           # Quando nenhuma saída deve mudar e você não quer executar nada depois...
from matplotlib.patches import Rectangle                                            # Usando para construir o Termômetro...
import matplotlib.gridspec as gridspec                                              # Layouts flexíveis de subplots...
from matplotlib.lines import Line2D                                                 # Fazer a Linha Pontilhada do gráfico de Tendência...
from collections import defaultdict                                                 # Permite criar dicionários com valores padrão automáticos...
import plotly.graph_objects as go                                                   # Criação de gráficos interativos com Plotly...
from dash import callback_context                                                   # Resetando os filtros no Relatório Geral...
from urllib.parse import parse_qs                                                   # Interpreta/parsa parâmetros da URL (query string) em dicionário...
import matplotlib.pyplot as plt                                                     # Criação de gráficos...
from flask_caching import Cache                                                     # Aumentando a velocidade do aplicativo (cache de callbacks/dados para evitar recomputar)...
from datetime import datetime, timedelta                                            # Manipulação de datas e intervalos...
from collections import deque                                                       # É uma fila dupla: inserir/remover itens em ambas as pontas (útil p/ janelas móveis)...
import matplotlib as mpl                                                            # Formatando a biblioteca para deixar padronizada com demais variáveis...
import datetime as dt                                                               # Datas e horários (alias)...
import seaborn as sns                                                               # Visualização de dados estatísticos...
import pandas as pd                                                                 # Análise e manipulação de dados...
import numpy as np                                                                  # Cálculos numéricos e arrays...
import unicodedata                                                                  # Normalização de texto Unicode (ex.: remover acentos, padronizar caracteres)...
import warnings                                                                     # Controle de avisos...
import base64                                                                       # Transformando o Relatório Específico em uma imagem...
import locale                                                                       # Recebendo localidade...
import dash                                                                         # Recebendo a biblioteca dash...
import time                                                                         # Visualizar se foi clicado no primeiro gráfico do Relatório Geral...
import json                                                                         # Serialização segura para sanitização...
import math                                                                         # Utilização de cálculos matemáticos...
import io                                                                           # Manipulação de streams em memória (BytesIO/StringIO p/ imagens/arquivos sem salvar em disco)...
import re                                                                           # Trabalhar com expressões regulares (regex) em Python...
import sqlite3                                                                      # Conexão com o DW (SQLite)...
from pathlib import Path                                                            # Caminhos portáveis...

warnings.filterwarnings("ignore", category=FutureWarning)                           # Oculta avisos de futura depreciação...
locale.setlocale(locale.LC_TIME, "pt_BR.utf8")                                      # Inserindo a localização para automatização do mês...
pd.set_option('mode.chained_assignment', None)                                      # Desativa avisos de atribuição encadeada...


# '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Base de Dados (DW / SQLite) ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
# Origem ÚNICA de dados: DW em SQLite (sem dependência de arquivos externos de fatos).
# ------------------------------------------------------------
# Base path robusto (script vs notebook).
# Em notebooks, '__file__' não existe; caímos em Path.cwd().
# ------------------------------------------------------------
BASE_DIR = Path.cwd()  # notebook .ipynb
SQLITE_PATH = (BASE_DIR / ".." / "project" / "database" / "dw_ronda.sqlite").resolve()

def find_dw_sqlite(start_dir: Path) -> Path:
    """
    Procura por project/database/dw_ronda.sqlite subindo a árvore de diretórios.
    """
    for p in [start_dir, *start_dir.parents]:
        cand = p / "project" / "database" / "dw_ronda.sqlite"
        if cand.exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Não encontrei 'project/database/dw_ronda.sqlite' subindo a partir de:\n"
        f"{start_dir}\n"
        "Dica: confira o CWD do notebook com Path.cwd() ou ajuste BASE_DIR manualmente."
    )

# (Opcional) garante que as views existam no DW.
# Deixe True para o app auto-criar/atualizar as views no início.
APPLY_DW_VIEWS_ON_STARTUP = True

DW_VIEWS_SQL = r'''
-- ============================================================
-- Views auxiliares para o DW Ronda (dw_ronda.sqlite)
-- Fonte única: STAR SCHEMA + vw_evento_acesso_flat
-- Objetivo: facilitar consumo do app com contratos “magros”
-- ============================================================
PRAGMA foreign_keys = ON;

-- =========================
-- 1) Dimensão "pronta" p/ app (alias PT-BR)
-- =========================
DROP VIEW IF EXISTS vw_colaboradores;

CREATE VIEW vw_colaboradores AS
SELECT
  c.cadastro                         AS "Cadastro",
  c.nome                             AS "Nome",
  c.cargo                            AS "Cargo",
  c.cargo_especifico                 AS "Cargo Especifico",
  c.area                             AS "Área",
  c.departamento                     AS "Departamento",
  c.heranca                          AS "Heranca",
  c.desligado                        AS "Desligado"
FROM dim_colaborador c;

-- =========================
-- 2) Eventos válidos para "presença" (equivalência ETL antigo)
--    Regras (do ETL):
--      - Planta != 999
--      - Tipo  != 0
-- =========================
DROP VIEW IF EXISTS vw_evento_acesso_validos_presenca;

CREATE VIEW vw_evento_acesso_validos_presenca AS
SELECT
  cadastro,
  data,
  hora,
  planta,
  tipo
FROM vw_evento_acesso_flat
WHERE
  -- planta e tipo são TEXT no DW; comparação robusta
  COALESCE(TRIM(planta), '') <> '' AND TRIM(planta) <> '999'
  AND COALESCE(TRIM(tipo), '') <> '' AND TRIM(tipo) <> '0'
  AND COALESCE(TRIM(data), '') <> '' AND COALESCE(TRIM(hora), '') <> '';

-- =========================
-- 3) Extremos do dia (primeiro e último evento por Cadastro+Data)
-- =========================
DROP VIEW IF EXISTS vw_presenca_extremos_dia;

CREATE VIEW vw_presenca_extremos_dia AS
SELECT
  cadastro,
  data,
  hora,
  planta
FROM (
  SELECT
    cadastro,
    data,
    hora,
    planta,
    MIN(hora) OVER (PARTITION BY cadastro, data) AS hora_min,
    MAX(hora) OVER (PARTITION BY cadastro, data) AS hora_max
  FROM vw_evento_acesso_validos_presenca
)
WHERE hora = hora_min OR hora = hora_max;

-- =========================
-- 4) df_diarios (contrato do app)
--    colunas: Cadastro, Planta, Data
-- =========================
DROP VIEW IF EXISTS vw_df_diarios;

CREATE VIEW vw_df_diarios AS
SELECT DISTINCT
  cadastro              AS "Cadastro",
  planta                AS "Planta",
  date(data)            AS "Data"
FROM vw_presenca_extremos_dia;

-- =========================
-- 5) df_registros_semanais (contrato do app)
--    colunas: Cadastro, Ano, Semana por Ano, Data, Contagem Presença
--    Semana ISO (equivalente ao pandas .dt.isocalendar):
--      - iso_anchor = quinta-feira da semana ISO: date(d, '-3 days', 'weekday 4')
--      - iso_year   = strftime('%Y', iso_anchor)
--      - iso_week   = strftime('%W', iso_anchor) + 1
--      - Data       = segunda-feira da semana ISO: date(iso_anchor, '-3 days')
-- =========================
DROP VIEW IF EXISTS vw_df_registros_semanais;

CREATE VIEW vw_df_registros_semanais AS
WITH base AS (
  SELECT
    "Cadastro",
    "Planta",
    "Data" AS data_dia,
    date("Data", '-3 days', 'weekday 4') AS iso_anchor
  FROM vw_df_diarios
)
SELECT
  "Cadastro"                                                AS "Cadastro",
  CAST(strftime('%Y', iso_anchor) AS INTEGER)               AS "Ano",
  CAST(strftime('%W', iso_anchor) AS INTEGER) + 1           AS "Semana por Ano",
  date(iso_anchor, '-3 days')                               AS "Data",
  COUNT(DISTINCT data_dia)                                  AS "Contagem Presença"
FROM base
GROUP BY
  "Cadastro",
  CAST(strftime('%Y', iso_anchor) AS INTEGER),
  CAST(strftime('%W', iso_anchor) AS INTEGER) + 1,
  date(iso_anchor, '-3 days');

-- =========================
-- 6) df_almoco (contrato do app)
--    colunas: Cadastro, Data, Hora, Dir, Planta, Coletor, Descricao Coletor
--    Regras (do ETL):
--      (A) Refeitório:
--          Dir='E' AND coletor contém 'REF' AND não contém 'GRILL'/'GRIL'
--      (B) Portaria:
--          Dir='S' AND coletor contém 'PORT' AND hora entre 11:00 e 14:00
-- =========================
DROP VIEW IF EXISTS vw_df_almoco;

CREATE VIEW vw_df_almoco AS
SELECT
  cadastro                         AS "Cadastro",
  date(data)                       AS "Data",
  hora                             AS "Hora",
  dir                              AS "Dir",
  planta                           AS "Planta",
  coletor                          AS "Coletor",
  descricao_coletor                AS "Descricao Coletor"
FROM vw_evento_acesso_flat
WHERE
  COALESCE(TRIM(data), '') <> '' AND COALESCE(TRIM(hora), '') <> ''
  AND (
    (
      UPPER(COALESCE(dir,'')) = 'E'
      AND UPPER(COALESCE(coletor,'')) LIKE '%REF%'
      AND UPPER(COALESCE(coletor,'')) NOT LIKE '%GRILL%'
      AND UPPER(COALESCE(coletor,'')) NOT LIKE '%GRIL%'
    )
    OR
    (
      UPPER(COALESCE(dir,'')) = 'S'
      AND UPPER(COALESCE(coletor,'')) LIKE '%PORT%'
      AND hora >= '11:00:00' AND hora <= '14:00:00'
    )
  );

-- ============================================================
-- 7) Views “placeholder” (para o app não quebrar hoje)
--    Você pode substituir por views reais assim que existir fato no DW.
-- ============================================================

-- GPTW (exemplo: mantenha no DW em tabela/vw própria quando tiver)
DROP VIEW IF EXISTS vw_gptw;
CREATE VIEW vw_gptw AS
SELECT
  CAST(NULL AS TEXT)    AS "Área",
  CAST(NULL AS INTEGER) AS "2025",
  CAST(NULL AS INTEGER) AS "2024",
  CAST(NULL AS INTEGER) AS "2023"
WHERE 0;

-- Movimentações (RH) - placeholder com contrato de colunas
DROP VIEW IF EXISTS vw_movimentacoes;
CREATE VIEW vw_movimentacoes AS
SELECT
  CAST(NULL AS TEXT) AS "Cadastro",
  CAST(NULL AS TEXT) AS "Abreviação UO antiga",
  CAST(NULL AS TEXT) AS "Desc. UO antiga",
  CAST(NULL AS TEXT) AS "Desc. Cargo antigo",
  CAST(NULL AS TEXT) AS "Abreviação UO",
  CAST(NULL AS TEXT) AS "Desc. UO",
  CAST(NULL AS TEXT) AS "Desc. Cargo",
  CAST(NULL AS TEXT) AS "Tipo da Movimentação",
  CAST(NULL AS TEXT) AS "Data de alteração"
WHERE 0;

-- Headcount (snapshot por ano) - placeholder
DROP VIEW IF EXISTS vw_headcount;
CREATE VIEW vw_headcount AS
SELECT
  CAST(NULL AS TEXT)    AS "Área",
  CAST(NULL AS INTEGER) AS "2023",
  CAST(NULL AS INTEGER) AS "2024",
  CAST(NULL AS INTEGER) AS "2025"
WHERE 0;
'''

# Bloco opcional de checks/asserts
ENABLE_SANITY_CHECKS = False


def _dw_connect(sqlite_path: Path) -> sqlite3.Connection:
    if not sqlite_path.exists():
        raise FileNotFoundError(f"DW SQLite não encontrado em: {sqlite_path}")
    conn = sqlite3.connect(str(sqlite_path))
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


def _ensure_dw_views(conn: sqlite3.Connection) -> None:
    if not APPLY_DW_VIEWS_ON_STARTUP:
        return
    if DW_VIEWS_SQL.strip():
        conn.executescript(DW_VIEWS_SQL)


def _load_view(conn: sqlite3.Connection, view_name: str) -> pd.DataFrame:
    return pd.read_sql_query(f"SELECT * FROM {view_name}", conn)


def load_dw_data(sqlite_path: Path):
    conn = _dw_connect(sqlite_path)
    try:
        _ensure_dw_views(conn)

        df_colaboradores = _load_view(conn, "vw_dim_colaboradores")
        df_diarios = _load_view(conn, "vw_df_diarios")
        df_registros_semanais = _load_view(conn, "vw_df_registros_semanais")
        df_almoco = _load_view(conn, "vw_df_almoco")

        # Apoios (podem ser placeholders no DW; mantemos para não quebrar UI)
        df_headcount = _load_view(conn, "vw_headcount")
        df_movimentacoes = _load_view(conn, "vw_movimentacoes")
        df_gptw = _load_view(conn, "vw_gptw")

    finally:
        conn.close()

    # Tipos obrigatórios
    for _df in (df_colaboradores, df_diarios, df_registros_semanais, df_almoco, df_headcount, df_movimentacoes, df_gptw):
        if "Cadastro" in _df.columns:
            _df["Cadastro"] = _df["Cadastro"].astype("string")

    # Datas
    if "Data" in df_diarios.columns:
        df_diarios["Data"] = pd.to_datetime(df_diarios["Data"], errors="coerce")
    if "Data" in df_registros_semanais.columns:
        df_registros_semanais["Data"] = pd.to_datetime(df_registros_semanais["Data"], errors="coerce")
    if "Data" in df_almoco.columns:
        df_almoco["Data"] = pd.to_datetime(df_almoco["Data"], errors="coerce")

    if "Data de alteração" in df_movimentacoes.columns:
        df_movimentacoes["Data de alteração"] = pd.to_datetime(df_movimentacoes["Data de alteração"], errors="coerce")

    # Desligado -> 'Sim'/'Não' (aceita 0/1, True/False, 'Sim'/'Não')
    if "Desligado" in df_colaboradores.columns:
        s = df_colaboradores["Desligado"].astype("string").str.strip().str.lower()
        df_colaboradores["Desligado"] = np.where(
            s.isin(["1", "true", "sim", "yes", "y", "t"]),
            "Sim",
            "Não"
        )


    # Garantir coluna com acento (aceita coluna sem acento vinda de fontes legadas)
    col_area_plain = "A" + "rea"
    if "Área" not in df_colaboradores.columns and col_area_plain in df_colaboradores.columns:
        df_colaboradores.rename(columns={col_area_plain: "Área"}, inplace=True)
    # Derivados para manter contrato atual do app
    if "Desligado" in df_colaboradores.columns:
        df_ativos = df_colaboradores[df_colaboradores["Desligado"] != "Sim"].copy()
        df_desligados = df_colaboradores[df_colaboradores["Desligado"] == "Sim"].copy()
    else:
        df_ativos = df_colaboradores.copy()
        df_desligados = df_colaboradores.iloc[0:0].copy()

    # Mantém colunas opcionais usadas em trechos legados do app
    for c in ("Data_Adm", "Data_Dem"):
        if c not in df_desligados.columns:
            df_desligados[c] = pd.NaT

    # Enriquecer registros semanais só com as colunas necessárias para UI (sem replicar dimensão inteira)
    cols_dim_ui = [c for c in ["Cadastro", "Nome", "Cargo", "Cargo Especifico", "Área", "Departamento", "Heranca", "Desligado"] if c in df_colaboradores.columns]
    dim_ui = df_colaboradores[cols_dim_ui].drop_duplicates(subset=["Cadastro"])

    df_semana_merged = df_registros_semanais.merge(dim_ui, on="Cadastro", how="left", validate="m:1")

    # Compatibilidade: o app usa df_datas como base diária
    df_datas = df_diarios

    # Preenche colunas esperadas no df_semana_merged, se necessário
    if "Dia Semana" not in df_semana_merged.columns and "Data" in df_semana_merged.columns:
        df_semana_merged["Dia Semana"] = df_semana_merged["Data"].dt.day_name()

    if ENABLE_SANITY_CHECKS:
        # Anti-uso de formatos proibidos (busca no próprio arquivo; evita strings diretas)
        code = Path(__file__).read_text(encoding="utf-8", errors="ignore").lower()
        banned = [
            "read_" + "par" + "quet",
            "fast" + "par" + "quet",
            "pyarrow" + "." + "par" + "quet",
            "." + "par" + "quet",
        ]
        found = [b for b in banned if b in code]
        if found:
            raise RuntimeError(f"Uso proibido detectado no código: {found}")

        assert df_colaboradores["Cadastro"].is_unique, "Cadastro não é único em df_colaboradores"

        print("[SANITY] shapes:")
        print("  df_diarios:", df_diarios.shape, list(df_diarios.columns))
        print("  df_registros_semanais:", df_registros_semanais.shape, list(df_registros_semanais.columns))
        print("  df_almoco:", df_almoco.shape, list(df_almoco.columns))

        # Soma de presenças semanais
        if "Contagem Presença" in df_registros_semanais.columns:
            print("[SANITY] soma presença semanal:", int(df_registros_semanais["Contagem Presença"].fillna(0).sum()))

        # Pessoas únicas por Área/Planta no período
        if "Área" in df_colaboradores.columns:
            print("[SANITY] pessoas únicas por Área (top 5):")
            print(df_colaboradores.groupby("Área")["Cadastro"].nunique().sort_values(ascending=False).head(5))

        if "Planta" in df_diarios.columns:
            print("[SANITY] pessoas únicas por Planta (top 5):")
            print(df_diarios.groupby("Planta")["Cadastro"].nunique().sort_values(ascending=False).head(5))

        if not df_almoco.empty:
            print("[SANITY] eventos almoço:", len(df_almoco))

    return df_ativos, df_semana_merged, df_datas, df_headcount, df_desligados, df_movimentacoes, df_gptw, df_almoco, df_colaboradores, df_registros_semanais, df_diarios


# '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Variáveis ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
# Carrega os DataFrames do DW (SQLite):
(df_ativos, df_semana_merged, df_datas, df_headcount, df_desligados, df_movimentacoes, df_gptw, df_almoco,
 df_colaboradores, df_registros_semanais, df_diarios) = load_dw_data(SQLITE_PATH)


# Normalizações importantes:
for _df in (df_semana_merged, df_datas, df_desligados, df_ativos):
    if 'Cadastro' in _df.columns:
        _df['Cadastro'] = _df['Cadastro'].astype(str)


# Datas:
if 'Data' in df_semana_merged.columns:
    df_semana_merged['Data'] = pd.to_datetime(df_semana_merged['Data'], errors='coerce')
if 'Data' in df_datas.columns:
    df_datas['Data'] = pd.to_datetime(df_datas['Data'], errors='coerce')
 

# Gera "Dia Semana" (se não existir) a partir de Data:
if 'Data' in df_semana_merged.columns and 'Dia Semana' not in df_semana_merged.columns:
    _dias_pt = {0:'Segunda', 1:'Terça', 2:'Quarta', 3:'Quinta', 4:'Sexta', 5:'Sábado', 6:'Domingo'}
    df_semana_merged['Dia Semana'] = df_semana_merged['Data'].dt.weekday.map(_dias_pt)


# E, por via das dúvidas, normalize o df_datas também:
if 'Data' in df_datas.columns:
    df_datas['Data'] = pd.to_datetime(df_datas['Data'], errors='coerce')


# >>> FIX: garantir coluna 'Cargo' quando ausente (usa Cargo Especifico):
if 'Cargo' not in df_semana_merged.columns and 'Cargo Especifico' in df_semana_merged.columns:
    df_semana_merged = df_semana_merged.assign(Cargo=df_semana_merged['Cargo Especifico'])


# Constantes visuais e mapas:
estilo_oculto = {'display': 'none'}
estilo_exibir = {'display': 'block', 'padding': '20px', 'backgroundColor': 'white'}
ordenacao_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']
dias_portugues = {'Monday': 'Segunda', 'Tuesday': 'Terça', 'Wednesday': 'Quarta', 'Thursday': 'Quinta',
                  'Friday': 'Sexta', 'Saturday': 'Sábado', 'Sunday': 'Domingo'}



# Helper: torna valores "hashable" para groupby/drop_duplicates (evita erro: unhashable type: 'numpy.ndarray')
def _safe_hashable(v):
    """Converte estruturas (np.ndarray/list/dict/...) em um escalar/string estável para uso em chaves."""
    try:
        if isinstance(v, np.ndarray):
            try:
                if v.shape == () or v.size == 1:
                    return v.item()
            except Exception:
                pass
            return "|".join(map(str, np.ravel(v).tolist()))
    except Exception:
        pass

    if isinstance(v, (list, tuple, set, frozenset)):
        try:
            if len(v) == 1:
                return next(iter(v))
        except Exception:
            pass
        try:
            return "|".join(map(str, list(v)))
        except Exception:
            return str(v)

    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True)
        except Exception:
            return str(v)

    return v


# =================================================================================================
# Helpers (Plotly/Dash): sanitização + subplots polares
# =================================================================================================
def _sanitize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Sanitiza colunas dtype=object para evitar 'unhashable' (ex.: np.ndarray/list/set) em groupby/plotly.

    - Colunas de local (ex.: 'Planta' / 'Cluster'): converte para frozenset (hashable), preservando semântica.
    - Demais colunas object: aplica _safe_hashable (conversão estável em escalar/string).
    """
    if df is None or getattr(df, "empty", True):
        return df

    d = df.copy()

    special_cols = {"Planta", "Plantas", "Cluster", "Clusters"}

    def _to_frozenset(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return frozenset()

        # já é hashable
        if isinstance(x, frozenset):
            return x

        # coleções
        if isinstance(x, set):
            return frozenset([str(t).strip().lower() for t in x if str(t).strip()])

        if isinstance(x, (list, tuple)):
            return frozenset([str(t).strip().lower() for t in x if str(t).strip()])

        if isinstance(x, (np.ndarray, pd.Series)):
            try:
                arr = np.ravel(x).tolist()
            except Exception:
                arr = list(x)
            return frozenset([str(t).strip().lower() for t in arr if str(t).strip()])

        # string com separadores
        s = str(x).strip()
        if not s:
            return frozenset()

        for sep in ["|", ";", ",", "/", "\\"]:
            if sep in s:
                parts = [p.strip().lower() for p in s.split(sep)]
                return frozenset([p for p in parts if p])

        return frozenset([s.lower()])

    for col in d.columns:
        if d[col].dtype == "object":
            if col in special_cols:
                d[col] = d[col].map(_to_frozenset)
            else:
                d[col] = d[col].map(_safe_hashable)

    return d


def _polar_key(i: int) -> str:
    """Chave do subplot polar no layout do Plotly."""
    i = int(i)
    return "polar" if i == 1 else f"polar{i}"


def _ajustar_domains_subplots_polares(
    fig: go.Figure,
    rows: int,
    cols: int,
    n: int,
    pad_x: float = 0.02,
    pad_y: float = 0.05
) -> go.Figure:
    """Ajuste leve de domains para subplots polares.

    Observação: o make_subplots já define domains. Este helper existe principalmente para:
    - evitar erro quando o código tenta ajustar domains e a chave ainda não existe
    - permitir um ajuste “soft” no futuro sem espalhar lógica pelo código
    """
    try:
        for i in range(1, int(n) + 1):
            key = _polar_key(i)
            if key not in fig.layout:
                continue
            # se não houver domain (raro), define algo padrão
            if getattr(fig.layout[key], "domain", None) is None:
                fig.layout[key].domain = dict(x=[0, 1], y=[0, 1])
    except Exception:
        pass
    return fig


# Férias coletivas (ex.: ajustar conforme sua regra):
ferias = [
    (datetime(2023, 12, 18), datetime(2024, 1, 7)),
    (datetime(2024, 12, 23), datetime(2025, 1, 12)),
]


def esta_em_ferias(data):
    for inicio, fim in ferias:
        if inicio <= data <= fim:
            return True
    return False


# Segundas-feiras únicas (início da semana ISO) a partir de df_semana_merged:
if 'Data' in df_semana_merged.columns:
    seg_period = df_semana_merged['Data'].dt.to_period('W').dropna().unique()
    segundas_feiras = [p.start_time for p in seg_period if p.start_time.weekday() == 0]
else:
    segundas_feiras = []


# Remove semanas de férias e semana 1 do ano (ISO week):
todas_semanas = [
    s for s in sorted(segundas_feiras)
    if not esta_em_ferias(s) and (getattr(s.isocalendar(), 'week', s.isocalendar()[1]) != 1)
]


# Count por dia da semana (para heatmap específico) — usando df_semana_merged:
if 'Dia Semana' in df_semana_merged.columns:
    df_count_por_semana = (
        df_semana_merged
        .groupby(['Cadastro', 'Nome', 'Área', 'Departamento', 'Heranca', 'Cargo Especifico', 'Desligado'])['Dia Semana']
        .value_counts()
        .reset_index()
    )
    df_count_por_semana.columns = [
        'Cadastro', 'Nome', 'Área', 'Departamento', 'Heranca',
        'Cargo Especifico', 'Desligado', 'Dia Semana', 'Contagem Presença'
    ]
else:
    # fallback vazio se não existir a coluna:
    df_count_por_semana = pd.DataFrame(columns=['Cadastro','Nome','Área','Departamento','Heranca','Cargo Especifico','Desligado','Dia Semana','Contagem Presença'])


# ======== Range de meses global ========
LT_INICIO_ABS = dt.date(2024, 1, 1)
LT_FIM_ABS = dt.date.today()


# Gera uma sequência mensal (1º dia de cada mês):
LT_MONTHS = pd.date_range(LT_INICIO_ABS, LT_FIM_ABS, freq="MS").date


def lt_idx_of(y, m):
    """Retorna o índice do mês/ano dentro de LT_MONTHS"""
    for i, d in enumerate(LT_MONTHS):
        if d.year == y and d.month == m:
            return i
    raise ValueError(f"Mês {m}/{y} não encontrado em LT_MONTHS")


# ======== NOVOS CENÁRIOS ========
# Cenário 1 ⭢ Jan/2024 até Abril/2025:
LT_C1_MIN = lt_idx_of(2024, 1)
LT_C1_MAX = lt_idx_of(2025, 4)


# Cenário 2 ⭢ Maio/2025 até o último mês disponível (dinâmico):
LT_C2_MIN = lt_idx_of(2025, 5)
LT_C2_MAX = len(LT_MONTHS) - 1


# Marcas trimestrais no slider:
def lt_label(d): return d.strftime("%b/%Y").lower()
LT_MARKS = {
    i: (lt_label(d) if d.month in [1, 4, 7, 10] else "")
    for i, d in enumerate(LT_MONTHS)
}


# Exibição trimestral (para o slider):
def lt_label(d): return d.strftime("%b/%Y").lower()
LT_MARKS = {i: (lt_label(d) if d.month in [1, 4, 7, 10] else "") for i, d in enumerate(LT_MONTHS)}


ADM_CACHE = None
DEM_CACHE = None


# =========================
# Fonte padrão (Volkswagen)
# =========================
VW_FONT_FAMILY = "'VW Text', Arial, sans-serif"
VW_PLOTLY_FONT = "VW Text"

# Matplotlib/Seaborn (quando houver figuras renderizadas como imagem)
mpl.rcParams["font.family"] = "VW Text"

# CSS global para forçar a fonte em todo o app (inclui textos de componentes Dash)
GLOBAL_CSS = f"""
@font-face {{
  font-family: 'VW Text';
  src: local('VW Text');
}}
* {{
  font-family: {VW_FONT_FAMILY} !important;
}}
"""


# São áreas que normalmente não são analisadas para esses insights:
areas_fora = ['B-SNP', 'B-V']


# --- estilos básicos para fallbacks de tabelas (evita NameError) ---
PAD_STY = {
    'table':    {'borderCollapse': 'collapse', 'width': '100%', 'fontFamily': 'VW Text'},
    'th':       {'padding': '6px 10px', 'background': '#f0f0f0', 'border': '1px solid #ccc', 'fontWeight': 'bold', 'textAlign': 'center', 'fontFamily': 'VW Text'},
    'td':       {'padding': '6px 10px', 'border': '1px solid #eee', 'textAlign': 'center', 'fontFamily': 'VW Text'},
    'td_first': {'padding': '6px 10px', 'border': '1px solid #eee', 'textAlign': 'left', 'fontWeight': 'bold', 'fontFamily': 'VW Text'}
}


TOGGLE_STY = {
    'wrap': {'display':'flex','alignItems':'center','gap':'12px','flexWrap':'nowrap'},
    'btn_base': {
        'position':'relative',
        'width':'92px',
        'height':'42px',
        'borderRadius':'999px',
        'border':'1px solid rgba(0,0,0,.14)',
        'background':'linear-gradient(#f7f7f7,#e7e7e7)',
        'boxShadow':'0 1px 2px rgba(0,0,0,.12) inset, 0 1px 2px rgba(0,0,0,.05)',
        'cursor':'pointer',
        'transition':'all .18s ease-in-out',
        'padding':'0',
        'fontFamily':'VW Text'
    },
    'btn_on': {
        'background':'linear-gradient(#181818,#2a2a2a)',
        'border':'1px solid rgba(255,255,255,.12)'
    },
    'inner': {
        'position':'absolute',
        'inset':'0',
        'display':'flex',
        'alignItems':'center',
        'justifyContent':'space-between',
        'padding':'0 12px',
        'userSelect':'none',
        'fontFamily':'VW Text'
    },
    'ico_base': {'fontSize':'18px','lineHeight':'1','transition':'opacity .18s ease, color .18s ease'},
    'ico_heat_color': {'color':'#0057b8'},
    'ico_net_color':  {'color':'#00a37a'},
    'ico_dim': {'opacity':0.28},
    'ico_on_color': {'color':'#f3f3f3'},
    'knob_base': {
        'position':'absolute',
        'top':'3px',
        'left':'3px',
        'width':'36px',
        'height':'36px',
        'borderRadius':'50%',
        'background':'#ffffff',
        'boxShadow':'0 1px 2px rgba(0,0,0,.28), 0 0 0 1px rgba(0,0,0,.06) inset',
        'transform':'translateX(0)',
        'transition':'transform .18s ease-in-out, background .18s ease'
    },
    'knob_on': {
        'transform':'translateX(50px)',
        'background':'#1e1e1e',
        'boxShadow':'0 1px 2px rgba(0,0,0,.28), 0 0 0 1px rgba(255,255,255,.08) inset'
    },
    'knob_ico_base': {
        'position':'absolute',
        'inset':'0',
        'display':'flex',
        'alignItems':'center',
        'justifyContent':'center',
        'fontSize':'20px',
        'fontFamily':'VW Text',
        'transform':'translateX(-25px)',
        'transition':'transform .18s ease-in-out, color .18s ease-in-out'
    },
    'label': {'display':'none'}
}


ARROW_UP, ARROW_DOWN, ARROW_NONE = '▲', '▼', '—'


# '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Funções utilitárias ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Geral ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
def aplicar_filtro_faixas(df, coluna, faixas):
    """Mantém linhas cujo valor em 'coluna' cai em alguma das faixas tipo '0 a 1'."""
    if not faixas or not all(" a " in f for f in faixas):
        return df
    return df[df[coluna].apply(lambda x: esta_em_alguma_faixa(x, faixas))]


def aplicar_filtro_por_area(df, area, subarea, her_df):
    """Filtra por área/subárea respeitando a herança de ativos (alinha com o original)."""
    if area and area != 'GERAL':
        if subarea:
            deps = her_df[her_df['Área'] == subarea]['Heranca'].unique()
            return df[(df['Área'] == subarea) & (df['Heranca'].isin(deps))]
        else:
            deps = her_df[her_df['Área'] == area]['Heranca'].unique()
            return df[df['Área'].str.startswith(area) & df['Heranca'].isin(deps)]
    return df


def _normaliza_str(v):
    s = str(v).strip()
    return s if s.lower() not in ('nan', 'none') else ''


def _startswith_any(val, prefixes):
    val = _normaliza_str(val)
    if not val or not prefixes:
        return False
    for p in prefixes:
        p = _normaliza_str(p)
        if p and (val == p or val.startswith(p)):
            return True
    return False


def resolver_areas_viajantes(her_df: pd.DataFrame, areas_base=None):
    """
    Retorna (areas_prefixos, herancas_prefixos) para filtrar 'áreas que viajam'
    respeitando herança.
    
    areas_base: lista de prefixos/áreas-macro consideradas 'viajantes'
    """
    if areas_base is None:
        areas_base = areas_fora  # ajuste aqui se necessário

    areas_base = [ _normaliza_str(a) for a in areas_base if _normaliza_str(a) ]

    if her_df is None or her_df.empty or not {'Área','Heranca'}.issubset(her_df.columns):
        # Sem tabela de herança: ainda assim devolve apenas as áreas base
        return set(areas_base), set()

    tmp = her_df[['Área','Heranca']].copy()
    tmp['Área'] = tmp['Área'].astype(str).str.strip()
    tmp['Heranca'] = tmp['Heranca'].astype(str).str.strip()

    # heranças ligadas às áreas base (match por prefixo)
    herancas_base = (
        tmp.loc[tmp['Área'].apply(lambda a: _startswith_any(a, areas_base)), 'Heranca']
        .dropna().unique().tolist()
    )
    herancas_base = [ _normaliza_str(h) for h in herancas_base if _normaliza_str(h) ]

    # conjunto final de prefixos de área e de herança que definem 'viajantes'
    areas_prefixos = set(areas_base)
    herancas_prefixos = set(herancas_base)

    return areas_prefixos, herancas_prefixos


def esta_em_alguma_faixa(valor, faixas):
    for faixa in faixas:
        try:
            fmin, fmax = map(float, faixa.split(" a "))
            if fmin <= float(valor) < fmax:
                return True
        except Exception:
            continue
    return False


def interpolar_cor_hex(cor1, cor2, t):
    c1 = [int(cor1[i:i + 2], 16) for i in (1, 3, 5)]
    c2 = [int(cor2[i:i + 2], 16) for i in (1, 3, 5)]
    interp = [int(c1[j] + (c2[j] - c1[j]) * t) for j in range(3)]
    return f"#{interp[0]:02x}{interp[1]:02x}{interp[2]:02x}"


def normalizar_str(s):
    return str(s).strip().replace("_", " ")


def slug(s):
    return str(s).replace(" ", "_").replace("/", "-")


def periodo_limites(indices):
    """Retorna os limites reais de datas a partir dos índices do slider"""
    i0, i1 = indices
    i0 = max(0, min(i0, len(LT_MONTHS) - 1))
    i1 = max(0, min(i1, len(LT_MONTHS) - 1))
    dt_ini = pd.to_datetime(LT_MONTHS[i0]).replace(day=1)
    dt_fim = (pd.to_datetime(LT_MONTHS[i1]) + pd.offsets.MonthEnd(0))
    return dt_ini, dt_fim


def filtrar_por_periodo(df, dt_ini, dt_fim):
    if df is None or df.empty:
        return df

    dt_ini = pd.Timestamp(dt_ini).normalize()
    dt_fim = pd.Timestamp(dt_fim).normalize()
    if dt_ini > dt_fim:
        dt_ini, dt_fim = dt_fim, dt_ini

    dff = df.copy()

    if 'Data' in dff.columns:
        dff['Data'] = pd.to_datetime(dff['Data'], errors='coerce')
        if pd.api.types.is_datetime64tz_dtype(dff['Data']):
            dff['Data'] = dff['Data'].dt.tz_localize(None)
        return dff[dff['Data'].between(dt_ini, dt_fim, inclusive='both')]
    
    if 'Início da Semana' in dff.columns:
        dff['_IniSem'] = pd.to_datetime(dff['Início da Semana'], errors='coerce')
        if pd.api.types.is_datetime64tz_dtype(dff['_IniSem']):
            dff['_IniSem'] = dff['_IniSem'].dt.tz_localize(None)
        out = dff[dff['_IniSem'].between(dt_ini, dt_fim, inclusive='both')].drop(columns=['_IniSem'])
        return out
    
    if {'Ano', 'Semana por Ano'}.issubset(dff.columns):
        tmp = dff[['Ano', 'Semana por Ano']].copy()
        tmp['Ano'] = pd.to_numeric(tmp['Ano'], errors='coerce')
        tmp['Semana por Ano'] = pd.to_numeric(tmp['Semana por Ano'], errors='coerce')
        mask_valid = tmp['Ano'].notna() & tmp['Semana por Ano'].between(1,53, inclusive='both')
        tmp = tmp[mask_valid]
        inicios = pd.to_datetime(
            tmp['Ano'].astype(int).astype(str) + tmp['Semana por Ano'].astype(int).astype(str).str.zfill(2) + '1',
            format='%G%V%u', errors='coerce'
        )
        dff = dff.loc[tmp.index].copy()
        dff['_IniSem'] = inicios
        out = dff[dff['_IniSem'].between(dt_ini, dt_fim, inclusive='both')].drop(columns=['_IniSem'])
        return out

    return dff


def aplicar_tema_pb(fig, *, largura_borda=2, so_y_boxplot=False, grade=False, clean=False):
    """
    Tema PB:
    - Fundo branco
    - Bordas pretas
    - Linhas pretas
    se so_y_boxplot=True: apenas eixo Y com grid,
    mas mantém a borda completa em volta do gráfico
    """
    fig.update_layout(
        template=None,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=60, r=30, t=60, b=50),
        font=dict(family=VW_PLOTLY_FONT, color="#000000"),
        hoverlabel=dict(font=dict(family=VW_PLOTLY_FONT)),
    )

    # Garante fonte VW na legenda sem quebrar configurações já existentes
    try:
        _legend = fig.layout.legend.to_plotly_json() if fig.layout.legend else {}
        _legend_font = (_legend.get("font") or {})
        _legend_font["family"] = VW_PLOTLY_FONT
        _legend["font"] = _legend_font
        fig.update_layout(legend=_legend)
    except Exception:
        pass


    if so_y_boxplot:
        # Borda preta em X e Y
        fig.update_xaxes(
            showline=True, linewidth=largura_borda, linecolor="black",
            mirror=True, ticks="outside", tickcolor="black",
            zeroline=True, zerolinecolor="black",
            showgrid=False
        )
        fig.update_yaxes(
            showline=True, linewidth=largura_borda, linecolor="black",
            mirror=True, ticks="outside", tickcolor="black",
            zeroline=True, zerolinecolor="black",
            gridcolor="#a7a7a7"   # grid só no Y
        )
    else:
        if grade:
            # Borda preta em X e Y (sem grade)
            fig.update_xaxes(
                showline=True, linewidth=largura_borda, linecolor="black",
                mirror=True, ticks="outside", tickcolor="black",
                zeroline=False,      # tira a linha zero (opcional)
                showgrid=False       # remove grid no X
            )
            fig.update_yaxes(
                showline=True, linewidth=largura_borda, linecolor="black",
                mirror=True, ticks="outside", tickcolor="black",
                zeroline=False,      # tira a linha zero (opcional)
                gridcolor='#a7a7a7',
                griddash="dot"
            )
        else:
            if clean:
                # Borda preta em X e Y (sem grade)
                fig.update_xaxes(
                    showline=True, linewidth=largura_borda, linecolor="black",
                    mirror=True, ticks="outside", tickcolor="black",
                    zeroline=False,      # tira a linha zero (opcional)
                    showgrid=False       # remove grid no X
                )
                fig.update_yaxes(
                    showline=True, linewidth=largura_borda, linecolor="black",
                    mirror=True, ticks="outside", tickcolor="black",
                    zeroline=False,      # tira a linha zero (opcional)
                    showgrid=False       # remove grid no Y
                )
            else:
                # Normal: borda e grid em ambos os eixos
                fig.update_xaxes(
                    showline=True, linewidth=largura_borda, linecolor="black",
                    mirror=True, ticks="outside", tickcolor="black",
                    zeroline=True, zerolinecolor="black",
                    gridcolor="#a7a7a7"
                )
                fig.update_yaxes(
                    showline=True, linewidth=largura_borda, linecolor="black",
                    mirror=True, ticks="outside", tickcolor="black",
                    zeroline=True, zerolinecolor="black",
                    gridcolor="#a7a7a7"
                )
    return fig


def area_col(dff: pd.DataFrame) -> str | None:
    # retorna o nome correto da coluna de área
    if 'Área' in dff.columns:  return 'Área'
    if 'Área' in dff.columns:  return 'Área'
    return None


def tok(s: str) -> list[str]:
    return [t.strip() for t in str(s).replace('/', '-').split('-') if t.strip()]


def akey(a: str):
    t = tok(a)
    return (len(t), tuple(t))


def rel(am, ae) -> bool:
    if pd.isna(am) or pd.isna(ae):
        return False
    sm = str(am).replace('/', '-').lower().strip()
    se = str(ae).replace('/', '-').lower().strip()
    return bool(se) and sm.startswith(se)


def _carregar_adm_dem_cache():
    global ADM_CACHE, DEM_CACHE
    if ADM_CACHE is None or DEM_CACHE is None:
        ADM_CACHE, DEM_CACHE = coletar_admissoes_e_demissoes()


def _sobrepoe_periodo(adm_ts, dem_ts, dt_ini, dt_fim):
    # adm_ts/dem_ts podem ser NaT/None
    if pd.isna(adm_ts) or adm_ts is None:
        adm_ts = pd.Timestamp.min
    if pd.isna(dem_ts) or dem_ts is None:
        dem_ts = pd.Timestamp.max
    # há interseção se início <= fim e fim >= início, etc.
    return (adm_ts <= dt_fim) and (dem_ts >= dt_ini)


def filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas):
    # 1) período
    dt_ini, dt_fim = periodo_limites(periodo_idx)
    df_filtrado = (filtrar_por_periodo(df_semana_merged, dt_ini, dt_fim)).copy()

    # 2) QUADRO ELEGÍVEL: só quem tem vínculo que sobrepõe o período
    _carregar_adm_dem_cache()
    if not df_filtrado.empty:
        # calcula quem é elegível
        cad_unique = df_filtrado['Cadastro'].astype(str).unique()
        elegiveis = []
        for cad in cad_unique:
            adm = ADM_CACHE.get(str(cad), None)
            dem = DEM_CACHE.get(str(cad), None)
            if _sobrepoe_periodo(adm, dem, pd.Timestamp(dt_ini), pd.Timestamp(dt_fim)):
                elegiveis.append(str(cad))
        df_filtrado = df_filtrado[df_filtrado['Cadastro'].astype(str).isin(set(elegiveis))]

    # 3) desligados
    if desligados == 'right' and 'Desligado' in df_filtrado.columns:
        df_filtrado = df_filtrado[df_filtrado['Desligado'] != 'Sim']

    # 4) viajam (respeita herança)
    if viajam == 'right':
        areas_prefixos, herancas_prefixos = resolver_areas_viajantes(her)

        mask_area = False
        mask_her = False

        if 'Área' in df_filtrado.columns:
            mask_area = df_filtrado['Área'].apply(lambda a: _startswith_any(a, areas_prefixos))

        if 'Heranca' in df_filtrado.columns and len(herancas_prefixos) > 0:
            mask_her = df_filtrado['Heranca'].apply(lambda h: _startswith_any(h, herancas_prefixos))

        if isinstance(mask_area, pd.Series) and isinstance(mask_her, pd.Series):
            df_filtrado = df_filtrado[~(mask_area | mask_her)]
        elif isinstance(mask_area, pd.Series):
            df_filtrado = df_filtrado[~mask_area]
        elif isinstance(mask_her, pd.Series):
            df_filtrado = df_filtrado[~mask_her]

    # 5) área/subárea
    df_filtrado = aplicar_filtro_por_area(df_filtrado, area_selecionada, subarea_selecionada, her)

    return df_filtrado


'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Individual ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return f"data:image/png;base64,{encoded}"


def fig_to_bytes(fig, fmt="png"):
    buf = io.BytesIO()
    fig.savefig(buf, format=fmt, bbox_inches="tight")
    buf.seek(0)
    return buf.read()


def obter_periodo_admissao(chapa):
    pessoa = df_desligados[df_desligados['Cadastro'] == str(chapa)]
    if not pessoa.empty:
        adm = pd.to_datetime(pessoa['Data_Adm'].iloc[0], errors='coerce')
        dem = pd.to_datetime(pessoa['Data_Dem'].iloc[0], errors='coerce')
        if pd.isna(dem): dem = pd.Timestamp.max
        return adm, dem

    pessoa = df_ativos[df_ativos['Cadastro'] == str(chapa)]
    if not pessoa.empty:
        adm = pd.to_datetime(pessoa['Data_Admissão'].iloc[0], dayfirst=True, errors='coerce')
        return adm, pd.Timestamp.max

    return None, None


DATA_MUDANCA_REGRA_PRESENCIAL = pd.Timestamp("2025-05-01")
MIN_PRESENCIAL_ANTES_MUDANCA = 2
MIN_PRESENCIAL_APOS_MUDANCA = 3

def min_presencial_exigido(semana_ano: str) -> int:
    """Retorna o mínimo de dias presenciais exigido para a semana ISO 'AAAA-SS'.

    - Até 30/04/2025: mínimo 2 dias presenciais/semana
    - A partir de 01/05/2025: mínimo 3 dias presenciais/semana
    """
    try:
        ano_str, sem_str = str(semana_ano).split("-")
        ano, sem = int(ano_str), int(sem_str)
        dt_ini_semana = datetime.fromisocalendar(ano, sem, 1)  # segunda-feira
        dt_fim_semana = dt_ini_semana + timedelta(days=6)
        # se a semana encosta na data de mudança (01/05/2025), aplica a regra nova
        return MIN_PRESENCIAL_APOS_MUDANCA if pd.Timestamp(dt_fim_semana) >= DATA_MUDANCA_REGRA_PRESENCIAL else MIN_PRESENCIAL_ANTES_MUDANCA
    except Exception:
        # fallback conservador: aplica a regra mais recente
        return MIN_PRESENCIAL_APOS_MUDANCA


def calcular_percentual_presenca(dias_presenca_por_semana):
    """Calcula o percentual médio de presença no período com base na regra vigente por semana.

    Score semanal = min(dias_presenciais / mínimo_exigido_na_semana, 1).
    Considera mudança de regra a partir de maio/2025 (mínimo 3 dias presenciais).
    """
    if dias_presenca_por_semana is None or len(dias_presenca_por_semana) == 0:
        return 0.0

    scores = []
    for semana_ano, qtd in dias_presenca_por_semana.items():
        req = float(min_presencial_exigido(semana_ano))
        qtd = 0.0 if pd.isna(qtd) else float(qtd)
        score = 0.0 if req <= 0 else min(qtd / req, 1.0)
        scores.append(score)

    return round(float(np.mean(scores)) * 100, 1)

def gerar_relatorio_formatado(nome_input, chapa_input, media_presenca_semana, percentuais_presenca, periodo_label_completo):
    return (f"   Relatório Corporativo - Sistema Integrado de Gestão de Frequência da Volkswagen\n\n"
            f"   Este relatório tem como finalidade fornecer uma base de dados consolidada, com visualizações estratégicas e indicadores de performance "
            f"relacionados à presença e ao regime de trabalho dos colaboradores entre {periodo_label_completo}.\n\n"
            rf"  O colaborador {nome_input} ({chapa_input}) apresentou uma média semanal de presença de "
            rf"$\bf{{{media_presenca_semana}}}$ dia(s) no formato presencial. Para fins de cálculo, foi considerada a regra vigente por semana:"
            f"\n- até Abril/2025: mínimo 2 dias presenciais/semana;\n"
            f"- a partir de Maio/2025: mínimo 3 dias presenciais/semana.\n"
            r"A pontuação semanal é calculada como: min(dias_presenciais / mínimo_exigido_na_semana, 1).\n"
            r"Ao longo do período analisado, a presença semanal média do colaborador no formato presencial foi de "
            rf"$\bf{{{percentuais_presenca}\%}}$."
            f"\n\n   Importante destacar que, para este relatório, não foram consideradas como justificativas válidas de ausência: feriados, férias individuais, "
            f"viagens a trabalho ou ausências previamente acordadas. A única exceção aplicada foi para períodos de férias coletivas, devidamente identificadas "
            f"e excluídas do cálculo para evitar distorções. "
            f'Foram identificadas as seguintes semanas com presença abaixo do mínimo exigido (Ano-Semana):'
    )


'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Áreas ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
def coletar_admissoes_e_demissoes():
    """
    Retorna dois dicts:
    adm: {Cadastro -> Timestamp(admissão)}
    dem: {Cadastro -> Timestamp(demissão)}  (opcional)
    Considera:
    - df_desligados: 'Data_Adm', 'Data_Dem'
    - df_ativos: 'Data_Admissão'
    - df_semana_merged: tenta alguns nomes comuns (fallback)
    Mantém a MENOR data de admissão encontrada e a MENOR data de demissão (mais antiga).
    """
    adm = {}
    dem = {}

    def _norm_df(df):
        return df if isinstance(df, pd.DataFrame) and not df.empty else None

    def _put_adm(k, v):
        if pd.isna(v): 
            return
        if (k not in adm) or (v < adm[k]):
            adm[k] = v

    def _put_dem(k, v):
        if pd.isna(v): 
            return
        if (k not in dem) or (v < dem[k]):
            dem[k] = v

    # --- desligados: Data_Adm / Data_Dem ---
    dfa = _norm_df(globals().get('df_desligados', None))
    if dfa is not None and 'Cadastro' in dfa.columns:
        tmp = dfa[['Cadastro']].copy()
        tmp['Cadastro'] = tmp['Cadastro'].astype(str)

        if 'Data_Adm' in dfa.columns:
            da = pd.to_datetime(dfa['Data_Adm'], errors='coerce')
            for k, v in zip(tmp['Cadastro'], da):
                _put_adm(k, v)

        if 'Data_Dem' in dfa.columns:
            dd = pd.to_datetime(dfa['Data_Dem'], errors='coerce')
            for k, v in zip(tmp['Cadastro'], dd):
                _put_dem(k, v)

    # --- ativos: Data_Admissão ---
    dfd = _norm_df(globals().get('df_ativos', None))
    if dfd is not None and 'Cadastro' in dfd.columns and 'Data_Admissão' in dfd.columns:
        tmp = dfd[['Cadastro', 'Data_Admissão']].copy()
        tmp['Cadastro'] = tmp['Cadastro'].astype(str)
        tmp['Data_Admissão'] = pd.to_datetime(tmp['Data_Admissão'], errors='coerce')
        tmp = tmp.dropna(subset=['Data_Admissão']).drop_duplicates('Cadastro', keep='first')
        for k, v in zip(tmp['Cadastro'], tmp['Data_Admissão']):
            _put_adm(k, v)

    # --- semana_merged: fallback com nomes comuns de admissão ---
    dfsm = _norm_df(globals().get('df_semana_merged', None))
    if dfsm is not None and 'Cadastro' in dfsm.columns:
        possiveis = ['Admissao', 'Admissão', 'Data Admissao', 'Data Admissão', 'DtAdmissao', 'Dt_Admissao', 'Hire Date', 'Admission']
        col = next((c for c in possiveis if c in dfsm.columns), None)
        if col:
            tmp = dfsm[['Cadastro', col]].copy()
            tmp['Cadastro'] = tmp['Cadastro'].astype(str)
            tmp[col] = pd.to_datetime(tmp[col], errors='coerce')
            tmp = tmp.dropna(subset=[col]).drop_duplicates('Cadastro', keep='first')
            for k, v in zip(tmp['Cadastro'], tmp[col]):
                _put_adm(k, v)

    return adm, dem


def tempo_de_casa_str(adm_ts, ref=None, dem_ts=None):
    """
    Formata como 'Xa Ym' ou 'Xm'.
    - adm_ts: Timestamp da admissão
    - dem_ts: Timestamp da demissão (se houver). Se existir, usamos ref = min(ref, dem_ts)
    - ref: data de referência (default = hoje)
    """
    if pd.isna(adm_ts) or adm_ts is None:
        return '—'
    if ref is None:
        ref = pd.Timestamp.today().normalize()
    if pd.notna(dem_ts):
        ref = min(ref, dem_ts)

    # meses completos
    months = (ref.year - adm_ts.year) * 12 + (ref.month - adm_ts.month)
    if ref.day < getattr(adm_ts, 'day', ref.day):
        months -= 1
    months = max(0, months)
    y, m = divmod(months, 12)
    return f"{y}a {m}m" if y else f"{m}m"


def figura_vazia_linha(msg="Selecione um Mensalista e um Executivo para carregar o gráfico."):
    fig = go.Figure()
    fig.add_annotation(text=msg, showarrow=False, x=0.5, y=0.5, xref='paper', yref='paper')
    fig.update_layout(height=520, margin=dict(l=60, r=40, t=40, b=40))
    return fig


def calcular_ordem_heatmap(df_full: pd.DataFrame):
    """Retorna (ordem_mensalistas, ordem_executivos) como listas de strings de Cadastro,
    reproduzindo a ordenação usada no heatmap."""
    if df_full is None or df_full.empty:
        return [], []
    dfp = df_full.copy()
    dfp['Data'] = pd.to_datetime(dfp['Data'], errors='coerce')
    dfp = dfp.dropna(subset=['Data'])
    if dfp.empty:
        return [], []

    days = pd.date_range(dfp['Data'].min().normalize(), dfp['Data'].max().normalize(), freq='D')
    pres = (dfp.groupby(['Data','Cadastro'])['Contagem Presença']
            .max().unstack('Cadastro').reindex(days).fillna(0))

    area_col = 'Área' if 'Área' in dfp.columns else ('Área' if 'Área' in dfp.columns else None)
    meta = (dfp.sort_values('Data')
            .drop_duplicates('Cadastro', keep='last')
            [['Cadastro','Nome','Cargo', area_col]]
            .set_index('Cadastro')
            .rename(columns={area_col:'ÁreaNorm'}))

    def _tok(s): return [t.strip() for t in str(s).replace('/', '-').split('-') if t.strip()]
    def _akey(a):
        t=_tok(a); return (len(t), tuple(t))

    cad_M = sorted(meta.index[meta['Cargo']=='M'], key=lambda c: (_akey(meta.loc[c,'ÁreaNorm']),
                                                                str(meta.loc[c,'Nome']).lower(), str(c)))
    cad_E = sorted(meta.index[meta['Cargo']=='E'], key=lambda c: (_akey(meta.loc[c,'ÁreaNorm']),
                                                                str(meta.loc[c,'Nome']).lower(), str(c)))
    if not cad_M or not cad_E:
        return [str(x) for x in cad_M], [str(x) for x in cad_E]

    A = pres.reindex(columns=cad_M).fillna(0).to_numpy(float)
    B = pres.reindex(columns=cad_E).fillna(0).to_numpy(float)
    encontros = A.T @ B
    denom = A.sum(axis=0)[:,None]

    def _rel(am, ae):
        if pd.isna(am) or pd.isna(ae): return False
        sm=str(am).replace('/','-').lower().strip()
        se=str(ae).replace('/','-').lower().strip()
        return bool(se) and sm.startswith(se)

    areas_M = [meta.loc[c,'ÁreaNorm'] for c in cad_M]
    areas_E = [meta.loc[c,'ÁreaNorm'] for c in cad_E]
    allowed = np.array([[ _rel(areas_M[i], areas_E[j]) for j in range(len(cad_E)) ]
                        for i in range(len(cad_M))], dtype=bool)
    with np.errstate(invalid='ignore'):
        percent = np.divide(encontros, denom, out=np.full_like(encontros, np.nan), where=(denom>0)&allowed)*100

    m_scores = []
    for i, am in enumerate(areas_M):
        mask_e = [ae == am for ae in areas_E]
        vals = percent[i, mask_e]
        sc = np.nanmean(vals) if np.any(mask_e) else np.nan
        m_scores.append(-1e9 if np.isnan(sc) else sc)

    e_scores = []
    for j, ae in enumerate(areas_E):
        mask_m = [am == ae for am in areas_M]
        vals = percent[mask_m, j]
        sc = np.nanmean(vals) if np.any(mask_m) else np.nan
        e_scores.append(-1e9 if np.isnan(sc) else sc)

    order_M = sorted(range(len(cad_M)),
                    key=lambda i: (_akey(areas_M[i]), -m_scores[i], str(meta.loc[cad_M[i],'Nome']).lower(), str(cad_M[i])))
    order_E = sorted(range(len(cad_E)),
                    key=lambda j: (_akey(areas_E[j]), -e_scores[j], str(meta.loc[cad_E[j],'Nome']).lower(), str(cad_E[j])))

    cad_M = [str(cad_M[i]) for i in order_M]
    cad_E = [str(cad_E[j]) for j in order_E]
    return cad_M, cad_E


def get_area(pathname):
    try:
        base = pathname.split("/area/")[1]
    except:
        return None, None
    if "?" in base:
        areaslug, query = base.split("?", 1)
        params = dict([p.split("=",1) for p in query.split("&") if "=" in p])
        dep = normalizar_str(params.get("dep")) if "dep" in params else None
    else:
        areaslug = base; dep = None
    area = normalizar_str(areaslug)
    return area, dep


def build_dataframe_diario_area(area, subarea, her_df, df_datas_local, df_dept, dt_ini=None, dt_fim=None):
    def empty():
        cols = ['Data','Cadastro','Nome','Cargo','Área','Departamento','Contagem Presença']
        return pd.DataFrame(columns=cols)

    # Normalização leve
    for _df in (her_df, df_dept, df_datas_local):
        for c in ('Heranca','Área','Departamento','Cadastro'):
            if c in _df.columns:
                _df[c] = _df[c].astype(str).str.strip()

    alvo = subarea or area

    # Tenta filtrar por Área
    mask_area_ok = False
    if 'Área' in df_dept.columns:
        if subarea:
            mask_area = df_dept['Área'].eq(alvo)
        else:
            # quando não é subárea, usa contains no nome da área "macro"
            mask_area = df_dept['Área'].str.contains(str(area), case=False, na=False)
        base_dept = df_dept.loc[mask_area, ['Cadastro','Área','Departamento']].drop_duplicates()
        mask_area_ok = not base_dept.empty
    else:
        base_dept = pd.DataFrame(columns=['Cadastro','Área','Departamento'])

    # Se não deu match por Área, cai para Heranca
    if not mask_area_ok:
        # Heranças elegíveis para o alvo (com fallback por contains)
        herancas = (her_df.loc[her_df['Área'].eq(alvo), 'Heranca'].dropna().unique()
                    if 'Área' in her_df.columns and 'Heranca' in her_df.columns else [])
        if len(herancas) == 0 and 'Área' in her_df.columns and 'Heranca' in her_df.columns:
            herancas = (her_df[her_df['Área'].str.contains(str(alvo), case=False, na=False)]
                            ['Heranca'].dropna().unique())

        if len(herancas) == 0:
            # Se nem por Heranca encontramos algo, retorna vazio
            return empty()

        starts_any = lambda h: any(str(h).startswith(p) for p in herancas)

        if 'Heranca' in df_dept.columns:
            mask = df_dept['Heranca'].apply(starts_any)
            base_dept = df_dept.loc[mask, ['Cadastro','Área','Departamento']].drop_duplicates()
        else:
            # Fallback extremo: ainda tenta por Área (pode não existir, então já tratamos acima)
            base_dept = df_dept.loc[mask_area, ['Cadastro','Área','Departamento']].drop_duplicates()

    if base_dept.empty:
        return empty()

    # Nome/Cargo
    ref_cols = ['Cadastro','Nome','Cargo']
    df_ref = (df_semana_merged[ref_cols].drop_duplicates()
            if 'df_semana_merged' in globals() else pd.DataFrame(columns=ref_cols))
    chaves = base_dept.merge(df_ref, on='Cadastro', how='left')

    # Fonte diária
    dfp = df_datas_local[df_datas_local['Cadastro'].isin(chaves['Cadastro'])].copy()
    if dfp.empty:
        return empty()
    dfp['Data'] = pd.to_datetime(dfp['Data'], errors='coerce')
    dfp = dfp.dropna(subset=['Data'])
    
    # >>>> NOVO: filtre pelo período o mais cedo possível <<<<
    if dt_ini is not None and dt_fim is not None:
        dt_ini = pd.Timestamp(dt_ini).normalize()
        dt_fim = pd.Timestamp(dt_fim).normalize()
        dfp = dfp[(dfp['Data'] >= dt_ini) & (dfp['Data'] <= dt_fim)]
    if dfp.empty:
        return empty()

    # Grade Data×Cadastro
    datas = pd.date_range(dfp['Data'].min().normalize(), dfp['Data'].max().normalize(), freq='D')
    cad_unicos = chaves['Cadastro'].unique()
    grade = (pd.MultiIndex.from_product([datas, cad_unicos], names=['Data','Cadastro'])
            .to_frame(index=False)
            .merge(chaves, on='Cadastro', how='left'))

    # Presença diária (0/1)
    pres = (dfp.sort_values('Data')
        .groupby(['Cadastro','Data'], as_index=False)
        .agg(Planta=('Planta', lambda s: s.dropna().iloc[-1] if s.dropna().size else np.nan)))

    # marca presença = 1 onde houver um registro no dia
    pres['Contagem Presença'] = 1

    df_full = (grade.merge(pres, on=['Cadastro','Data'], how='left')
                    .fillna({'Contagem Presença': 0}))
    df_full['Contagem Presença'] = df_full['Contagem Presença'].astype(int)
    df_full['Data'] = pd.to_datetime(df_full['Data'], errors='coerce')
    df_full = df_full.sort_values(['Cadastro','Data'])

    for c in ['Nome','Cargo','Área','Departamento']:
        if c not in df_full.columns:
            df_full[c] = np.nan

    return df_full


def grafico_heatmap_encontros_men(df_full: pd.DataFrame, titulo: str, validar_mesma_planta: bool = True):
    fig = go.Figure()

    def _final(_fig):
        _fig.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
            height=420
        )
        return _fig

    # ---------- validações básicas ----------
    if df_full is None or df_full.empty:
        return _final(fig)

    dfp = df_full.copy()
    dfp['Data'] = pd.to_datetime(dfp['Data'], errors='coerce')
    dfp = dfp.dropna(subset=['Data'])
    if dfp.empty:
        return _final(fig)

    area_col = 'Área' if 'Área' in dfp.columns else ('Área' if 'Área' in dfp.columns else None)

    # eixo de dias contínuos (usado no caminho "sem planta")
    days = pd.date_range(dfp['Data'].min().normalize(),
                        dfp['Data'].max().normalize(), freq='D')

    # presença diária (0/1) por pessoa (para caminho sem planta)
    pres = (dfp.groupby(['Data', 'Cadastro'])['Contagem Presença']
            .max().unstack('Cadastro').reindex(days).fillna(0))

    # ---------- metadados ----------
    meta_src_cols = ['Cadastro', 'Nome', 'Cargo'] + ([area_col] if area_col else [])
    meta_src = (dfp.sort_values('Data')
                .drop_duplicates('Cadastro', keep='last')[meta_src_cols])

    meta = meta_src.set_index('Cadastro').copy()
    if area_col:
        meta = meta.rename(columns={area_col: 'ÁreaNorm'})
    else:
        meta['ÁreaNorm'] = ''

    # ---------- tenure ----------
    global ADM_CACHE, DEM_CACHE
    if 'ADM_CACHE' not in globals():
        ADM_CACHE = None
    if 'DEM_CACHE' not in globals():
        DEM_CACHE = None
    if ADM_CACHE is None or DEM_CACHE is None:
        ADM_CACHE, DEM_CACHE = coletar_admissoes_e_demissoes()

    ref_dt = days.max() if len(days) else pd.Timestamp.today().normalize()
    tenure = {}
    for c in meta.index:
        cid = str(c)
        adm_ts = ADM_CACHE.get(cid, None)
        dem_ts = DEM_CACHE.get(cid, None)
        tenure[cid] = tempo_de_casa_str(adm_ts, ref=ref_dt, dem_ts=dem_ts)

    # ---------- helpers ----------
    def _tok(s): 
        return [t.strip() for t in str(s).replace('/', '-').split('-') if t.strip()]

    def _akey(a):
        t = _tok(a)
        return (len(t), tuple(t))

    # Relacionamento permitido (mesma hierarquia/raiz de área):
    # no seu original: mensalista (am) deve "começar com" a área do executivo (ae)
    def _rel(am, ae):
        if pd.isna(am) or pd.isna(ae):
            return False
        sm = str(am).replace('/', '-').lower().strip()
        se = str(ae).replace('/', '-').lower().strip()
        return bool(se) and sm.startswith(se)

    # ---------- separa M e E + ordenação inicial ----------
    # Mantém a mesma ideia do seu original: agrupar por área e nome
    mask_M = meta['Cargo'].astype(str).eq('M')
    mask_E = meta['Cargo'].astype(str).eq('E')

    cad_M = sorted(meta.index[mask_M],
                key=lambda c: (_akey(meta.loc[c, 'ÁreaNorm']),
                                str(meta.loc[c, 'Nome']).lower(), str(c)))
    cad_E = sorted(meta.index[mask_E],
                key=lambda c: (_akey(meta.loc[c, 'ÁreaNorm']),
                                str(meta.loc[c, 'Nome']).lower(), str(c)))

    if not cad_M or not cad_E:
        return _final(fig)

    # ---------- MATRIZ DE ENCONTROS ----------
    if validar_mesma_planta and ('Planta' in dfp.columns):
        # --- co-presença na MESMA PLANTA (aceita múltiplas plantas por pessoa/dia) ---

        df_day = dfp[['Data', 'Cadastro', 'Planta']].copy()
        df_day['Data'] = pd.to_datetime(df_day['Data'], errors='coerce').dt.normalize()
        df_day = df_day.dropna(subset=['Data', 'Cadastro'])

        M_ids = cad_M
        E_ids = cad_E

        # helper: converte célula em conjunto de plantas (lida com lista/tupla/set/strings com separadores)
        def _to_set(x):
            if x is None or (isinstance(x, float) and pd.isna(x)):
                return frozenset()
            if isinstance(x, (set, frozenset)):
                return frozenset(map(lambda t: str(t).strip().lower(), x))
            if isinstance(x, (list, tuple)):
                return frozenset(map(lambda t: str(t).strip().lower(), x))
            s = str(x).strip()
            if not s:
                return frozenset()
            for sep in ['|', ';', ',', '/', '\\']:
                if sep in s:
                    return frozenset(map(lambda t: t.strip().lower(), s.split(sep)))
            return frozenset([s.lower()])

        encontros_same = np.zeros((len(M_ids), len(E_ids)), dtype=float)
        dias_pres_m = np.zeros((len(M_ids),), dtype=float)  # denominador por M

        for dia, g in df_day.groupby('Data'):
            # resolve duplicidades no mesmo dia por cadastro
            g = g.sort_values('Cadastro').drop_duplicates(subset=['Cadastro'], keep='last')
            s = g.set_index('Cadastro')['Planta']

            # plantas dos M e dos E no dia
            pM = s.reindex(M_ids)
            pE = s.reindex(E_ids)

            # conjuntos por cadastro
            m_sets = [_to_set(val) for val in pM.tolist()]
            e_sets = [_to_set(val) for val in pE.tolist()]

            # presença = conjunto não-vazio
            presM = np.array([len(ms) > 0 for ms in m_sets], dtype=bool)
            presE = np.array([len(es) > 0 for es in e_sets], dtype=bool)

            # matriz NxM: há interseção de plantas e ambos presentes
            same = np.array(
                [[(len(ms) > 0 and len(es) > 0 and not ms.isdisjoint(es)) for es in e_sets] for ms in m_sets],
                dtype=bool
            )

            valid = same & presM[:, None] & presE[None, :]

            encontros_same += valid.astype(float)
            dias_pres_m += presM.astype(float)

        denom = dias_pres_m[:, None]
        with np.errstate(invalid='ignore', divide='ignore'):
            percent = np.divide(
                encontros_same, denom,
                out=np.full_like(encontros_same, np.nan, dtype=float),
                where=(denom > 0)
            ) * 100.0
    else:
        # --- caminho original: co-presença dia a dia (sem validar planta) ---
        A = pres.reindex(columns=cad_M).fillna(0).to_numpy(float)  # dias × M
        B = pres.reindex(columns=cad_E).fillna(0).to_numpy(float)  # dias × E
        encontros = A.T @ B                                        # M × E
        denom = A.sum(axis=0)[:, None]                             # por M
        with np.errstate(invalid='ignore', divide='ignore'):
            percent = np.divide(
                encontros, denom,
                out=np.full_like(encontros, np.nan, dtype=float),
                where=(denom > 0)
            ) * 100.0

    # ---------- máscara de áreas permitidas (igual ao seu original) ----------
    allowed = np.array([[ _rel(meta.loc[m, 'ÁreaNorm'], meta.loc[e, 'ÁreaNorm']) for e in cad_E ]
                        for m in cad_M], dtype=bool)
    # zera onde não é permitido + mantém NaN onde não há denom
    percent = np.where(allowed, percent, np.nan)

    # ---------- ordenar por percentual dentro da mesma área ----------
    areas_M = [meta.loc[c, 'ÁreaNorm'] for c in cad_M]
    areas_E = [meta.loc[c, 'ÁreaNorm'] for c in cad_E]

    # score de cada M: média dos % somente com E da mesma área
    m_scores = []
    for i, am in enumerate(areas_M):
        mask_e = [ae == am for ae in areas_E]
        vals = percent[i, mask_e]
        with np.errstate(invalid='ignore'):
            sc = np.nanmean(vals) if np.any(mask_e) else np.nan
        m_scores.append(-1e9 if np.isnan(sc) else sc)

    # score de cada E: média dos % somente com M da mesma área
    e_scores = []
    for j, ae in enumerate(areas_E):
        mask_m = [am == ae for am in areas_M]
        vals = percent[mask_m, j]
        with np.errstate(invalid='ignore'):
            sc = np.nanmean(vals) if np.any(mask_m) else np.nan
        e_scores.append(-1e9 if np.isnan(sc) else sc)

    # reordenar mantendo agrupamento por área e depois score desc e nome
    order_M = sorted(range(len(cad_M)),
                    key=lambda i: (_akey(areas_M[i]),
                                    -m_scores[i],
                                    str(meta.loc[cad_M[i], 'Nome']).lower(),
                                    str(cad_M[i])))
    order_E = sorted(range(len(cad_E)),
                    key=lambda j: (_akey(areas_E[j]),
                                    -e_scores[j],
                                    str(meta.loc[cad_E[j], 'Nome']).lower(),
                                    str(cad_E[j])))

    # aplicar ordem
    cad_M = [cad_M[i] for i in order_M]
    cad_E = [cad_E[j] for j in order_E]
    percent = percent[np.ix_(order_M, order_E)]

    # ---------- labels ----------
    def _abbr(n):
        p = str(n).split()
        return "" if not p else " ".join([p[0].capitalize()] + [x[0].upper() + "." for x in p[1:]])

    y_labels = [f"{_abbr(meta.loc[c, 'Nome'])} "
                f"<span style='font-size:10px;opacity:.85'><b>({tenure.get(str(c), '—')})</b></span>"
                for c in cad_M]  # Mensalistas (eixo Y)

    x_labels = [f"{_abbr(meta.loc[c, 'Nome'])}"
                f" <span style='font-size:10px;opacity:.85'><b>({tenure.get(str(c), '—')})</b></span>"
                f"<br>({str(meta.loc[c, 'ÁreaNorm'])})"
                for c in cad_E]  # Executivos (eixo X)

    y_full = [f"{str(meta.loc[c, 'Nome']).title()} ({c})" for c in cad_M]
    x_full = [f"{str(meta.loc[c, 'Nome']).title()} ({c})" for c in cad_E]
    customdata = [[(xe, ye) for xe in x_full] for ye in y_full]

    # ---------- aparência ----------
    colors = [
        [0.00, "#af3333"], [0.35, "#fc9898"],
        [0.35, "#ffef09"], [0.60, "#fff9a9"],
        [0.60, "#9effd0"], [1.00, "#267951"]
    ]
    text = np.vectorize(
        lambda v: "" if (v is None or (isinstance(v, float) and np.isnan(v))) else f"{v:.1f}%"
    )(percent)

    hover_title = "Encontros" if validar_mesma_planta and ('Planta' in dfp.columns) \
                else "Encontros (co-presença)"

    fig = go.Figure(data=go.Hotmap if False else go.Heatmap(  # guardinha anti-typo
        z=percent, x=x_labels, y=y_labels, customdata=customdata,
        colorscale=colors, zmin=0, zmax=100, text=text, texttemplate="%{text}",
        hovertemplate = (
            "<b>%{customdata[1]}</b> × <b>%{customdata[0]}</b><br>"
            f"{hover_title}: %{{z:.1f}}%<extra></extra>"
        ),
        showscale=False, hoverongaps=False, hoverlabel=dict(bgcolor="white", font=dict(color="black"))
    ))

    linhas = max(len(y_labels), 1)
    altura = max(460, 40 + 23 * max(linhas, 10))
    fig.update_layout(title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
                    xaxis_title="", yaxis_title="", height=altura,
                    margin=dict(l=200, r=40, t=70, b=150))
    fig.update_xaxes(tickangle=-45, autorange="reversed", automargin=True)
    fig.update_yaxes(automargin=True)

    return fig


def _construir_encontros_exec_men(df_full: pd.DataFrame) -> pd.DataFrame:
    """Cria um DataFrame de pares (Data, Exec, Men) com co-presença na mesma planta/dia."""
    if df_full is None or df_full.empty:
        return pd.DataFrame(columns=['Data','Exec','Men'])
    d = df_full.copy()
    d['Data'] = pd.to_datetime(d['Data'], errors='coerce')
    d = d.dropna(subset=['Data'])
    if 'Planta' not in d.columns:
        d['Planta'] = 'X'
    # Normaliza chaves
    d['Cadastro'] = d['Cadastro'].map(_safe_hashable).astype(str)
    d['Cargo'] = d['Cargo'].map(_safe_hashable).astype(str).str.upper().str[:1]
    d['Planta'] = d['Planta'].map(_safe_hashable).astype(str)
    # por dia/planta, forma todos os pares (E, M)
    pares = []
    for (dia, planta), g in d.groupby(['Data','Planta']):
        execs = g.loc[g['Cargo'].eq('E'), 'Cadastro'].unique().tolist()
        mens  = g.loc[~g['Cargo'].eq('E'), 'Cadastro'].unique().tolist()
        for e in execs:
            for m in mens:
                pares.append((dia.normalize(), e, m))
    return pd.DataFrame(pares, columns=['Data','Exec','Men'])


def _metricas_rel_exec_men(df_full: pd.DataFrame):
    """Calcula métricas para radar: encontros, pessoas únicas, dias, semanas, recência(28d)."""
    pares = _construir_encontros_exec_men(df_full)
    if pares.empty:
        return (
            pd.DataFrame(columns=['Exec','encontros','pessoas_unicas','dias','semanas','recencia']),
            pd.DataFrame(columns=['Men','encontros','pessoas_unicas','dias','semanas','recencia']),
            pd.DataFrame(columns=['Data','encontros_semana'])
        )
    pares['Semana'] = pares['Data'].dt.to_period('W').apply(lambda p: p.start_time)
    # Exec
    e_grp = pares.groupby('Exec')
    exec_df = pd.DataFrame({
        'Exec': e_grp.size().index,
        'encontros': e_grp.size().values,
        'pessoas_unicas': e_grp['Men'].nunique().values,
        'dias': e_grp['Data'].nunique().values,
        'semanas': e_grp['Semana'].nunique().values,
        'recencia': e_grp.apply(lambda g: g.loc[g['Data'] >= (pares['Data'].max() - pd.Timedelta(days=28))].shape[0]).values
    })
    # Mensalistas
    m_grp = pares.groupby('Men')
    men_df = pd.DataFrame({
        'Men': m_grp.size().index,
        'encontros': m_grp.size().values,
        'pessoas_unicas': m_grp['Exec'].nunique().values,
        'dias': m_grp['Data'].nunique().values,
        'semanas': m_grp['Semana'].nunique().values,
        'recencia': m_grp.apply(lambda g: g.loc[g['Data'] >= (pares['Data'].max() - pd.Timedelta(days=28))].shape[0]).values
    })
    # Série semanal total
    serie_sem = pares.groupby('Semana').size().reset_index(name='encontros_semana')
    return exec_df, men_df, serie_sem


def _radar_from_metrics(df: pd.DataFrame, titulo: str, label_col: str, top_n: int = 5):
    """Cria um radar (pentágono) com 5 eixos para top_n entidades (por 'encontros')."""
    fig = go.Figure()
    if df is None or df.empty:
        fig.update_layout(title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}}, height=420)
        return fig
    # Seleciona top_n pelo total de encontros
    df = df.sort_values('encontros', ascending=False).head(top_n).copy()
    # Normaliza cada métrica para [0, 100]
    metrics = ['encontros','pessoas_unicas','dias','semanas','recencia']
    for m in metrics:
        vmax = df[m].max() or 1
        df[m] = (df[m] / vmax) * 100.0
    categorias = ['Interações','Pessoas únicas','Dias ativos','Semanas ativas','Recência']
    for _, row in df.iterrows():
        r = [row['encontros'], row['pessoas_unicas'], row['dias'], row['semanas'], row['recencia']]
        fig.add_trace(go.Scatterpolar(r=r + [r[0]], theta=categorias + [categorias[0]], name=str(row[label_col]), fill='toself'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0,100])),
        showlegend=True,
        height=420,
        title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}}
    )
    return fig


def grafico_pentagono_rel_exec(df_full: pd.DataFrame, titulo: str):
    exec_df, _, _ = _metricas_rel_exec_men(df_full)
    return _radar_from_metrics(exec_df.rename(columns={'Exec':'Label'}), titulo, 'Label', top_n=5)




def grafico_pentagonos_rel_exec_multiplos(
    df_full: pd.DataFrame,
    titulo: str,
    max_exec: int = 12,
    max_eixos: int | None = None,        # None = mostra todos (sem limite); senão limita N eixos
    range_mode: str | None = None,      # compatibilidade: ignorado (range fixo 0–100%)
    validar_mesma_planta: bool = True,  # respeita planta/cluster (se houver)
    df_hierarquia: pd.DataFrame | None = None,
    is_almoco: bool = False,            # se True: hover mostra somente o percentual
    strict: bool = True,                # se True: levanta erro claro; se False: devolve fig com mensagem
    n_cols: int = 3,                    # layout dos subplots
    include_exec_subordinados: bool = False,  # inclui "executivos menores" no radar do executivo "maior"
    max_exec_subordinados: int = 3,          # reserva até N eixos para executivos subordinados
):
    """Radares (polar) por executivo — eixos dinâmicos seguindo a MESMA regra do heatmap.
    Obs.: parâmetro range_mode é aceito por compatibilidade, mas o radar usa sempre range 0–100%.

    Regra base (igual ao heatmap Mensalistas×Executivos):
      - Eixo = mensalista comparável ao executivo pela Área
      - Comparável quando: area_m.startswith(area_e) após normalização (barra -> hífen, lower)

    Adicional solicitado:
      - No radar de um executivo "maior" (Área mais abrangente), incluir também os executivos "menores"
        cuja área começa com a área do "maior" (area_exec_sub.startswith(area_exec_maior)), área diferente.

    Métrica (no mesmo espírito do heatmap):
      pct(rel, exec) = dias_em_comum(rel, exec) / dias_presenca_rel(rel) * 100

      - Para mensalistas, o denominador é a presença do mensalista (igual heatmap M×E).
      - Para executivos subordinados (quando incluídos), o denominador é a presença do executivo subordinado.
    """
    # ------------------------- helpers locais -------------------------
    def _fig_erro(msg: str) -> go.Figure:
        fig = go.Figure()
        fig.update_layout(
            title={'text': f"{titulo} — ERRO", 'font': {'family': 'VW Text', 'size': 22}, 'x': 0.5, 'xanchor': 'center', 'pad': {'t': 28, 'b': 28}},
            height=420,
            margin=dict(l=60, r=40, t=80, b=40),
            font=dict(family="VW Text"),
            showlegend=False
        )
        fig.add_annotation(
            text=str(msg),
            x=0.5, y=0.5, xref="paper", yref="paper",
            showarrow=False,
            font=dict(family="VW Text", size=14),
            align="center"
        )
        return fig

    def _raise_or_fig(msg: str) -> go.Figure:
        if strict:
            raise ValueError(msg)
        return _fig_erro(msg)

    # abreviação (se existir no projeto, usa)
    try:
        _abbr = _abbr_nome_vw  # noqa: F821
    except Exception:
        def _abbr(x: str) -> str:
            x = "" if x is None else str(x).strip()
            if not x:
                return ""
            parts = x.split()
            return parts[0] if len(parts) == 1 else f"{parts[0]} {parts[-1]}"

    # ------------------------- cores (mesmas do heatmap) -------------------------
    colorscale = [
        [0.00, "#af3333"], [0.35, "#fc9898"],
        [0.35, "#ffef09"], [0.60, "#fff9a9"],
        [0.60, "#9effd0"], [1.00, "#267951"],
    ]

    def _hex_rgba(hex_color: str, alpha: float = 0.22) -> str:
        try:
            r_, g_, b_ = _hex_to_rgb(hex_color)  # noqa: F821 (função global no projeto)
        except Exception:
            h = hex_color.strip().lstrip("#")
            if len(h) == 3:
                h = "".join([c * 2 for c in h])
            r_, g_, b_ = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        return f"rgba({r_},{g_},{b_},{alpha})"

    def _cor_heatmap(pct: float) -> str:
        # mapeia 0–100 para a mesma escala do heatmap (com transições em 35% e 60%)
        try:
            v = float(pct)
        except Exception:
            v = 0.0
        if np.isnan(v):
            return "#cccccc"
        v = max(0.0, min(100.0, v))
        if v <= 35.0:
            t = v / 35.0 if 35.0 else 0.0
            return interpolar_cor_hex("#af3333", "#fc9898", t)
        elif v <= 60.0:
            t = (v - 35.0) / 25.0 if 25.0 else 0.0
            return interpolar_cor_hex("#ffef09", "#fff9a9", t)
        else:
            t = (v - 60.0) / 40.0 if 40.0 else 0.0
            return interpolar_cor_hex("#9effd0", "#267951", t)


    def _norm_area(x) -> str:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return ""
        return str(x).replace("/", "-").strip().lower()

    def _rel_por_area(area_a, area_b) -> bool:
        """True se area_a começa com area_b (após normalização)."""
        sa = _norm_area(area_a)
        sb = _norm_area(area_b)
        return bool(sb) and sa.startswith(sb)

    # compatibilidade planta/cluster (sempre retorna bool)
    def _planta_ok(a, b) -> bool:
        if a is None or b is None:
            return False

        if isinstance(a, (np.ndarray, pd.Series)):
            a = a.tolist()
        if isinstance(b, (np.ndarray, pd.Series)):
            b = b.tolist()

        if isinstance(a, (set, frozenset, list, tuple)) or isinstance(b, (set, frozenset, list, tuple)):
            try:
                sa = set(a) if not isinstance(a, (set, frozenset)) else set(a)
                sb = set(b) if not isinstance(b, (set, frozenset)) else set(b)
                return len(sa.intersection(sb)) > 0
            except Exception:
                return False

        try:
            return str(a) == str(b)
        except Exception:
            return False

    def _planta_nonempty(x) -> bool:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return False
        if isinstance(x, (np.ndarray, pd.Series)):
            try:
                x = x.tolist()
            except Exception:
                pass
        if isinstance(x, (set, frozenset, list, tuple)):
            return len(x) > 0
        s = str(x).strip()
        return bool(s) and s.lower() not in ("nan", "none")


    def _detect_area_col(df: pd.DataFrame) -> str | None:
        if df is None or df.empty:
            return None
        if "Área" in df.columns:
            return "Área"
        if "Área" in df.columns:
            return "Área"
        for c in df.columns:
            if "area" in str(c).lower():
                return c
        return None

    # ------------------------- validações base -------------------------
    if df_full is None or getattr(df_full, "empty", True):
        fig_empty = go.Figure()
        fig_empty.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
            height=420, showlegend=False, font=dict(family="VW Text")
        )
        return fig_empty

    # sanitiza (resolve np.ndarray/list/set em colunas object)
    df_rel = _sanitize_object_columns(df_full.copy())

    required = ["Data", "Cadastro", "Cargo"]
    missing = [c for c in required if c not in df_rel.columns]
    if missing:
        return _raise_or_fig(f"Colunas obrigatórias ausentes para o modo Teia: {missing}")

    df_rel["Data"] = pd.to_datetime(df_rel["Data"], errors="coerce")
    df_rel = df_rel.dropna(subset=["Data"])
    if df_rel.empty:
        fig_empty = go.Figure()
        fig_empty.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
            height=420, showlegend=False, font=dict(family="VW Text")
        )
        return fig_empty

    df_rel["Cadastro"] = df_rel["Cadastro"].map(_safe_hashable).astype(str)

    # normaliza Cargo para E/M
    c = df_rel["Cargo"].astype(str).str.strip()
    cu = c.str.upper()
    df_rel["Cargo"] = np.where(
        cu.str.startswith("E") | cu.str.contains("EXEC"),
        "E",
        np.where(cu.str.startswith("M") | cu.str.contains("MENS"), "M", cu.str[:1])
    )

    if "Planta" not in df_rel.columns:
        df_rel["Planta"] = "X"

    # dedup para não inflar presença (mesma abordagem do heatmap)
    df_rel = df_rel.drop_duplicates(subset=["Data", "Cadastro"], keep="last")

    # ------------------------- adiciona Área (se necessário) -------------------------
    hier = df_hierarquia
    if hier is None:
        hier = globals().get("df_ativos", None)

    area_col = _detect_area_col(df_rel)
    if area_col is None and isinstance(hier, pd.DataFrame) and not hier.empty and "Cadastro" in hier.columns:
        area_col_h = _detect_area_col(hier)
        if area_col_h:
            h = hier[["Cadastro", area_col_h]].copy()
            h["Cadastro"] = h["Cadastro"].map(_safe_hashable).astype(str)
            df_rel = df_rel.merge(h, on="Cadastro", how="left")
            area_col = area_col_h

    if area_col:
        df_rel["ÁreaNorm"] = df_rel[area_col].astype(str).replace({"nan": "", "None": ""})
    else:
        df_rel["ÁreaNorm"] = ""

    # ------------------------- meta (nome/área/cargo) -------------------------
    meta_cols = ["Cadastro", "Cargo", "ÁreaNorm"]
    if "Nome" in df_rel.columns:
        meta_cols.insert(1, "Nome")

    meta_src = df_rel.sort_values("Data").drop_duplicates("Cadastro", keep="last")[meta_cols].copy()
    meta_src["Cadastro"] = meta_src["Cadastro"].astype(str)

    name_map = {r["Cadastro"]: ("" if ("Nome" not in r or pd.isna(r.get("Nome"))) else str(r.get("Nome")))
                for _, r in meta_src.iterrows()}
    area_map = {r["Cadastro"]: ("" if pd.isna(r.get("ÁreaNorm")) else str(r.get("ÁreaNorm")))
                for _, r in meta_src.iterrows()}

    exec_ids = sorted(meta_src.loc[meta_src["Cargo"].eq("E"), "Cadastro"].unique().tolist())
    men_ids = sorted(meta_src.loc[meta_src["Cargo"].eq("M"), "Cadastro"].unique().tolist())

    if not exec_ids:
        return _raise_or_fig("Nenhum executivo (Cargo=E) encontrado no filtro atual.")

    # ------------------------- relacionamento por Área (igual heatmap) -------------------------
    has_area_info = any(_norm_area(area_map.get(e, "")) for e in exec_ids) and any(_norm_area(area_map.get(m, "")) for m in men_ids)

    related_m_map: dict[str, set[str]] = {}
    related_e_map: dict[str, set[str]] = {}

    if has_area_info:
        for e in exec_ids:
            ae = area_map.get(e, "")
            # mensalistas comparáveis (igual ao heatmap)
            rel_m = {m for m in men_ids if _rel_por_area(area_map.get(m, ""), ae)}
            related_m_map[str(e)] = rel_m

            # executivos "menores" no relacionamento com o "maior"
            if include_exec_subordinados:
                rel_e = {e2 for e2 in exec_ids if (e2 != e and _rel_por_area(area_map.get(e2, ""), ae))}
            else:
                rel_e = set()
            related_e_map[str(e)] = rel_e
    else:
        # fallback: top relacionados (co-presença) — somente quando não há área
        pares = _construir_encontros_exec_men(df_rel)  # usa rotina existente
        if pares is None or pares.empty:
            return _raise_or_fig("Sem dados suficientes para construir a Teia (sem Área e sem co-presença).")
        pares["Exec"] = pares["Exec"].astype(str)
        pares["Men"] = pares["Men"].astype(str)
        for e, g in pares.groupby("Exec"):
            top = (g.groupby("Men")
                   .size()
                   .reset_index(name="encontros")
                   .sort_values("encontros", ascending=False))
            if max_eixos is not None:
                top = top.head(int(max_eixos))
            related_m_map[str(e)] = set(top["Men"].astype(str).tolist())
            related_e_map[str(e)] = set()

    # ------------------------- calcula percentuais (mesma lógica do heatmap) -------------------------
    # denom por "relacionado" (pode ser M ou E subordinado)
    denom_rel: dict[str, int] = {}
    numer_er: dict[tuple[str, str], int] = {}

    df_day = df_rel[["Data", "Cadastro", "Cargo", "Planta"]].copy()
    df_day["Cadastro"] = df_day["Cadastro"].astype(str)

    # conjunto de pessoas que podem aparecer como eixo em algum radar
    candidatos_rel = set()
    for e in exec_ids:
        candidatos_rel |= set(map(str, related_m_map.get(str(e), set())))
        candidatos_rel |= set(map(str, related_e_map.get(str(e), set())))

    for day, g in df_day.groupby("Data"):
        planta_by = dict(zip(g["Cadastro"].astype(str), g["Planta"].tolist()))
        cargo_by = dict(zip(g["Cadastro"].astype(str), g["Cargo"].astype(str)))
        present_set = set(planta_by.keys())

        # denominadores: presença por relacionado (somente para quem é candidato a eixo)
        for pid in present_set.intersection(candidatos_rel):
            if validar_mesma_planta and not _planta_nonempty(planta_by.get(pid, None)):
                continue
            denom_rel[pid] = denom_rel.get(pid, 0) + 1

        # executivos presentes no dia (somente os que têm mapa de relacionamento)
        execs_present = [cad for cad, cargo in zip(g["Cadastro"].astype(str), g["Cargo"].astype(str)) if cargo == "E"]
        execs_present = [e for e in execs_present if e in related_m_map]

        for e in execs_present:
            rels = set()
            rels |= set(map(str, related_m_map.get(e, set())))
            rels |= set(map(str, related_e_map.get(e, set())))
            if not rels:
                continue

            rels_present = present_set.intersection(rels)
            pe = planta_by.get(e, None)

            for pid in rels_present:
                if validar_mesma_planta:
                    pp = planta_by.get(pid, None)
                    if not _planta_ok(pe, pp):
                        continue
                numer_er[(e, pid)] = numer_er.get((e, pid), 0) + 1

    # métricas por par (Exec, Rel)
    if numer_er:
        rows = []
        for (e, pid), num in numer_er.items():
            den = int(denom_rel.get(pid, 0))
            pct = (float(num) / den * 100.0) if den > 0 else np.nan
            rows.append((e, pid, num, den, pct))
        rel_metrics = pd.DataFrame(rows, columns=["Exec", "Rel", "num", "den", "pct"])
    else:
        rel_metrics = pd.DataFrame(columns=["Exec", "Rel", "num", "den", "pct"])

    # score para escolher executivos exibidos
    if not rel_metrics.empty:
        score = (rel_metrics.groupby("Exec")["num"].sum()
                 .sort_values(ascending=False)
                 .reset_index(name="total_num"))
        exec_sel = score["Exec"].astype(str).tolist()
    else:
        pres = (df_rel.loc[df_rel["Cargo"].eq("E")]
                .groupby("Cadastro")["Data"].nunique()
                .sort_values(ascending=False))
        exec_sel = pres.index.astype(str).tolist()

    exec_sel = [e for e in exec_sel if e in exec_ids][:max_exec]
    if not exec_sel:
        return _raise_or_fig("Nenhum executivo elegível após cálculo (verifique filtros/período).")

    # ------------------------- subplots polares -------------------------
    n = len(exec_sel)
    cols = max(1, min(int(n_cols), n))
    rows = int(math.ceil(n / cols))

    subplot_titles = []
    for e in exec_sel:
        nm = name_map.get(str(e), str(e))
        subplot_titles.append(f"{_abbr(nm)} ({e})")
    while len(subplot_titles) < rows * cols:
        subplot_titles.append("")

    specs = [[{"type": "polar"} for _ in range(cols)] for _ in range(rows)]
    fig = make_subplots(rows=rows, cols=cols, specs=specs, subplot_titles=subplot_titles)

    # desloca títulos ~20px PARA CIMA (yshift positivo)
    try:
        for ann in (fig.layout.annotations or []):
            if getattr(ann, "text", "") and "(" in str(ann.text) and ")" in str(ann.text):
                ann.yshift = 30
    except Exception:
        pass

    # range visual: ticks até 100% sempre, mas com um "respiro" para afastar labels (~8 unidades)
    FIXED_TICK_MAX = 100.0
    PAD_VISUAL = 8.0  # 3 + 5 (pedido incremental)
    RANGE_MAX = FIXED_TICK_MAX + PAD_VISUAL

    # ------------------------- 1 radar por executivo -------------------------
    for i, e in enumerate(exec_sel):
        r = (i // cols) + 1
        c = (i % cols) + 1
        e = str(e)

        rel_execs = sorted(list(map(str, related_e_map.get(e, set()))))
        rel_mens  = sorted(list(map(str, related_m_map.get(e, set()))))

        # métricas do executivo
        if not rel_metrics.empty:
            g = rel_metrics.loc[rel_metrics["Exec"].eq(e)].copy()
        else:
            g = pd.DataFrame(columns=["Rel", "pct", "num", "den"])

        val_map = dict(zip(g["Rel"].astype(str), g["pct"].astype(float))) if not g.empty else {}
        num_map = dict(zip(g["Rel"].astype(str), g["num"].astype(int))) if not g.empty else {}
        den_map = dict(zip(g["Rel"].astype(str), g["den"].astype(int))) if not g.empty else {}

        def _mk(pid: str):
            pid = str(pid)
            pct = float(val_map.get(pid, 0.0))
            nnum = int(num_map.get(pid, 0))
            nden = int(den_map.get(pid, int(denom_rel.get(pid, 0))))
            return (pid, pct, nnum, nden)

        exec_vals = [_mk(pid) for pid in rel_execs]
        men_vals  = [_mk(pid) for pid in rel_mens]

        # ordena por pct desc, depois num desc
        exec_vals.sort(key=lambda x: (x[1], x[2]), reverse=True)
        men_vals.sort(key=lambda x: (x[1], x[2]), reverse=True)

        # eixos = exatamente as mesmas pessoas do heatmap (mensalistas comparáveis pela Área)
        rel_vals = men_vals
        if max_eixos is not None:
            rel_vals = rel_vals[:int(max_eixos)]

        if not rel_vals:
            rel_vals = [("Sem pessoas comparáveis (Área)", 0.0, 0, 0)]

        theta = []
        rvals = []
        custom = []

        for pid, pct, nnum, nden in rel_vals:
            if pid == "Sem pessoas comparáveis (Área)":
                p_nome = pid
                pid_show = ""
            else:
                p_nome = name_map.get(pid, pid)
                pid_show = pid

            # Nome em 1ª linha; 2ª linha: Cadastro — % (no estilo do heatmap)
            if pid_show:
                theta.append(f"{_abbr(p_nome)}<br>{pid_show} — {pct:.1f}%")
            else:
                theta.append(f"{_abbr(p_nome)}")
            rvals.append(float(pct))
            _c_hex = _cor_heatmap(pct)
            custom.append([p_nome, pid, nnum, nden, _c_hex, e])

        # fecha polígono
        theta_c = theta + [theta[0]]
        r_c = rvals + [rvals[0]]
        custom_c = custom + [custom[0]]
        # cores por ponto (mesma lógica do heatmap) — usadas nos marcadores e no hover
        _colors = [cd[4] if len(cd) > 4 else '#cccccc' for cd in custom]
        colors_c = _colors + [_colors[0]]

        exec_nome = name_map.get(e, e)


        # cor (igual ao heatmap): linha/preenchimento pelo pct médio; marcadores por pct individual
        _vals_color = [v for v in rvals if isinstance(v, (int, float)) and not np.isnan(v)]
        _pct_mean = float(np.nanmean(_vals_color)) if _vals_color else 0.0
        _line_color = _cor_heatmap(_pct_mean)
        _fill_color = _hex_rgba(_line_color, 0.22)
        # Hover (almoço e geral) no mesmo formato, com cor do ponto (igual ao heatmap)
        hover = (
            "<b>%{customdata[0]}</b> × <b>%{text}</b>"
            "<br><b>%{customdata[5]} x %{customdata[1]} — %{r:.1f}%</b>"
            "<extra></extra>"
        )

        fig.add_trace(
            go.Scatterpolar(
                r=r_c,
                theta=theta_c,
                fill="toself",
                fillcolor=_fill_color,
                mode="lines+markers",
                line=dict(width=2, color=_line_color),
                marker=dict(size=10, line=dict(width=1.5, color='#1a1a1a'), color=colors_c, showscale=False),
                showlegend=False,
                text=[exec_nome] * len(theta_c),
                customdata=custom_c,
                hovertemplate=hover,
            ),
            row=r, col=c
        )

        pol_key = _polar_key(i + 1)

        fig.layout[pol_key].radialaxis.range = [0, RANGE_MAX]
        fig.layout[pol_key].radialaxis.tick0 = 0
        fig.layout[pol_key].radialaxis.dtick = 20
        fig.layout[pol_key].radialaxis.showticklabels = True
        fig.layout[pol_key].radialaxis.tickfont = dict(family="VW Text", size=11)
        fig.layout[pol_key].angularaxis.tickfont = dict(family="VW Text", size=11)
        fig.layout[pol_key].radialaxis.ticksuffix = "%"

    # ------------------------- layout geral -------------------------
    fig.update_layout(
        title={
            'text': titulo,
            'font': {'family': 'VW Text', 'size': 22},
            'x': 0.5, 'xanchor': 'center',
            'pad': {'t': 28, 'b': 20}
        },
        font=dict(family="VW Text"),
        hoverlabel=dict(bgcolor="white", bordercolor="black", font=dict(family="VW Text", color="black")),
        margin=dict(l=70, r=40, t=125, b=60),
        height=max(520, rows * 420),
        showlegend=False,
    )

    # centralização/ajuste de domains
    if n != 1:
        fig = _ajustar_domains_subplots_polares(fig, rows, cols, n)
    else:
        # centraliza suavemente
        try:
            fig.layout["polar"].domain.x = [0.15, 0.85]
            fig.layout["polar"].domain.y = [0.12, 0.88]
        except Exception:
            pass

    return fig



def grafico_pentagonos_exec_exec_multiplos(
    df_full: pd.DataFrame,
    titulo: str,
    max_exec: int = 12,
    max_eixos: int | None = None,        # None = mostra todos (sem limite); senão limita N eixos
    range_mode: str | None = None,      # compatibilidade: ignorado (range fixo 0–100%)
    validar_mesma_planta: bool = True,
    df_hierarquia: pd.DataFrame | None = None,
    is_almoco: bool = False,            # se True: hover mostra somente o percentual
    strict: bool = True,                # se True: levanta erro claro; se False: devolve fig com mensagem
    n_cols: int = 3,                    # layout dos subplots
):
    """Radares (polar) por executivo — relacionamento entre executivos (Exec × Exec).

    Eixos (por executivo):
      - Outros executivos comparáveis pela Área, seguindo a mesma ideia do heatmap:
        area_outro.startswith(area_exec) após normalização (barra -> hífen, lower).

    Valor radial (percentual):
      - pct(outro_exec, exec) = dias_em_comum(outro_exec, exec) / dias_presenca(outro_exec) * 100
      - Isso corresponde à célula do heatmap Exec×Exec na *linha* do "outro_exec" e *coluna* do "exec".

    Observações:
      - range_mode é aceito por compatibilidade, mas o radar usa sempre 0–100% (com ticks até 100%).
      - Se não houver Área disponível, faz fallback por top co-presença (somente entre executivos).
    """
    # ------------------------- helpers locais -------------------------
    def _fig_erro(msg: str) -> go.Figure:
        fig = go.Figure()
        fig.update_layout(
            title={'text': f"{titulo} — ERRO", 'font': {'family': 'VW Text', 'size': 22},
                   'x': 0.5, 'xanchor': 'center', 'pad': {'t': 28, 'b': 28}},
            height=420,
            margin=dict(l=60, r=40, t=125, b=40),
            font=dict(family="VW Text"),
            showlegend=False
        )
        fig.add_annotation(
            text=str(msg),
            x=0.5, y=0.5, xref="paper", yref="paper",
            showarrow=False,
            font=dict(family="VW Text", size=14),
            align="center"
        )
        return fig

    def _raise_or_fig(msg: str) -> go.Figure:
        if strict:
            raise ValueError(msg)
        return _fig_erro(msg)

    try:
        _abbr = _abbr_nome_vw  # noqa: F821
    except Exception:
        def _abbr(x: str) -> str:
            x = "" if x is None else str(x).strip()
            if not x:
                return ""
            parts = x.split()
            return parts[0] if len(parts) == 1 else f"{parts[0]} {parts[-1]}"

    # ------------------------- cores (mesmas do heatmap) -------------------------
    colorscale = [
        [0.00, "#af3333"], [0.35, "#fc9898"],
        [0.35, "#ffef09"], [0.60, "#fff9a9"],
        [0.60, "#9effd0"], [1.00, "#267951"],
    ]

    def _hex_rgba(hex_color: str, alpha: float = 0.22) -> str:
        try:
            r_, g_, b_ = _hex_to_rgb(hex_color)  # noqa: F821 (função global no projeto)
        except Exception:
            h = hex_color.strip().lstrip("#")
            if len(h) == 3:
                h = "".join([c * 2 for c in h])
            r_, g_, b_ = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        return f"rgba({r_},{g_},{b_},{alpha})"

    def _cor_heatmap(pct: float) -> str:
        # mapeia 0–100 para a mesma escala do heatmap (com transições em 35% e 60%)
        try:
            v = float(pct)
        except Exception:
            v = 0.0
        if np.isnan(v):
            return "#cccccc"
        v = max(0.0, min(100.0, v))
        if v <= 35.0:
            t = v / 35.0 if 35.0 else 0.0
            return interpolar_cor_hex("#af3333", "#fc9898", t)
        elif v <= 60.0:
            t = (v - 35.0) / 25.0 if 25.0 else 0.0
            return interpolar_cor_hex("#ffef09", "#fff9a9", t)
        else:
            t = (v - 60.0) / 40.0 if 40.0 else 0.0
            return interpolar_cor_hex("#9effd0", "#267951", t)


    def _norm_area(x) -> str:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return ""
        return str(x).replace("/", "-").strip().lower()

    def _rel_por_area(area_outro, area_exec) -> bool:
        so = _norm_area(area_outro)
        se = _norm_area(area_exec)
        return bool(se) and so.startswith(se)

    def _planta_ok(a, b) -> bool:
        if a is None or b is None:
            return False
        if isinstance(a, (np.ndarray, pd.Series)):
            a = a.tolist()
        if isinstance(b, (np.ndarray, pd.Series)):
            b = b.tolist()
        if isinstance(a, (set, frozenset, list, tuple)) or isinstance(b, (set, frozenset, list, tuple)):
            try:
                sa = set(a) if not isinstance(a, (set, frozenset)) else set(a)
                sb = set(b) if not isinstance(b, (set, frozenset)) else set(b)
                return len(sa.intersection(sb)) > 0
            except Exception:
                return False
        try:
            return str(a) == str(b)
        except Exception:
            return False

    def _planta_nonempty(x) -> bool:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return False
        if isinstance(x, (np.ndarray, pd.Series)):
            try:
                x = x.tolist()
            except Exception:
                pass
        if isinstance(x, (set, frozenset, list, tuple)):
            return len(x) > 0
        s = str(x).strip()
        return bool(s) and s.lower() not in ("nan", "none")

    def _detect_area_col(df: pd.DataFrame) -> str | None:
        if df is None or df.empty:
            return None
        if "Área" in df.columns:
            return "Área"
        if "Área" in df.columns:
            return "Área"
        for c in df.columns:
            if "area" in str(c).lower():
                return c
        return None

    # ------------------------- validações base -------------------------
    if df_full is None or getattr(df_full, "empty", True):
        fig_empty = go.Figure()
        fig_empty.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22},
                   'x': 0.5, 'xanchor': 'center', 'pad': {'t': 28, 'b': 28}},
            height=420, showlegend=False, font=dict(family="VW Text")
        )
        return fig_empty

    df_rel = _sanitize_object_columns(df_full.copy())

    required = ["Data", "Cadastro", "Cargo"]
    missing = [c for c in required if c not in df_rel.columns]
    if missing:
        return _raise_or_fig(f"Colunas obrigatórias ausentes para o modo Teia (Exec×Exec): {missing}")

    df_rel["Data"] = pd.to_datetime(df_rel["Data"], errors="coerce")
    df_rel = df_rel.dropna(subset=["Data"])
    if df_rel.empty:
        fig_empty = go.Figure()
        fig_empty.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22},
                   'x': 0.5, 'xanchor': 'center', 'pad': {'t': 28, 'b': 28}},
            height=420, showlegend=False, font=dict(family="VW Text")
        )
        return fig_empty

    df_rel["Cadastro"] = df_rel["Cadastro"].map(_safe_hashable).astype(str)

    # normaliza Cargo para E/M
    c = df_rel["Cargo"].astype(str).str.strip()
    cu = c.str.upper()
    df_rel["Cargo"] = np.where(
        cu.str.startswith("E") | cu.str.contains("EXEC"),
        "E",
        np.where(cu.str.startswith("M") | cu.str.contains("MENS"), "M", cu.str[:1])
    )

    # apenas executivos
    df_rel = df_rel[df_rel["Cargo"].eq("E")].copy()
    if df_rel.empty:
        return _raise_or_fig("Nenhum executivo (Cargo=E) encontrado no filtro atual.")

    if "Planta" not in df_rel.columns:
        df_rel["Planta"] = "X"

    # dedup para não inflar presença
    df_rel = df_rel.drop_duplicates(subset=["Data", "Cadastro"], keep="last")

    # ------------------------- adiciona Área (se necessário) -------------------------
    hier = df_hierarquia
    if hier is None:
        hier = globals().get("df_ativos", None)

    area_col = _detect_area_col(df_rel)
    if area_col is None and isinstance(hier, pd.DataFrame) and not hier.empty and "Cadastro" in hier.columns:
        area_col_h = _detect_area_col(hier)
        if area_col_h:
            h = hier[["Cadastro", area_col_h]].copy()
            h["Cadastro"] = h["Cadastro"].map(_safe_hashable).astype(str)
            df_rel = df_rel.merge(h, on="Cadastro", how="left")
            area_col = area_col_h

    if area_col:
        df_rel["ÁreaNorm"] = df_rel[area_col].astype(str).replace({"nan": "", "None": ""})
    else:
        df_rel["ÁreaNorm"] = ""

    # ------------------------- meta (nome/área) -------------------------
    meta_cols = ["Cadastro", "ÁreaNorm"]
    if "Nome" in df_rel.columns:
        meta_cols.insert(1, "Nome")
    meta_src = df_rel.sort_values("Data").drop_duplicates("Cadastro", keep="last")[meta_cols].copy()
    meta_src["Cadastro"] = meta_src["Cadastro"].astype(str)

    name_map = {r["Cadastro"]: ("" if ("Nome" not in r or pd.isna(r.get("Nome"))) else str(r.get("Nome")))
                for _, r in meta_src.iterrows()}
    area_map = {r["Cadastro"]: ("" if pd.isna(r.get("ÁreaNorm")) else str(r.get("ÁreaNorm")))
                for _, r in meta_src.iterrows()}

    exec_ids = sorted(meta_src["Cadastro"].astype(str).unique().tolist())
    if not exec_ids:
        return _raise_or_fig("Nenhum executivo encontrado após consolidar metadados.")

    # ------------------------- mapa de relacionados (exec -> outros execs) -------------------------
    has_area_info = any(_norm_area(area_map.get(e, "")) for e in exec_ids)

    related_e_map: dict[str, list[str]] = {}

    if has_area_info:
        # Pedido: na 2ª seção (Exec×Exec), mostrar a relação de TODOS os executivos
        # com TODOS os demais (não apenas "subordinados" por prefixo de Área).
        # Mantemos Área apenas para ordenação estável.
        def _exec_sort_key(cad: str):
            return (
                _norm_area(area_map.get(cad, "")),
                str(name_map.get(cad, cad)).lower(),
                str(cad)
            )

        for e in exec_ids:
            rels = [e2 for e2 in exec_ids if e2 != e]
            rels.sort(key=_exec_sort_key)
            related_e_map[str(e)] = rels
    else:
        # fallback: top relacionados por co-presença bruta (encontros) — entre executivos
        # (usa a própria base filtrada e valida planta, se disponível)
        # estratégia: contar co-presença simples por dia (sem %), depois escolher top.
        df_day = df_rel[["Data", "Cadastro", "Planta"]].copy()
        df_day["Cadastro"] = df_day["Cadastro"].astype(str)

        from collections import defaultdict
        counts = defaultdict(int)
        for day, g in df_day.groupby("Data"):
            g = g.sort_values("Cadastro").drop_duplicates(subset=["Cadastro"], keep="last")
            ids = g["Cadastro"].astype(str).tolist()
            planta_by = dict(zip(g["Cadastro"].astype(str), g["Planta"].tolist()))

            for i in range(len(ids)):
                for j in range(len(ids)):
                    if i == j:
                        continue
                    a, b = ids[i], ids[j]
                    if validar_mesma_planta and not _planta_ok(planta_by.get(a, None), planta_by.get(b, None)):
                        continue
                    counts[(a, b)] += 1

        for e in exec_ids:
            rels = [(b, counts.get((e, b), 0)) for b in exec_ids if b != e]
            rels.sort(key=lambda x: x[1], reverse=True)
            # fallback: se max_eixos definido, respeita; senão, usa todos com contagem>0
            if max_eixos is None:
                rel_list = [b for b, v in rels if v > 0]
            else:
                rel_list = [b for b, v in rels[:int(max_eixos)]]
            related_e_map[str(e)] = rel_list

    # ------------------------- calcula matriz % (mesma lógica do heatmap Exec×Exec) -------------------------
    idx_map = {cad: i for i, cad in enumerate(exec_ids)}
    N = len(exec_ids)

    encontros_same = np.zeros((N, N), dtype=float)
    dias_pres = np.zeros((N,), dtype=float)

    df_day = df_rel[["Data", "Cadastro", "Planta"]].copy()
    df_day["Cadastro"] = df_day["Cadastro"].astype(str)

    for day, g in df_day.groupby("Data"):
        g = g.sort_values("Cadastro").drop_duplicates(subset=["Cadastro"], keep="last")
        planta_by = dict(zip(g["Cadastro"].astype(str), g["Planta"].tolist()))
        present = set(planta_by.keys())

        sets_list = []
        pres_mask = np.zeros((N,), dtype=bool)

        # prepara vetores por ordem exec_ids
        for i, cad in enumerate(exec_ids):
            pl = planta_by.get(cad, None)
            if validar_mesma_planta:
                ok = _planta_nonempty(pl)
            else:
                ok = cad in present
            pres_mask[i] = bool(ok)
            sets_list.append(pl)

        dias_pres += pres_mask.astype(float)

        if not np.any(pres_mask):
            continue

        # computa co-presença (mesma planta se validar)
        if validar_mesma_planta:
            # matriz booleana NxN: interseção de plantas e ambos presentes
            same = np.zeros((N, N), dtype=bool)
            for i in range(N):
                if not pres_mask[i]:
                    continue
                for j in range(N):
                    if not pres_mask[j]:
                        continue
                    if i == j:
                        continue
                    if _planta_ok(sets_list[i], sets_list[j]):
                        same[i, j] = True
            valid = same
        else:
            # sem planta: basta ambos presentes
            valid = pres_mask[:, None] & pres_mask[None, :]
            np.fill_diagonal(valid, False)

        encontros_same += valid.astype(float)

    denom = dias_pres[:, None]
    with np.errstate(invalid="ignore", divide="ignore"):
        percent = np.divide(
            encontros_same,
            denom,
            out=np.full_like(encontros_same, np.nan, dtype=float),
            where=(denom > 0)
        ) * 100.0

    np.fill_diagonal(percent, np.nan)

    # ------------------------- seleção de executivos exibidos -------------------------
    pres_exec = (df_rel.groupby("Cadastro")["Data"].nunique()
                 .sort_values(ascending=False))
    exec_sel = pres_exec.index.astype(str).tolist()[:max_exec]
    exec_sel = [e for e in exec_sel if e in exec_ids]

    if not exec_sel:
        return _raise_or_fig("Nenhum executivo elegível após cálculo (verifique filtros/período).")

    # ------------------------- subplots polares -------------------------
    n = len(exec_sel)
    cols = max(1, min(int(n_cols), n))
    rows = int(math.ceil(n / cols))

    subplot_titles = []
    for e in exec_sel:
        nm = name_map.get(str(e), str(e))
        subplot_titles.append(f"{_abbr(nm)} ({e})")
    while len(subplot_titles) < rows * cols:
        subplot_titles.append("")

    specs = [[{"type": "polar"} for _ in range(cols)] for _ in range(rows)]
    fig = make_subplots(rows=rows, cols=cols, specs=specs, subplot_titles=subplot_titles)

    # títulos dos subplots: deslocados para cima (yshift positivo)
    try:
        for ann in (fig.layout.annotations or []):
            if getattr(ann, "text", "") and "(" in str(ann.text) and ")" in str(ann.text):
                ann.yshift = 30
    except Exception:
        pass

    FIXED_TICK_MAX = 100.0
    PAD_VISUAL = 8.0
    RANGE_MAX = FIXED_TICK_MAX + PAD_VISUAL

    # ------------------------- 1 radar por executivo -------------------------
    for i, e in enumerate(exec_sel):
        r = (i // cols) + 1
        c = (i % cols) + 1
        e = str(e)

        # eixos = executivos comparáveis pela área (mesma ideia do heatmap)
        rels = list(map(str, related_e_map.get(e, [])))
        if max_eixos is not None:
            rels = rels[:int(max_eixos)]

        if not rels:
            rels = ["Sem executivos comparáveis (Área)"]

        e_idx = idx_map.get(e, None)

        theta = []
        rvals = []
        custom = []

        for pid in rels:
            if pid == "Sem executivos comparáveis (Área)" or e_idx is None or pid not in idx_map:
                p_nome = pid
                pid_show = ""
                pct = 0.0
            else:
                p_nome = name_map.get(pid, pid)
                pid_show = pid
                p_idx = idx_map[pid]
                # percent do HEATMAP: linha = pid (rel), coluna = e
                v = percent[p_idx, e_idx]
                pct = 0.0 if (v is None or (isinstance(v, float) and np.isnan(v))) else float(v)

            if pid_show:
                theta.append(f"{_abbr(p_nome)}<br>{pid_show} — {pct:.1f}%")
            else:
                theta.append(f"{_abbr(p_nome)}")
            rvals.append(float(pct))
            _c_hex = _cor_heatmap(pct)
            custom.append([p_nome, pid, _c_hex, e])

        theta_c = theta + [theta[0]]
        r_c = rvals + [rvals[0]]
        custom_c = custom + [custom[0]]
        _colors = [cd[2] if len(cd) > 2 else '#cccccc' for cd in custom]
        colors_c = _colors + [_colors[0]]

        exec_nome = name_map.get(e, e)


        # cor (igual ao heatmap): linha/preenchimento pelo pct médio; marcadores por pct individual
        _vals_color = [v for v in rvals if isinstance(v, (int, float)) and not np.isnan(v)]
        _pct_mean = float(np.nanmean(_vals_color)) if _vals_color else 0.0
        _line_color = _cor_heatmap(_pct_mean)
        _fill_color = _hex_rgba(_line_color, 0.22)
        # Hover (almoço e geral) no mesmo formato, com cor do ponto (igual ao heatmap)
        hover = (
            "<b>%{customdata[0]}</b> × <b>%{text}</b>"
            "<br><b>%{customdata[3]} x %{customdata[1]} — %{r:.1f}%</b>"
            "<extra></extra>"
        )

        fig.add_trace(
            go.Scatterpolar(
                r=r_c,
                theta=theta_c,
                fill="toself",
                fillcolor=_fill_color,
                mode="lines+markers",
                line=dict(width=2, color=_line_color),
                marker=dict(size=10, line=dict(width=1.5, color='#1a1a1a'), color=colors_c, showscale=False),
                showlegend=False,
                text=[exec_nome] * len(theta_c),
                customdata=custom_c,
                hovertemplate=hover,
            ),
            row=r, col=c
        )

        pol_key = _polar_key(i + 1)

        fig.layout[pol_key].radialaxis.range = [0, RANGE_MAX]
        fig.layout[pol_key].radialaxis.tick0 = 0
        fig.layout[pol_key].radialaxis.dtick = 20
        fig.layout[pol_key].radialaxis.showticklabels = True
        fig.layout[pol_key].radialaxis.tickfont = dict(family="VW Text", size=11)
        fig.layout[pol_key].angularaxis.tickfont = dict(family="VW Text", size=11)
        fig.layout[pol_key].radialaxis.ticksuffix = "%"

    # ------------------------- layout geral -------------------------
    fig.update_layout(
        title={
            'text': titulo,
            'font': {'family': 'VW Text', 'size': 22},
            'x': 0.5, 'xanchor': 'center',
            'pad': {'t': 28, 'b': 20}
        },
        font=dict(family="VW Text"),
        hoverlabel=dict(bgcolor="white", bordercolor="black", font=dict(family="VW Text", color="black")),
        margin=dict(l=70, r=40, t=125, b=60),
        height=max(520, rows * 420),
        showlegend=False,
    )

    if n != 1:
        fig = _ajustar_domains_subplots_polares(fig, rows, cols, n)
    else:
        try:
            fig.layout["polar"].domain.x = [0.15, 0.85]
            fig.layout["polar"].domain.y = [0.12, 0.88]
        except Exception:
            pass

    return fig

def grafico_pentagono_rel_men(df_full: pd.DataFrame, titulo: str):
    _, men_df, _ = _metricas_rel_exec_men(df_full)
    return _radar_from_metrics(men_df.rename(columns={'Men':'Label'}), titulo, 'Label', top_n=5)


def grafico_linhas_relacionamento(df_full: pd.DataFrame, titulo: str):
    _, _, serie = _metricas_rel_exec_men(df_full)
    fig = go.Figure()
    if serie is None or serie.empty:
        fig.update_layout(title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}}, height=320)
        return fig
    serie = serie.sort_values('Semana')
    fig.add_trace(go.Scatter(x=serie['Semana'], y=serie['encontros_semana'], mode='lines+markers', name='Encontros/semana'))
    fig.update_layout(title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}}, height=320, xaxis_title='Semana', yaxis_title='Encontros')
    return fig


def grafico_heatmap_encontros_exec(df_full: pd.DataFrame, titulo: str, validar_mesma_planta: bool = True):
    fig = go.Figure()

    def _final(_fig):
        _fig.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
            height=420
        )
        return _fig

    if df_full is None or df_full.empty:
        return _final(fig)

    dfp = df_full.copy()

    # Sanitização defensiva de chaves (evita: unhashable type: 'numpy.ndarray')
    for _c in ['Cadastro','Cargo','Nome','Planta','Área','Área','Departamento','Setor']:
        if _c in dfp.columns:
            dfp[_c] = dfp[_c].map(_safe_hashable)
    if 'Cadastro' in dfp.columns:
        dfp['Cadastro'] = dfp['Cadastro'].astype(str)
    if 'Cargo' in dfp.columns:
        dfp['Cargo'] = dfp['Cargo'].astype(str)

    dfp['Data'] = pd.to_datetime(dfp['Data'], errors='coerce')
    dfp = dfp.dropna(subset=['Data'])
    if dfp.empty:
        return _final(fig)

    area_col = 'Área' if 'Área' in dfp.columns else ('Área' if 'Área' in dfp.columns else None)

    # ----- apenas executivos -----
    dfE = dfp[dfp['Cargo'].astype(str).eq('E')].copy()
    if dfE.empty:
        return _final(fig)

    # eixo de dias contínuos (para o caminho "sem planta")
    days = pd.date_range(dfp['Data'].min().normalize(), dfp['Data'].max().normalize(), freq='D')

    # presença diária (0/1) por executivo (usado no caminho antigo)
    pres = (
        dfE.groupby(['Data', 'Cadastro'])['Contagem Presença']
            .max().unstack('Cadastro')
            .reindex(days).fillna(0)
    )

    # metadados por executivo (nome, área)
    meta_src = dfE.sort_values('Data').drop_duplicates('Cadastro', keep='last')
    if area_col:
        meta = (meta_src[['Cadastro', 'Nome', 'Cargo', area_col]]
                .set_index('Cadastro')
                .rename(columns={area_col: 'ÁreaNorm'}))
    else:
        meta = meta_src[['Cadastro', 'Nome', 'Cargo']].set_index('Cadastro')
        meta['ÁreaNorm'] = ''

    # =============== NEW: obter 'Local' (ativos > desligados > dfE) ===============
    local_map = {}
    try:
        # prioridades: df_ativos > df_desligados > dfE
        dfd = globals().get('df_ativos', None)
        if isinstance(dfd, pd.DataFrame) and {'Cadastro', 'Local'}.issubset(dfd.columns):
            tmp = dfd[['Cadastro', 'Local']].copy()
            tmp['Cadastro'] = tmp['Cadastro'].astype(str)
            tmp = tmp.dropna(subset=['Cadastro']).drop_duplicates('Cadastro', keep='last')
            local_map.update({str(k): ('' if pd.isna(v) else str(v)) for k, v in zip(tmp['Cadastro'], tmp['Local'])})

        dfa = globals().get('df_desligados', None)
        if isinstance(dfa, pd.DataFrame) and {'Cadastro', 'Local'}.issubset(dfa.columns):
            tmp = dfa[['Cadastro', 'Local']].copy()
            tmp['Cadastro'] = tmp['Cadastro'].astype(str)
            tmp = tmp.dropna(subset=['Cadastro']).drop_duplicates('Cadastro', keep='last')
            # só preenche se ainda não havia no map (ativos tem prioridade)
            for k, v in zip(tmp['Cadastro'], tmp['Local']):
                k = str(k)
                if k not in local_map:
                    local_map[k] = '' if pd.isna(v) else str(v)

        # fallback: se dfE tiver Local em alguma linha
        if 'Local' in dfE.columns:
            tmp = dfE[['Cadastro', 'Local']].copy()
            tmp['Cadastro'] = tmp['Cadastro'].astype(str)
            tmp = tmp.dropna(subset=['Cadastro']).drop_duplicates('Cadastro', keep='last')
            for k, v in zip(tmp['Cadastro'], tmp['Local']):
                k = str(k)
                if k not in local_map:
                    local_map[k] = '' if pd.isna(v) else str(v)
    except Exception:
        pass

    def _local_str(cad) -> str:
        val = local_map.get(str(cad), '')
        val = '' if (pd.isna(val) or str(val).strip().lower() in ('nan', 'none')) else str(val).strip()
        return val if val else '—'
    # =====================================================================

    # tenure (tempo de casa) usando caches globais
    global ADM_CACHE, DEM_CACHE
    if 'ADM_CACHE' not in globals():
        ADM_CACHE = None
    if 'DEM_CACHE' not in globals():
        DEM_CACHE = None
    if ADM_CACHE is None or DEM_CACHE is None:
        ADM_CACHE, DEM_CACHE = coletar_admissoes_e_demissoes()

    ref_dt = days.max() if len(days) else pd.Timestamp.today().normalize()
    tenure = {}
    for c in meta.index:
        cid = str(c)
        adm_ts = ADM_CACHE.get(cid, None)
        dem_ts = DEM_CACHE.get(cid, None)
        tenure[cid] = tempo_de_casa_str(adm_ts, ref=ref_dt, dem_ts=dem_ts)

    # helpers p/ ordenar por hierarquia de área
    def _tok(s):
        return [t.strip() for t in str(s).replace('/', '-').split('-') if t.strip()]

    def _akey(a):
        t = _tok(a)
        return (len(t), tuple(t))

    # ordem inicial por área e nome
    cads = sorted(
        meta.index,
        key=lambda c: (_akey(meta.loc[c, 'ÁreaNorm']),
                    str(meta.loc[c, 'Nome']).lower(),
                    str(c))
    )
    if not cads:
        return _final(fig)

    # ========= MATRIZ DE CO-PRESENÇA =========
    if validar_mesma_planta and ('Planta' in dfE.columns):
        df_day = dfE[['Data', 'Cadastro', 'Planta']].copy()
        df_day['Data'] = pd.to_datetime(df_day['Data'], errors='coerce').dt.normalize()
        df_day = df_day.dropna(subset=['Data', 'Cadastro'])

        N = len(cads)
        encontros_same = np.zeros((N, N), dtype=float)
        dias_pres = np.zeros((N,), dtype=float)

        def _to_set(x):
            if x is None or (isinstance(x, float) and pd.isna(x)):
                return frozenset()
            if isinstance(x, (set, frozenset)):
                return frozenset(map(lambda t: str(t).strip().lower(), x))
            if isinstance(x, (list, tuple)):
                return frozenset(map(lambda t: str(t).strip().lower(), x))
            s = str(x).strip()
            if not s:
                return frozenset()
            for sep in ['|', ';', ',', '/', '\\']:
                if sep in s:
                    return frozenset(map(lambda t: t.strip().lower(), s.split(sep)))
            return frozenset([s.lower()])

        for dia, g in df_day.groupby('Data'):
            g = g.sort_values('Cadastro').drop_duplicates(subset=['Cadastro'], keep='last')
            s = g.set_index('Cadastro')['Planta']
            p = s.reindex(cads)
            sets_list = [_to_set(val) for val in p.tolist()]
            pres_mask = np.array([len(st) > 0 for st in sets_list], dtype=bool)
            same = np.array(
                [[(len(sets_list[i]) > 0 and len(sets_list[j]) > 0 and not sets_list[i].isdisjoint(sets_list[j]))
                for j in range(N)]
                for i in range(N)],
                dtype=bool
            )
            valid = same & pres_mask[:, None] & pres_mask[None, :]
            encontros_same += valid.astype(float)
            dias_pres += pres_mask.astype(float)

        denom = dias_pres[:, None]
        with np.errstate(invalid='ignore', divide='ignore'):
            percent = np.divide(
                encontros_same,
                denom,
                out=np.full_like(encontros_same, np.nan, dtype=float),
                where=(denom > 0)
            ) * 100.0
    else:
        A = pres.reindex(columns=cads).fillna(0).to_numpy(dtype=float)
        encontros = A.T @ A
        denom = A.sum(axis=0)[:, None]
        with np.errstate(invalid='ignore', divide='ignore'):
            percent = np.divide(
                encontros,
                denom,
                out=np.full_like(encontros, np.nan, dtype=float),
                where=(denom > 0)
            ) * 100.0

    np.fill_diagonal(percent, np.nan)

    # ordenar dentro de cada área por média de relação com todos (exclui self)
    areas = [meta.loc[c, 'ÁreaNorm'] for c in cads]
    e_scores = []
    for j in range(len(cads)):
        col = percent[:, j].astype(float).copy()
        col[j] = np.nan
        sc = np.nanmean(col) if np.any(~np.isnan(col)) else np.nan
        e_scores.append(-1e9 if np.isnan(sc) else sc)

    order = sorted(
        range(len(cads)),
        key=lambda j: (
            _akey(areas[j]),
            -e_scores[j],
            str(meta.loc[cads[j], 'Nome']).lower(),
            str(cads[j])
        )
    )
    cads = [cads[j] for j in order]
    percent = percent[np.ix_(order, order)]

    # rótulos (incluindo LOCAL)
    def _abbr(n):
        p = str(n).split()
        return "" if not p else " ".join([p[0].capitalize()] + [x[0].upper() + "." for x in p[1:]])

    x_labels = [
        f"{_abbr(meta.loc[c, 'Nome'])}"
        f" <span style='font-size:10px;opacity:.85'><b>({tenure.get(str(c), '—')})</b></span>"
        f"<br>({str(meta.loc[c, 'ÁreaNorm'])})"
        for c in cads
    ]
    y_labels = [
        f"{_abbr(meta.loc[c, 'Nome'])}"
        f" <span style='font-size:10px;opacity:.85'><b>({tenure.get(str(c), '—')})</b></span>"
        for c in cads
    ]

    x_full = [f"{str(meta.loc[c, 'Nome']).title()} ({c})" for c in cads]
    y_full = [f"{str(meta.loc[c, 'Nome']).title()} ({c})" for c in cads]
    custom = [[(x_full[x], y_full[y]) for x in range(len(cads))] for y in range(len(cads))]

    colors = [
        [0.00, "#af3333"], [0.35, "#fc9898"],
        [0.35, "#ffef09"], [0.60, "#fff9a9"],
        [0.60, "#9effd0"], [1.00, "#267951"]
    ]
    text = np.vectorize(lambda v: "" if (v is None or (isinstance(v, float) and np.isnan(v))) else f"{v:.1f}%")(percent)

    hover_title = "Encontros" if validar_mesma_planta and ('Planta' in dfE.columns) else "Encontros (co-presença)"

    fig = go.Figure(data=go.Heatmap(
        z=percent,
        x=x_labels,
        y=y_labels,
        customdata=custom,
        colorscale=colors,
        zmin=0, zmax=100,
        text=text, texttemplate="%{text}",
        hovertemplate=(
            "<b>%{customdata[1]}</b> × <b>%{customdata[0]}</b><br>"
            f"{hover_title}: %{{z:.1f}}%<extra></extra>"
        ),
        showscale=False,
        hoverongaps=False,
        hoverlabel=dict(bgcolor="white", font=dict(color="black"))
    ))

    linhas = max(len(y_labels), 1)
    altura = max(460, 40 + 22 * max(linhas, 10))
    fig.update_layout(
        title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
        xaxis_title="", yaxis_title="", height=altura,
        margin=dict(l=200, r=40, t=70, b=150)
    )
    fig.update_xaxes(tickangle=-45, automargin=True, autorange="reversed")
    fig.update_yaxes(autorange="reversed", automargin=True)

    return fig


def obter_heranca_do_cadastro(cad_exec: str, df_dept: pd.DataFrame, her_df: pd.DataFrame) -> str | None:
    if cad_exec is None:
        return None
    cad_exec = str(cad_exec)

    heranca = None
    if 'Cadastro' in df_dept.columns:
        linha = df_dept[df_dept['Cadastro'].astype(str).eq(cad_exec)]
        if not linha.empty:
            if 'Heranca' in df_dept.columns and pd.notna(linha.iloc[0].get('Heranca', None)):
                heranca = str(linha.iloc[0]['Heranca']).strip()
            elif 'Área' in df_dept.columns and pd.notna(linha.iloc[0].get('Área', None)):
                area_exec = str(linha.iloc[0]['Área']).strip()
                # tenta mapear área -> herança na tabela her
                if 'Área' in her_df.columns and 'Heranca' in her_df.columns:
                    h = her_df.loc[her_df['Área'].astype(str).eq(area_exec), 'Heranca']
                    if not h.empty:
                        heranca = str(h.iloc[0]).strip()
    return heranca


def obter_area_do_cadastro(cad: str, df_dept: pd.DataFrame) -> str | None:
    if not cad or df_dept is None or df_dept.empty:
        return None
    cad = str(cad)
    linha = df_dept[df_dept['Cadastro'].astype(str).eq(cad)]
    if linha.empty:
        return None
    area = linha.iloc[-1].get('Área', None)
    return None if pd.isna(area) else str(area).strip()


def cache_key(area_base, subarea_sel, desligados_area, viajam_area, i0, i1):
    return (
        str(area_base or ""),
        str(subarea_sel or ""),
        str(desligados_area or ""),
        str(viajam_area or ""),
        int(i0 if isinstance(i0, (int, np.integer)) else 0),
        int(i1 if isinstance(i1, (int, np.integer)) else 0),
    )


def is_valid(val, options):
    if val is None or options in (None, no_update):
        return False
    return any(o.get("value") == val for o in options)


def mini_tabela_gptw(area_label: str | None, anos=(2023, 2024, 2025), incluir_macro=True):
    # === estilos ===
    wrapper_style = {
        'marginLeft': 'auto',
        'margin': '0 auto',
        'marginTop': '0',
    }
    th = {'padding':'4px 8px','border':'1px solid #bbb','backgroundColor':'#f5f5f5',
        'fontWeight':'bold','textAlign':'center','fontFamily':'VW Text', 'fontSize':'30px'}
    td = {'padding':'4px 8px','border':'1px solid #bbb','textAlign':'center',
        'fontFamily':'VW Text'}
    td_first = {**td, 'fontWeight':'bold', 'fontSize':'20px', 'fontFamily': 'VW Text'}  # destaque 1ª linha

    # setas
    ARROW_UP = '▲'
    ARROW_DOWN = '▼'
    ARROW_NONE = '—'
    arrow_style_base = {'marginLeft':'6px', 'fontSize':'12px', 'opacity':0.9}
    color_up = '#0a7f3f'
    color_down = '#b00020'
    color_none = '#999'

    # === anos na ordem que você quer exibir ===
    anos_ord = list(anos)[::]
    anos_str_ord = [str(a) for a in anos_ord]

    # >>> NOVO: detectar se a ordem exibida é crescente no tempo
    is_asc_time = anos_ord == sorted(anos_ord)

    def _fmt(v):
        if pd.isna(v): 
            return '—'
        try:
            v = float(v)
            return str(int(v)) if v.is_integer() else f'{v:.1f}'
        except Exception:
            return str(v)

    def _arrow(curr, prox):
        if curr is None or prox is None or pd.isna(curr) or pd.isna(prox):
            return (ARROW_NONE, color_none)
        try:
            c = float(curr); p = float(prox)
        except Exception:
            return (ARROW_NONE, color_none)
        if c > p:
            return (ARROW_UP, color_up)
        elif c < p:
            return (ARROW_DOWN, color_down)
        else:
            return (ARROW_NONE, color_none)

    # Sem área ⭢ placeholder
    if not area_label:
        tabela = html.Table([
            html.Thead(html.Tr([html.Th('Área', style=th), *[html.Th(str(a), style=th) for a in anos_ord]])),
            html.Tbody(html.Tr([html.Td('-', style={**td, 'textAlign':'left'}),
                                *[html.Td('—', style=td) for _ in anos_ord]]))
        ], style={'borderCollapse':'collapse','fontSize':'14px','boxShadow':'0 2px 6px rgba(0,0,0,0.10)'})
        return html.Div(tabela, style=wrapper_style)

    df = globals().get('df_gptw', None)
    if df is None or getattr(df, 'empty', True):
        tabela = html.Table([
            html.Thead(html.Tr([html.Th('Área', style=th), *[html.Th(str(a), style=th) for a in anos_ord]])),
            html.Tbody(html.Tr([html.Td(area_label, style={**td, 'textAlign':'left'}),
                                *[html.Td('—', style=td) for _ in anos_ord]]))
        ], style={'borderCollapse':'collapse','fontSize':'14px','boxShadow':'0 2px 6px rgba(0,0,0,0.10)'}
        )
        return html.Div(tabela, style=wrapper_style, title='df_gptw vazio/ausente')

    df = df.copy()
    # Coluna de área
    area_col = 'Área' if 'Área' in df.columns else ('Área' if 'Área' in df.columns else None)
    if area_col is None:
        return html.Div("df_gptw sem coluna Área/Área", style={**wrapper_style, 'opacity':0.7, 'fontSize':'12px'})

    # Detecta formato: largo (colunas "2023","2024","2025") x longo (Ano + Nota)
    anos_str_set = set(anos_str_ord)
    is_largo = anos_str_set.issubset(set(df.columns))

    if is_largo:
        cols = [area_col] + anos_str_ord
        tmp = df[cols].copy()
        for c in anos_str_ord:
            tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
        piv = (tmp.groupby(area_col)[anos_str_ord].mean())
        piv.columns = [int(c) for c in piv.columns]
        piv = piv.reindex(columns=anos_ord)
    else:
        cand_nota = ['Nota','nota','Score','Pontuação','Pontuacao','Índice','Indice','Valor','Value']
        nota_col = next((c for c in cand_nota if c in df.columns), None)
        if 'Ano' not in df.columns or nota_col is None:
            return html.Div("df_gptw precisa de colunas 'Ano' e 'Nota/Score/Pontuação'", 
                            style={**wrapper_style, 'opacity':0.7, 'fontSize':'12px'})
        df['Ano'] = pd.to_numeric(df['Ano'], errors='coerce')
        df[nota_col] = pd.to_numeric(df[nota_col], errors='coerce')
        piv = (df.groupby([area_col, 'Ano'])[nota_col]
                .mean().unstack('Ano'))
        piv = piv.reindex(columns=anos_ord)

    piv.index = piv.index.astype(str).str.strip()
    alvo = str(area_label).strip()

    # Descobre subáreas pela herança (se disponível)
    subareas = []
    her_df = globals().get('her', None)
    if isinstance(her_df, pd.DataFrame) and {'Área','Heranca'}.issubset(her_df.columns):
        her_df = her_df.copy()
        her_df['Área'] = her_df['Área'].astype(str).str.strip()
        herancas = her_df.loc[her_df['Área'].eq(alvo), 'Heranca'].dropna().unique()
        cand = her_df[her_df['Área'].str.startswith(alvo) & her_df['Área'].ne(alvo)]
        if len(herancas) > 0:
            cand = cand[cand['Heranca'].isin(herancas)]
        subareas = cand['Área'].dropna().unique().tolist()

    # Fallback: prefixo
    if not subareas:
        all_areas = piv.index.tolist()
        subareas = [a for a in all_areas if a != alvo and a.startswith(alvo)]

    def _akey(a):
        toks = [t.strip() for t in str(a).replace('/', '-').split('-') if t.strip()]
        return (len(toks), tuple(toks))

    linhas = []
    if incluir_macro:
        linhas.append(alvo)
    linhas.extend(sorted(subareas, key=_akey))
    if not linhas:
        linhas = [alvo]

    body_rows = []
    for idx, area_nome in enumerate(linhas):
        row = piv.loc[area_nome] if area_nome in piv.index else pd.Series(index=anos_ord, dtype=float)
        td_this = td_first if idx == 0 else td
        tds = [html.Td(area_nome, style={**td_this, 'textAlign':'left'})]
        for j, a in enumerate(anos_ord):
            val = row.get(a) if isinstance(row, pd.Series) else None

            # >>> NOVO: escolher a "vizinha temporal" correta
            ref_idx = (j - 1) if is_asc_time else (j + 1)
            prox_val = row.get(anos_ord[ref_idx]) if (0 <= ref_idx < len(anos_ord)) else None

            val_txt = _fmt(val)
            sym, color = _arrow(val, prox_val)
            arrow_span = html.Span(sym, style={**arrow_style_base, 'color': color})
            kids = [html.Span(val_txt)]
            if val_txt != '—':
                kids.append(arrow_span)
            tds.append(html.Td(kids, style=td_this))
        body_rows.append(html.Tr(tds))

    tabela = html.Table([
        html.Thead(html.Tr([
            html.Th('Área', style=th),
            *[html.Th(str(a), style=th) for a in anos_ord],
        ])),
        html.Tbody(body_rows)
    ], style={
        'borderCollapse':'collapse',
        'fontSize':'13px',
        'boxShadow':'0px 2px 6px rgba(0,0,0,0.10)',
        'maxWidth':'560px'
    })

    return html.Div(tabela, style=wrapper_style)


def build_dataframe_almoco_area(area, subarea, her_df, df_almoco_local, df_dept, dt_ini=None, dt_fim=None):
    """
    Constrói um DF diário por pessoa para ALMOÇO (11h–14h), com 'Planta' = conjunto de clusters de almoço.
    Dois colaboradores contam como “encontro” se tiverem interseção de clusters (<=3 min no mesmo dia/chave).
    """
    def empty():
        return pd.DataFrame(columns=['Data','Cadastro','Nome','Cargo','Área','Departamento','Planta','Contagem Presença'])

    if df_almoco_local is None or df_almoco_local.empty:
        return empty()

    # Recebendo os últimos 3 mêses:
    df_almoco_local = df_almoco_local[df_almoco_local['Data'] >= (df_almoco_local['Data'].max()) - pd.DateOffset(months=3)]

    # --- normalização leve de dept/her ---
    for _df in (her_df, df_dept):
        for c in ('Heranca','Área','Departamento','Cadastro'):
            if c in _df.columns:
                _df[c] = _df[c].astype(str).str.strip()

    alvo = subarea or area

    # base_dept para pegar lista de cadastros da área/subárea (mesma lógica da função diária)
    base_dept = pd.DataFrame(columns=['Cadastro','Área','Departamento'])
    mask_area_ok = False
    if 'Área' in df_dept.columns:
        if subarea:
            mask_area = df_dept['Área'].eq(alvo)
        else:
            mask_area = df_dept['Área'].str.contains(str(area), case=False, na=False)
        base_dept = df_dept.loc[mask_area, ['Cadastro','Área','Departamento']].drop_duplicates()
        mask_area_ok = not base_dept.empty

    if not mask_area_ok:
        # tenta por herança
        herancas = (her_df.loc[her_df['Área'].eq(alvo), 'Heranca'].dropna().unique()
                    if {'Área','Heranca'}.issubset(her_df.columns) else [])
        if len(herancas) == 0 and {'Área','Heranca'}.issubset(her_df.columns):
            herancas = (her_df[her_df['Área'].str.contains(str(alvo), case=False, na=False)]
                            ['Heranca'].dropna().unique())
        if len(herancas) == 0:
            return empty()
        starts_any = lambda h: any(str(h).startswith(p) for p in herancas)
        if 'Heranca' in df_dept.columns:
            mask = df_dept['Heranca'].apply(starts_any)
            base_dept = df_dept.loc[mask, ['Cadastro','Área','Departamento']].drop_duplicates()

    if base_dept.empty:
        return empty()

    # nomes/cargos
    ref_cols = ['Cadastro','Nome','Cargo']
    df_ref = (df_semana_merged[ref_cols].drop_duplicates()
            if 'df_semana_merged' in globals() else pd.DataFrame(columns=ref_cols))

    chaves = base_dept.merge(df_ref, on='Cadastro', how='left')
    cad_ok = set(chaves['Cadastro'].astype(str))

    # --- prepara df_almoco
    df = df_almoco_local.copy()
    df.columns = df.columns.str.strip()
    for c in ('Nome','Dir','Descricao Coletor','Cadastro'):
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
    df = df[df['Cadastro'].astype(str).isin(cad_ok)]

    # Data/Hora
    df['Data'] = pd.to_datetime(df['Data'], dayfirst=True, errors='coerce')
    horas_raw = df['Hora'].astype(str).str.strip()
    df['Hora'] = pd.to_datetime(horas_raw, format='%H:%M:%S', errors='coerce')
    # fallback HH:MM
    df.loc[df['Hora'].isna(), 'Hora'] = pd.to_datetime(horas_raw, format='%H:%M', errors='coerce')
    df = df.dropna(subset=['Data','Hora','Dir'])
    df['Data_norm'] = df['Data'].dt.normalize()

    # janela de almoço
    h_ini = pd.to_datetime('11:00:00', format='%H:%M:%S').time()
    h_fim = pd.to_datetime('14:00:00', format='%H:%M:%S').time()
    df = df[(df['Hora'].dt.time >= h_ini) & (df['Hora'].dt.time <= h_fim)]

    # mantemos só E e S
    df = df[df['Dir'].isin(['E','S'])]
    
    if dt_ini is not None and dt_fim is not None:
        dt_ini = pd.Timestamp(dt_ini).normalize()
        dt_fim = pd.Timestamp(dt_fim).normalize()
        df = df[(df['Data'] >= dt_ini) & (df['Data'] <= dt_fim)]
    if df.empty:
        return empty()

    # helper: clusterizar por janela de 3 min
    def cluster_por_janela(g, chave_prefix, janela_min=3):
        """Recebe um group (ordenado por Hora) e devolve série com cluster IDs."""
        g = g.sort_values('Hora').copy()
        tmins = g['Hora'].dt.hour * 60 + g['Hora'].dt.minute + g['Hora'].dt.second / 60.0
        # nova etiqueta quando diferença para anterior >= janela_min
        novo = (tmins.diff().fillna(9999) >= janela_min).astype(int).cumsum()
        g['_cluster'] = g['Data_norm'].dt.strftime('%Y-%m-%d') + '|' + chave_prefix + '|' + novo.astype(str)
        return g

    # E: agrupar por (Data_norm, Dir='E', Descricao Coletor)
    # === CLUSTERS ===
    parts = []

    def normalizar_coletor(desc):
        """Normaliza descrição do coletor para o agrupamento:
        - Se contiver 'ref' ⭢ exige igualdade exata
        - Se contiver 'port' ⭢ todas com 'port' são tratadas como iguais
        - Caso contrário ⭢ mantém o texto original
        """
        if not isinstance(desc, str):
            return 'NA'
        desc_low = desc.lower().strip()
        if 'ref' in desc_low:
            return desc_low  # igualdade exata
        elif 'port' in desc_low:
            return 'portaria'  # agrupa tudo com 'port'
        else:
            return desc_low  # mantém outras descrições

    # E: agrupar por (Data_norm, Dir='E', coletor normalizado)
    if {'Descricao Coletor'}.issubset(df.columns):
        df['Descricao Agrupada'] = df['Descricao Coletor'].apply(normalizar_coletor)
        for (d, _dir, col_norm), g in df[df['Dir'] == 'E'].groupby(['Data_norm', 'Dir', 'Descricao Agrupada'], dropna=False):
            chave = f"E|{str(col_norm) if pd.notna(col_norm) else 'NA'}"
            # define janela de acordo com coletor
            janela_min = 1 if str(col_norm).startswith('ref') else 3
            parts.append(cluster_por_janela(g, chave, janela_min))
    else:
        for (d, _dir), g in df[df['Dir'] == 'E'].groupby(['Data_norm', 'Dir']):
            parts.append(cluster_por_janela(g, "E", 3))

    # S: agrupar por (Data_norm, Dir='S') – saída da empresa
    for (d, _dir), g in df[df['Dir']=='S'].groupby(['Data_norm','Dir']):
        chave = f"S"
        parts.append(cluster_por_janela(g, chave))

    if not parts:
        return empty()

    dfc = pd.concat(parts, ignore_index=True)
    # set de clusters por pessoa/dia
    clust = (dfc.groupby(['Data_norm','Cadastro'])['_cluster']
                .apply(lambda s: frozenset(s.dropna().unique().tolist()))
                .reset_index())
    clust.rename(columns={'Data_norm':'Data'}, inplace=True)

    # Junta metadados
    out = clust.merge(chaves, on='Cadastro', how='left')

    # Calcular presença ainda com '_cluster'
    out['Contagem Presença'] = out['_cluster'].apply(lambda st: int(len(st) > 0))

    # Só então renomear para 'Planta'
    out = out.rename(columns={'_cluster':'Planta'})
    out['Data'] = pd.to_datetime(out['Data'], errors='coerce')
    out['Planta'] = out['Planta'].apply(lambda x: x if isinstance(x, (set, frozenset)) else frozenset())

    # colunas finais garantidas
    for c in ['Nome','Cargo','Área','Departamento']:
        if c not in out.columns: out[c] = np.nan

    # ==================== NOVO FILTRO DINÂMICO: mantém apenas dias com encontros reais ====================
    # 1. Mantém apenas dias em que houve 2 ou mais pessoas (encontros possíveis)
    pres_por_dia = out.groupby('Data')['Cadastro'].nunique().reset_index(name='qtde_pessoas')
    dias_com_encontros = set(pres_por_dia.loc[pres_por_dia['qtde_pessoas'] > 1, 'Data'])
    out = out[out['Data'].isin(dias_com_encontros)]

    # 2. (Opcional) Se o usuário quiser focar em um par específico, faz o recorte
    pessoas_alvo = globals().get("PESSOAS_ALVO", None)
    if isinstance(pessoas_alvo, (list, tuple)) and len(pessoas_alvo) == 2:
        c1, c2 = map(str, pessoas_alvo)
        dias_c1 = set(out.loc[out['Cadastro'] == c1, 'Data'])
        dias_c2 = set(out.loc[out['Cadastro'] == c2, 'Data'])
        dias_comuns = dias_c1 & dias_c2
        out = out[out['Data'].isin(dias_comuns)]

    return out[['Data','Cadastro','Nome','Cargo','Área','Departamento','Planta','Contagem Presença']]


def painel_turnover_gptw_centralizado(tbl_turn, tbl_gptw):
    card_style = {
        'backgroundColor': '#fff',
        'border': '1px solid #dcdcdc',
        'borderRadius': '12px',
        'padding': '12px',
        'boxShadow': '0 2px 6px rgba(0,0,0,0.08)',
        'flex': '0 1 520px',
        'maxWidth': '560px'
    }
    title = lambda t: html.H4(t, style={'textAlign': 'center', 'margin': '0 0 8px'})

    return html.Div([
        html.Div([title("Headcount & Turnover"), tbl_turn], style=card_style),  # ← AQUI
        html.Div([title("GPTW"),                 tbl_gptw],  style=card_style),
    ], style={
        'display': 'flex',
        'justifyContent': 'center',
        'alignItems': 'flex-start',
        'gap': '24px',
        'flexWrap': 'wrap',
        'maxWidth': '1200px',
        'margin': '8px auto 16px'
    })


def _fmt_num(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return '—'
    try:
        v = float(v)
        return str(int(v)) if v.is_integer() else f'{v:.1f}'
    except Exception:
        return str(v)


def _arrow(curr, prev):
    # seta em função do (curr vs prev); se não dá pra comparar ⭢ '—'
    if curr is None or prev is None: return (ARROW_NONE, PAD_STY['color_none'])
    if (isinstance(curr, float) and np.isnan(curr)) or (isinstance(prev, float) and np.isnan(prev)):
        return (ARROW_NONE, PAD_STY['color_none'])
    try:
        c = float(curr); p = float(prev)
    except Exception:
        return (ARROW_NONE, PAD_STY['color_none'])
    if c > p:  return (ARROW_UP,   PAD_STY['color_up'])
    if c < p:  return (ARROW_DOWN, PAD_STY['color_down'])
    return (ARROW_NONE, PAD_STY['color_none'])


def _areas_macro_e_subareas(area_label, piv_index, her_df=None):
    """Devolve [macro, subáreas...] usando SEMPRE o universo vindo do piv_index.

    - Se existir her_df (catálogo de áreas/herança), ele é usado para
      identificar subáreas "oficiais".
    - Além disso, QUALQUER área presente em piv_index que comece com
      a macro (ex.: 'E-RL') é acrescentada como subárea.
    Assim, quando o piv_index vem do GPTW, a lista final de áreas
    será exatamente a do GPTW (apenas reordenada de forma hierárquica).
    """
    alvo = (area_label or "").strip()
    if not alvo:  # sem área ⭢ só placeholder
        return ['—']

    # universo de áreas do pivot (GPTW ou df_head)
    all_areas = None
    if isinstance(piv_index, (list, pd.Index)):
        all_areas = [str(a).strip() for a in list(piv_index)]

    subareas = []

    # 1) tenta coletar subáreas por herança (igual ao mini_tabela_gptw),
    #    mas restringindo ao universo do pivot, se existir
    if isinstance(her_df, pd.DataFrame) and {'Área','Heranca'}.issubset(her_df.columns):
        her_df_loc = her_df.copy()
        her_df_loc['Área'] = her_df_loc['Área'].astype(str).str.strip()
        herancas = her_df_loc.loc[her_df_loc['Área'].eq(alvo), 'Heranca'].dropna().unique()
        cand = her_df_loc[her_df_loc['Área'].str.startswith(alvo) & her_df_loc['Área'].ne(alvo)]
        if len(herancas) > 0:
            cand = cand[cand['Heranca'].isin(herancas)]
        subareas = cand['Área'].dropna().astype(str).str.strip().unique().tolist()

        # se temos um universo de áreas vindo do pivot, mantém só as que existem nele
        if all_areas is not None:
            subareas = [a for a in subareas if a in all_areas]

    # 2) COMPLEMENTO pelo índice do pivot (GPTW / headcount):
    #    qualquer área do pivot que:
    #       - não seja o próprio macro
    #       - comece com o macro
    #       - ainda não esteja em subareas
    if all_areas is not None:
        extra = [
            a for a in all_areas
            if a != alvo and a.startswith(alvo) and a not in subareas
        ]
        subareas.extend(extra)

    # 3) ordena hierarquicamente (mesma lógica que já existia)
    def _akey(a):
        toks = [t.strip() for t in str(a).replace('/', '-').split('-') if t.strip()]
        return (len(toks), tuple(toks))

    linhas = [alvo] + sorted(subareas, key=_akey)

    # 4) se tiver universo vindo do pivot, garante que só devolve coisas dele
    if all_areas is not None:
        return [a for a in linhas if a in all_areas]
    return linhas


def _areas_turnover_filtradas(
    area_label: str,
    df_head: pd.DataFrame,
    df_gptw: pd.DataFrame | None = None,
    area_col_head: str = "Área",
) -> list[str]:
    """
    Linhas da tabela Headcount & Turnover alinhadas ao universo de Áreas do GPTW.

    Regras:
    - Usa SEMPRE o universo de áreas do GPTW para montar [macro + subáreas].
    - Se o GPTW não existir/estiver vazio, cai para o universo do Headcount.
    - Mesmo que uma subárea não exista no Headcount, ela é listada (valores ficam "—").
    """
    alvo = (area_label or "").strip()
    if not alvo:
        return ['—']

    def _norm_cols(df):
        return { ''.join(ch for ch in unicodedata.normalize('NFKD', c) if not unicodedata.combining(ch)).lower(): c
                 for c in df.columns }

    # 1) Universo do GPTW (preferencial)
    gptw_index = None
    if isinstance(df_gptw, pd.DataFrame) and not df_gptw.empty:
        cmap = _norm_cols(df_gptw)
        # procura por "area" sem acento; aceita variantes como "área", "AREA", etc.
        area_key = None
        for k, orig in cmap.items():
            if 'area' == re.sub(r'[^a-z0-9]+','', k):  # nome exato "area"
                area_key = orig
                break
        if area_key is None:
            # também aceita col que contenha "area" como substring
            for k, orig in cmap.items():
                if 'area' in re.sub(r'[^a-z0-9]+','', k):
                    area_key = orig
                    break
        if area_key:
            gptw_index = pd.Index(
                df_gptw[area_key].astype(str).str.strip().dropna().unique()
            )

    # 2) Fallback: universo do Headcount
    head_index = None
    if isinstance(df_head, pd.DataFrame) and not df_head.empty:
        col_head = None
        cmap_h = _norm_cols(df_head)
        # tenta coluna passada, senão busca "area" por normalização
        if area_col_head in df_head.columns:
            col_head = area_col_head
        else:
            for k, orig in cmap_h.items():
                if 'area' == re.sub(r'[^a-z0-9]+','', k):
                    col_head = orig
                    break
            if col_head is None:
                for k, orig in cmap_h.items():
                    if 'area' in re.sub(r'[^a-z0-9]+','', k):
                        col_head = orig
                        break
        if col_head:
            head_index = pd.Index(
                df_head[col_head].astype(str).str.strip().dropna().unique()
            )

    dominio = gptw_index if (gptw_index is not None and len(gptw_index) > 0) else (head_index if head_index is not None else pd.Index([]))

    # 3) Constrói [macro, subáreas...] respeitando a hierarquia, restrito ao domínio escolhido
    her_df = globals().get('her', None)
    areas = _areas_macro_e_subareas(alvo, dominio, her_df=her_df)

    # 3.5) Restringe explicitamente ao domínio escolhido (evita subáreas do Headcount)
    if isinstance(dominio, (list, pd.Index)) and len(dominio) > 0:
        dom_set = set([str(a).strip() for a in list(dominio)])
        areas = [alvo] + [a for a in areas if a != alvo and str(a).strip() in dom_set]

    # 4) Garante que a macro esteja presente no topo, mesmo se o domínio não a contiver
    if alvo not in areas:
        areas = [alvo] + [a for a in areas if a != alvo]

    ## HEADCOUNT EXTRAS: adiciona subáreas do Headcount que compartilham prefixo
    ## (e mesma herança, se 'her' existir), mesmo que não estejam no GPTW
    try:
        extras = []
        if head_index is not None and len(head_index) > 0:
            head_list = [str(a).strip() for a in list(head_index)]
            # subáreas que começam com a macro
            extras = [a for a in head_list if a != alvo and a.startswith(alvo) and a not in areas]
            # se houver her_df, restringe à mesma herança
            if isinstance(her_df, pd.DataFrame) and {'Área','Heranca'}.issubset(her_df.columns):
                her_loc = her_df.copy()
                her_loc['Área'] = her_loc['Área'].astype(str).str.strip()
                herancas = her_loc.loc[her_loc['Área'].eq(alvo), 'Heranca'].dropna().unique()
                if len(herancas) > 0:
                    valid = set(her_loc.loc[her_loc['Heranca'].isin(herancas), 'Área'].astype(str).str.strip())
                    extras = [a for a in extras if a in valid]
        # ordena extras hierarquicamente
        def _akey(a: str):
            a = str(a).strip()
            toks = [t.strip() for t in a.replace('/', '-').split('-') if t.strip()]
            return (len(toks), tuple(toks))
        areas = areas + sorted(set(extras), key=_akey)
    except Exception:
        pass

    return areas


def tabela_padrao_turnover(area_label: str, anos=(2023, 2024, 2025), incluir_macro=True):
    """
    Versão robusta da tabela "Headcount & Turnover" baseada em df_headcount.

    - Usa o mesmo "visual" da mini_tabela_gptw (altura de célula, fonte VW Text).
    - Mostra área macro + subáreas seguindo o filtro:
        Área começa com area_label e tem mesma Heranca.
    - Apenas HC tem seta (comparando com o ano anterior).
    - E e S aparecem só com o valor.
    """

    # --------- estilos (copiados da mini_tabela_gptw) ----------
    wrapper_style = {
        'marginLeft': 'auto',
        'margin': '0 auto',
        'marginTop': '0',
    }
    th = {
        'padding': '4px 8px',
        'border': '1px solid #bbb',
        'backgroundColor': '#f5f5f5',
        'fontWeight': 'bold',
        'textAlign': 'center',
        'fontFamily': 'VW Text',
        'fontSize': '30px',
    }
    td = {
        'padding': '4px 8px',
        'border': '1px solid #bbb',
        'textAlign': 'center',
        'fontFamily': 'VW Text',
    }
    # 1ª linha (área macro): maior e em negrito
    td_first = {**td, 'fontWeight': 'bold', 'fontSize': '20px'}
    # subáreas: estilo "normal" da GPTW
    td_sub = {
        **td,
        "fontSize": "13px",      # fonte menor
    }
    # E / S: fonte um pouco menor
    td_es = {**td, 'fontSize': '12px'}

    table_style = {
        'borderCollapse': 'collapse',
        'fontSize': '14px',  # mesma fonte base da GPTW
        'boxShadow': '0 2px 6px rgba(0,0,0,0.10)',
    }

    # setas (só para HC)
    arrow_style_base = {'marginLeft': '6px', 'fontSize': '12px', 'opacity': 0.7}
    color_up, color_down, color_none = '#0a7f3f', '#b00020', '#999999'

    def _arrow_from_diff(curr, prev):
        """Retorna (símbolo, cor) comparando valor atual com anterior."""
        if curr is None or prev is None:
            return '', None
        try:
            c = float(curr); p = float(prev)
        except Exception:
            return '', None
        if c > p:
            return ARROW_UP, color_up
        if c < p:
            return ARROW_DOWN, color_down
        return ARROW_NONE, color_none

    anos_ord = list(anos)

    # --------- placeholder se não tem área ----------
    if not area_label:
        header_cells = [html.Th('Área', style=th)]
        for ano in anos_ord:
            header_cells.append(html.Th(str(ano), style=th))
            header_cells.append(html.Th('E', style=th))
            header_cells.append(html.Th('S', style=th))
        thead = html.Thead(html.Tr(header_cells))

        row_cells = [html.Td('-', style={**td, 'textAlign': 'left'})]
        for _ in anos_ord:
            row_cells.append(html.Td('—', style=td))
            row_cells.append(html.Td('—', style=td))
            row_cells.append(html.Td('—', style=td))

        tbody = html.Tbody(html.Tr(row_cells))
        table = html.Table([thead, tbody], style=table_style)
        return html.Div(table, style=wrapper_style)

    # --------- pega df_headcount ----------
    df_head = globals().get('df_headcount', None)
    if not isinstance(df_head, pd.DataFrame) or df_head.empty:
        thead = html.Thead(html.Tr([html.Th('Headcount & Turnover', style=th)]))
        tbody = html.Tbody(
            html.Tr([html.Td('Sem dados de Headcount & Turnover.', style=td)])
        )
        table = html.Table([thead, tbody], style=table_style)
        return html.Div(table, style=wrapper_style)

    df_head = df_head.copy()

    # --------- acha coluna de área de forma flexível ----------
    area_col = None
    for cand in ('Área', 'Área', 'area'):
        if cand in df_head.columns:
            area_col = cand
            break
    if area_col is None:
        for c in df_head.columns:
            if 'area' in str(c).lower():
                area_col = c
                break
    if area_col is None:
        thead = html.Thead(
            html.Tr([
                html.Th('df_headcount precisa ter alguma coluna de área', style=th)
            ])
        )
        table = html.Table([thead, html.Tbody([])], style=table_style)
        return html.Div(table, style=wrapper_style)

    df_head[area_col] = df_head[area_col].astype(str).str.strip()

    # --------- descobrir quais LINHAS (áreas) exibir ----------
    df_gptw = globals().get("df_gptw", None)

    linhas_areas = _areas_turnover_filtradas(
        area_label=area_label,
        df_head=df_head,
        df_gptw=df_gptw,
        area_col_head=area_col,
    )

    if not incluir_macro:
        alvo = (area_label or "").strip()
        linhas_areas = [a for a in linhas_areas if a != alvo] or linhas_areas

    if not linhas_areas:
        linhas_areas = [area_label]

    # --------- mapeamento de colunas por ano (HC / E / S) ----------
    def _normalize_col(c: str) -> str:
        return re.sub(r"[^0-9a-zA-Z]+", "", str(c)).lower()

    norm_map = {_normalize_col(c): c for c in df_head.columns}

    def _col_for(ano: int, kind: str):
        year_str = str(ano)
        if kind == 'hc':
            tokens = ('hc', 'headcount', 'colab', 'colaboradores', 'total')
        elif kind == 'e':
            tokens = ('e', 'entrada', 'entradas', 'admissao', 'admissoes')
        else:
            tokens = ('s', 'saida', 'saidas', 'demissao', 'demissoes', 'turnover')

        # padrão 2023hc
        for t in tokens:
            key = f"{year_str}{t}"
            if key in norm_map:
                return norm_map[key]
        # padrão hc2023
        for t in tokens:
            key = f"{t}{year_str}"
            if key in norm_map:
                return norm_map[key]
        # qualquer coluna contendo ano + token
        for norm, orig in norm_map.items():
            if year_str in norm and any(tok in norm for tok in tokens):
                return orig
        return None

    def _fmt_val(v):
        if v is None or (isinstance(v, float) and (pd.isna(v) or not np.isfinite(v))):
            return '—'
        try:
            v = float(v)
            return str(int(v)) if v.is_integer() else f"{v:.1f}"
        except Exception:
            return str(v)

    def _to_float(v):
        try:
            if v is None:
                return None
            if isinstance(v, str) and v.strip() == '':
                return None
            f = float(v)
            if not np.isfinite(f):
                return None
            return f
        except Exception:
            return None

    # --------- cabeçalho: Área | 2023 | E | S | 2024 | E | S | ... ----------
    header_cells = [html.Th('Área', style=th)]
    for ano in anos_ord:
        header_cells.append(html.Th(str(ano), style={**th, 'textAlign': 'center'}))
        header_cells.append(html.Th('E', style={**th, 'fontSize': '16px', 'color': '#008000'}))
        header_cells.append(html.Th('S', style={**th, 'fontSize': '16px', 'color': '#cc0000'}))
    thead = html.Thead(html.Tr(header_cells))

    # --------- corpo ----------

    # --------- filtrar linhas SEM informação (todas as colunas dos anos vêm vazias) ----------
    def _has_info_for_area(area_nome: str) -> bool:
        # pega a série da área; se não existir, não tem info
        serie = df_head[df_head[area_col] == area_nome].iloc[0] if (area_nome in df_head[area_col].values) else None
        if serie is None:
            return False
        for ano in anos:
            hc_col = _col_for(ano, 'hc')
            e_col  = _col_for(ano, 'e')
            s_col  = _col_for(ano, 's')
            vals = []
            if hc_col and hc_col in df_head.columns:
                vals.append(serie.get(hc_col))
            if e_col and e_col in df_head.columns:
                vals.append(serie.get(e_col))
            if s_col and s_col in df_head.columns:
                vals.append(serie.get(s_col))
            # se pelo menos uma dessas células não for NaN, consideramos que tem informação
            if any(pd.notna(v) for v in vals if v is not None):
                return True
        return False

    linhas_areas = [a for a in linhas_areas if _has_info_for_area(a)]
    if not linhas_areas:
        # como fallback absoluto, mantém ao menos a macro selecionada
        linhas_areas = [area_label]


    body_rows = []

    # indexa por área pra facilitar lookup direto (uma linha por área)
    df_idx = df_head.set_index(area_col, drop=False)

    for idx, area_nome in enumerate(linhas_areas):
        est_area = td_first.copy() if idx == 0 else td_sub.copy()

        area_text = str(area_nome)
        first_cell = html.Td(area_text, style={**est_area, 'textAlign': 'left'})
        row_cells = [first_cell]

        # pega a série exata daquela área (sem colapsar por código)
        if area_nome in df_idx.index:
            linha_df = df_idx.loc[area_nome]
            if isinstance(linha_df, pd.DataFrame):
                serie = linha_df.iloc[0]
            else:
                serie = linha_df
        else:
            serie = pd.Series(dtype=float)

        prev_hc_num = None

        for ano in anos_ord:
            hc_col = _col_for(ano, 'hc')
            e_col  = _col_for(ano, 'e')
            s_col  = _col_for(ano, 's')

            hc_raw = serie.get(hc_col) if hc_col else None
            e_raw  = serie.get(e_col)  if e_col  else None
            s_raw  = serie.get(s_col)  if s_col  else None

            hc_num = _to_float(hc_raw)
            e_num  = _to_float(e_raw)
            s_num  = _to_float(s_raw)

            hc_txt = _fmt_val(hc_raw) if hc_col else '—'
            e_txt  = _fmt_val(e_raw)  if e_col  else '—'
            s_txt  = _fmt_val(s_raw)  if s_col  else '—'

            sym_hc, color_hc = _arrow_from_diff(hc_num, prev_hc_num)
            hc_children = [html.Span(hc_txt)]
            if sym_hc and hc_txt != '—':
                hc_children.append(
                    html.Span(sym_hc, style={**arrow_style_base, 'color': color_hc})
                )

            row_cells.append(
                html.Td(
                    hc_children,
                    style={**(td_first if idx == 0 else td_sub), 'textAlign': 'center'},
                )
            )
            row_cells.append(
                html.Td(
                    e_txt,
                    style={**td_es, 'textAlign': 'center', 'color': '#008000'},
                )
            )
            row_cells.append(
                html.Td(
                    s_txt,
                    style={**td_es, 'textAlign': 'center', 'color': '#cc0000'},
                )
            )

            prev_hc_num = hc_num

        body_rows.append(html.Tr(row_cells))

    # --------- DRILLDOWN POR ANO (Turnover) ----------
    # A seta aparece abaixo de cada ANO (colSpan=3). Ao abrir, mostra movimentações do ano
    # para a área analisada (Entrada/Saída) com Data, Cadastro, Nome, De -> Para.

    def _first_col(df_, *cands):
        for c in cands:
            if c in df_.columns:
                return c
        # fallback por contains
        cols = list(df_.columns)
        for c in cols:
            cl = str(c).lower()
            for cand in cands:
                if str(cand).lower() in cl:
                    return c
        return None

    def _mov_turnover_ano(ano: int, area_ref: str) -> pd.DataFrame:
        # Base principal para indivíduos: df_ativos (Área atual) + df_desligados (adm/dem)
        df_dep = globals().get('df_ativos', None)
        if not isinstance(df_dep, pd.DataFrame) or df_dep.empty:
            return pd.DataFrame(columns=['Data','Cadastro','Nome','De','Para','Tipo'])

        df_dep = df_dep.copy()
        cad_col  = _first_col(df_dep, 'Cadastro', 'cadastro', 'Matricula', 'matricula', 'Chapa', 'chapa')
        nome_col = _first_col(df_dep, 'Nome', 'nome')
        area_col_dep = _first_col(df_dep, 'Área', 'Área', 'area')
        sub_col  = _first_col(df_dep, 'Subarea', 'Subárea', 'Subarea', 'subárea', 'subarea', 'sub_area', 'Sub Área')
        adm_dep_col = _first_col(df_dep, 'Data_Admissão', 'Data Admissão', 'DataAdmissao', 'Data_Adm', 'Admissao', 'Admissão')

        if not cad_col or not area_col_dep:
            return pd.DataFrame(columns=['Data','Cadastro','Nome','De','Para','Tipo'])

        # normaliza strings
        df_dep[cad_col] = df_dep[cad_col].astype(str).str.strip()
        df_dep[area_col_dep] = df_dep[area_col_dep].astype(str).str.strip()
        if sub_col and sub_col in df_dep.columns:
            df_dep[sub_col] = df_dep[sub_col].astype(str).str.strip()
        if nome_col and nome_col in df_dep.columns:
            df_dep[nome_col] = df_dep[nome_col].astype(str).str.strip()


        # --- opcional: se df_ativos tiver historico (coluna de data + varias linhas por Cadastro),
        # infere movimentacoes (De -> Para) dentro/para fora da area analisada.
        dt_hist_col = _first_col(
            df_dep,
            'Data_Movimentacao', 'Data Movimentacao', 'DataMovimentacao',
            'Data_Alteracao', 'Data Alteracao', 'DataAlteracao',
            'Data_Atualizacao', 'Data Atualizacao', 'DataAtualizacao',
            'Data', 'Dt'
        )

        eventos_mov = []
        if dt_hist_col and dt_hist_col in df_dep.columns:
            dfh_cols = [cad_col]
            for c in (nome_col, area_col_dep, sub_col, dt_hist_col):
                if c and c in df_dep.columns and c not in dfh_cols:
                    dfh_cols.append(c)
            dfh = df_dep[dfh_cols].copy()
            dfh[dt_hist_col] = pd.to_datetime(dfh[dt_hist_col], errors='coerce', dayfirst=True)
            dfh = dfh.dropna(subset=[dt_hist_col])

            # so faz sentido se houver mais de um registro por cadastro
            if not dfh.empty and (dfh.duplicated(subset=[cad_col]).any()):
                dfh = dfh.sort_values([cad_col, dt_hist_col])
                prev_area = dfh.groupby(cad_col)[area_col_dep].shift(1)
                prev_sub  = dfh.groupby(cad_col)[sub_col].shift(1) if (sub_col and sub_col in dfh.columns) else None

                def _lbl_hist(a, s):
                    a = '' if a is None or str(a).lower() in ('nan','none') else str(a).strip()
                    if not a:
                        return ''
                    if s is None:
                        return a
                    s = '' if str(s).lower() in ('nan','none') else str(s).strip()
                    return f"{a} / {s}" if s else a

                curr_lbl = [
                    _lbl_hist(r[area_col_dep], (r[sub_col] if (sub_col and sub_col in dfh.columns) else None))
                    for _, r in dfh.iterrows()
                ]
                prev_lbl = [
                    _lbl_hist(a, (prev_sub.iloc[i] if prev_sub is not None else None))
                    for i, a in enumerate(prev_area.tolist())
                ]

                dfh['_PREV_AREA'] = prev_area
                dfh['_PREV_LBL'] = prev_lbl
                dfh['_CURR_LBL'] = curr_lbl

                # mudou de area/subarea
                dfchg = dfh[(dfh['_PREV_LBL'].astype(str).str.len() > 0) & (dfh['_CURR_LBL'] != dfh['_PREV_LBL'])].copy()
                if not dfchg.empty:
                    area_ref2 = (area_ref or '').strip()
                    prev_in = dfchg['_PREV_AREA'].astype(str).str.startswith(area_ref2)
                    curr_in = dfchg[area_col_dep].astype(str).str.startswith(area_ref2)
                    dfchg = dfchg[(prev_in | curr_in) & (dfchg[dt_hist_col].dt.year == int(ano))].copy()

                    if not dfchg.empty:
                        for _, rr in dfchg.iterrows():
                            eventos_mov.append({
                                'Data': rr.get(dt_hist_col),
                                'Cadastro': str(rr.get(cad_col,'')),
                                'Nome': str(rr.get(nome_col,'')) if (nome_col and nome_col in dfh.columns) else '',
                                'De': str(rr.get('_PREV_LBL','')),
                                'Para': str(rr.get('_CURR_LBL','')),
                                'Tipo': 'Movimentacao',
                            })

        area_ref = (area_ref or '').strip()
        df_dep = df_dep[df_dep[area_col_dep].str.startswith(area_ref)].copy()
        if df_dep.empty:
            return pd.DataFrame(columns=['Data','Cadastro','Nome','De','Para','Tipo'])

        # Uma linha por cadastro (último registro)
        df_dep = df_dep.drop_duplicates(subset=[cad_col], keep='last')

        # df_desligados (adm/dem)
        dfa = globals().get('df_desligados', None)
        if isinstance(dfa, pd.DataFrame) and not dfa.empty and 'Cadastro' in dfa.columns:
            dfa = dfa.copy()
            dfa['Cadastro'] = dfa['Cadastro'].astype(str).str.strip()
            adm_col = _first_col(dfa, 'Data_Adm', 'Data Adm', 'DataAdmissao', 'Data_Admissão')
            dem_col = _first_col(dfa, 'Data_Dem', 'Data Dem', 'DataDemissao')
            if adm_col and adm_col in dfa.columns:
                dfa[adm_col] = pd.to_datetime(dfa[adm_col], errors='coerce')
            if dem_col and dem_col in dfa.columns:
                dfa[dem_col] = pd.to_datetime(dfa[dem_col], errors='coerce')
            dfa = dfa[['Cadastro'] + ([adm_col] if adm_col else []) + ([dem_col] if dem_col else [])].drop_duplicates('Cadastro')
        else:
            dfa = None
            adm_col = dem_col = None

        if adm_dep_col and adm_dep_col in df_dep.columns:
            df_dep[adm_dep_col] = pd.to_datetime(df_dep[adm_dep_col], errors='coerce', dayfirst=True)

        # join
        if dfa is not None:
            dfj = df_dep.merge(dfa, how='left', left_on=cad_col, right_on='Cadastro', suffixes=('','_afa'))
        else:
            dfj = df_dep.copy()

        # datas efetivas
        # adm: prioriza df_desligados.Data_Adm, senão df_ativos.Data_Admissão
        if dfa is not None and adm_col:
            dfj['_ADM'] = pd.to_datetime(dfj.get(adm_col), errors='coerce')
        else:
            dfj['_ADM'] = pd.NaT
        if adm_dep_col and adm_dep_col in dfj.columns:
            dfj['_ADM'] = dfj['_ADM'].fillna(pd.to_datetime(dfj[adm_dep_col], errors='coerce', dayfirst=True))

        if dfa is not None and dem_col:
            dfj['_DEM'] = pd.to_datetime(dfj.get(dem_col), errors='coerce')
        else:
            dfj['_DEM'] = pd.NaT

        # label de área (com subarea se tiver)
        def _lbl(a, s):
            a = '' if a is None or str(a).lower() in ('nan','none') else str(a).strip()
            if not a:
                return ''
            if s is None:
                return a
            s = '' if str(s).lower() in ('nan','none') else str(s).strip()
            return f"{a} / {s}" if s else a

        dfj['_AREA_LBL'] = [
            _lbl(r[area_col_dep], (r[sub_col] if (sub_col and sub_col in dfj.columns) else None))
            for _, r in dfj.iterrows()
        ]

        y0 = pd.Timestamp(int(ano), 1, 1)
        y1 = pd.Timestamp(int(ano), 12, 31)

        events = list(eventos_mov)
        for _, r in dfj.iterrows():
            cad = str(r.get(cad_col,''))
            nome = str(r.get(nome_col,'')) if (nome_col and nome_col in dfj.columns) else ''
            area_lbl = str(r.get('_AREA_LBL',''))
            adm = r.get('_ADM', pd.NaT)
            dem = r.get('_DEM', pd.NaT)

            if pd.notna(adm) and (adm >= y0) and (adm <= y1):
                events.append({'Data': adm, 'Cadastro': cad, 'Nome': nome, 'De': 'Fora', 'Para': area_lbl, 'Tipo': 'Entrada'})
            if pd.notna(dem) and (dem >= y0) and (dem <= y1):
                events.append({'Data': dem, 'Cadastro': cad, 'Nome': nome, 'De': area_lbl, 'Para': 'Fora', 'Tipo': 'Saída'})

        out = pd.DataFrame(events)
        if out.empty:
            return pd.DataFrame(columns=['Data','Cadastro','Nome','De','Para','Tipo'])
        out = out.drop_duplicates(subset=['Data','Cadastro','De','Para','Tipo'], keep='last')
        out = out.sort_values(['Data','Cadastro'])
        return out

    def _mov_table_html(df_mov: pd.DataFrame) -> html.Div:
        if df_mov is None or df_mov.empty:
            return html.Div('Sem movimentações detectadas no ano.', style={'fontFamily':'VW Text', 'fontSize':'12px', 'opacity':0.8})

        th_m = {**th, 'fontSize': '12px'}
        td_m = {**td, 'fontSize': '12px', 'textAlign': 'left'}

        head = html.Thead(html.Tr([
            html.Th('Data', style=th_m),
            html.Th('Cadastro', style=th_m),
            html.Th('Nome', style=th_m),
            html.Th('De', style=th_m),
            html.Th('Para', style=th_m),
            html.Th('Tipo', style=th_m),
        ]))

        rows = []
        for _, r in df_mov.iterrows():
            dtv = pd.to_datetime(r.get('Data'), errors='coerce')
            dts = dtv.strftime('%d/%m/%Y') if pd.notna(dtv) else '-'
            rows.append(html.Tr([
                html.Td(dts, style=td_m),
                html.Td(str(r.get('Cadastro','')), style=td_m),
                html.Td(str(r.get('Nome','')), style=td_m),
                html.Td(str(r.get('De','')), style=td_m),
                html.Td(str(r.get('Para','')), style=td_m),
                html.Td(str(r.get('Tipo','')), style=td_m),
            ]))

        tbl = html.Table([head, html.Tbody(rows)], style={'width':'100%', 'borderCollapse':'collapse'})
        return html.Div(tbl, style={'maxHeight':'260px','overflowY':'auto','border':'1px solid #ddd','borderRadius':'6px'})

    # --------- DETALHAMENTO POR ANO (abaixo da tabela, lado a lado) ---------
    # UI: "⬇ 2023" | "⬇ 2024" | "⬇ 2025" (cada um abre a tabela analítica do ano)
    drilldown_footer = None
    try:
        drill_cards = []
        for ano in anos_ord:
            df_mov = _mov_turnover_ano(int(ano), str(area_label))
            details = html.Details(
                [
                    html.Summary(
                        html.Div(
                            [
                                html.Span('⬇', style={'fontSize': '16px'}),
                                html.Span(str(int(ano)), style={'fontSize': '14px', 'fontWeight': 'bold'}),
                            ],
                            style={
                                'display': 'inline-flex',
                                'alignItems': 'center',
                                'gap': '6px',
                                'cursor': 'pointer',
                                'fontFamily': 'VW Text',
                            }
                        ),
                        style={
                            'listStyle': 'none',
                            'textAlign': 'left',
                            'fontFamily': 'VW Text',
                            'padding': '6px 8px',
                            'border': '1px solid #ddd',
                            'borderRadius': '8px',
                            'backgroundColor': '#fafafa',
                        }
                    ),
                    html.Div(
                        [
                            html.Div(
                                f"{int(ano)} — {len(df_mov)} movimentações",
                                style={
                                    'fontFamily': 'VW Text',
                                    'fontSize': '12px',
                                    'fontWeight': 'bold',
                                    'marginBottom': '6px'
                                }
                            ),
                            _mov_table_html(df_mov),
                        ],
                        style={'padding': '8px'}
                    ),
                ],
                open=False,
                style={'width': '100%'}
            )

            drill_cards.append(
                html.Div(
                    details,
                    style={
                        'flex': '1 1 260px',
                        'maxWidth': '360px',
                    }
                )
            )

        drilldown_footer = html.Div(
            drill_cards,
            style={
                'display': 'flex',
                'flexWrap': 'wrap',
                'gap': '10px',
                'justifyContent': 'center',
                'marginTop': '10px',
            }
        )
    except Exception as e:
        drilldown_footer = html.Div(
            f"Detalhamento indisponível: {type(e).__name__}",
            style={
                'fontFamily': 'VW Text',
                'fontSize': '12px',
                'opacity': 0.75,
                'marginTop': '8px',
                'textAlign': 'center'
            }
        )


    tbody = html.Tbody(body_rows)
    table = html.Table([thead, tbody], style=table_style)
    if drilldown_footer is None:
        drilldown_footer = html.Div()
    return html.Div([table, drilldown_footer], style=wrapper_style)


def _piv_gptw(area_col, df, anos_ord):
    # aceita largo (colunas "2023"...) ou longo (Ano + Nota/Score...)
    anos_str = [str(a) for a in anos_ord]
    is_largo = set(anos_str).issubset(set(df.columns))
    if is_largo:
        tmp = df[[area_col]+anos_str].copy()
        for c in anos_str: tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
        piv = tmp.groupby(area_col)[anos_str].mean()
        piv.columns = [int(c) for c in piv.columns]
        return piv.reindex(columns=anos_ord)
    nota_cands = ['Nota','nota','Score','Pontuação','Pontuacao','Índice','Indice','Valor','Value']
    nota_col = next((c for c in nota_cands if c in df.columns), None)
    if 'Ano' not in df.columns or nota_col is None:
        return None
    df = df.copy()
    df['Ano'] = pd.to_numeric(df['Ano'], errors='coerce')
    df[nota_col] = pd.to_numeric(df[nota_col], errors='coerce')
    piv = df.groupby([area_col, 'Ano'])[nota_col].mean().unstack('Ano')
    return piv.reindex(columns=anos_ord)


def tabela_padrao_gptw(area_label, anos=(2023, 2024, 2025), incluir_macro=True):
    df = globals().get('df_gptw', None)
    anos_ord = list(anos)
    anos_str = [str(a) for a in anos_ord]

    if not isinstance(df, pd.DataFrame) or df.empty:
        thead = html.Thead(html.Tr([html.Th('Área', style=PAD_STY['th']), *[html.Th(a, style=PAD_STY['th']) for a in anos_str]]))
        tbody = html.Tbody([html.Tr([html.Td(area_label or '—', style=PAD_STY['td_first']), *[html.Td('—', style=PAD_STY['td']) for _ in anos_ord]])])
        return html.Div(html.Table([thead, tbody], style=PAD_STY['table']), style=PAD_STY['wrapper'])

    area_col = 'Área' if 'Área' in df.columns else ('Área' if 'Área' in df.columns else None)
    if not area_col:
        return html.Div("df_gptw sem coluna 'Área/Área'", style={'opacity':0.7, 'fontSize':'12px'})

    piv = _piv_gptw(area_col, df, anos_ord)
    if piv is None or piv.empty:
        thead = html.Thead(html.Tr([html.Th('Área', style=PAD_STY['th']), *[html.Th(a, style=PAD_STY['th']) for a in anos_str]]))
        tbody = html.Tbody([html.Tr([html.Td(area_label or '—', style=PAD_STY['td_first']), *[html.Td('—', style=PAD_STY['td']) for _ in anos_ord]])])
        return html.Div(html.Table([thead, tbody], style=PAD_STY['table']), style=PAD_STY['wrapper'])

    piv.index = piv.index.astype(str).str.strip()
    her_df = globals().get('her', None)
    linhas_areas = _areas_macro_e_subareas(area_label, piv.index, her_df=her_df)
    if not incluir_macro:
        linhas_areas = [a for a in linhas_areas if a != (area_label or "").strip()] or linhas_areas

    body_rows = []
    for idx, area_nome in enumerate(linhas_areas):
        row = piv.loc[area_nome] if area_nome in piv.index else pd.Series(index=anos_ord, dtype=float)
        c0_style = PAD_STY['td_first'] if idx == 0 else {**PAD_STY['td_first'], 'fontSize':'14px', 'fontWeight':'600'}
        tds = [html.Td(area_nome, style=c0_style)]
        prev_val = None
        for a in anos_ord:
            val = (row.get(a) if isinstance(row, pd.Series) else None)
            txt = _fmt_num(val)
            sym, col = _arrow(val, prev_val)
            arrow_span = html.Span(sym, style={**PAD_STY['arrow'], 'color': col})
            kids = [html.Span(txt)]
            if txt != '—':
                kids.append(arrow_span)   # mesma “assinatura” visual do Turn-Over
            tds.append(html.Td(kids, style=PAD_STY['td']))
            prev_val = val
        body_rows.append(html.Tr(tds))

    thead = html.Thead(html.Tr([html.Th('Área', style=PAD_STY['th']), *[html.Th(a, style=PAD_STY['th']) for a in anos_str]]))
    table = html.Table([thead, html.Tbody(body_rows)], style=PAD_STY['table'])
    return html.Div(table, style=PAD_STY['wrapper'])


def areas_toggle_ui():
    """Botão 'pill' ▦ / 🕸️ (Heatmap ⇄ Diagrama), sem texto e com ícone dentro do knob."""
    return html.Div(
        [
            dcc.Store(id='areas_vis_state', data='heatmap'),
            html.Button(
                id='areas_vis_btn',
                n_clicks=0,
                title="Alternar visualização: Heatmap ⇄ Diagrama",
                style=TOGGLE_STY['btn_base'],
                children=html.Div([
                    html.Span("▦", id='ico_heat',
                              style={**TOGGLE_STY['ico_base'], **TOGGLE_STY['ico_heat_color']}),
                    html.Span("🕸️", id='ico_net',
                              style={**TOGGLE_STY['ico_base'], **TOGGLE_STY['ico_net_color'], **TOGGLE_STY['ico_dim']}),
                    html.Span(id='knob', style=TOGGLE_STY['knob_base']),
                    html.Span("▦", id='knob_ico', style={**TOGGLE_STY['knob_ico_base'], 'color':'#0057b8'}),
                ], style=TOGGLE_STY['inner'])
            ),
        ],
        style=TOGGLE_STY['wrap']
    )


def _abbr_nome_vw(nome: str) -> str:
    """Abreviação simples: primeiro + último token (ex.: 'Ana M. Silva')."""
    try:
        tokens = [t for t in str(nome).split() if t.strip()]
        if not tokens:
            return str(nome)
        if len(tokens) == 1:
            return tokens[0]
        return f"{tokens[0]} {tokens[-1]}"
    except Exception:
        return str(nome)


def grafico_rede_encontros_men(df_full, titulo, validar_mesma_planta: bool = True):
    fig = go.Figure()

    def _final(_fig):
        _fig.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
            height=420,
            font={'family': 'VW Text'}
        )
        return _fig

    if df_full is None or getattr(df_full, "empty", True):
        return _final(fig)

    dfp = df_full.copy()
    dfp["Data"] = pd.to_datetime(dfp["Data"], errors="coerce")
    dfp = dfp.dropna(subset=["Data"])
    if dfp.empty:
        return _final(fig)

    # --- metadados & áreas ---
    area_col = "Área" if "Área" in dfp.columns else ("Área" if "Área" in dfp.columns else None)
    meta_src = dfp.sort_values("Data").drop_duplicates("Cadastro", keep="last")
    cols_meta = ["Cadastro", "Nome", "Cargo"]
    if area_col:
        cols_meta.append(area_col)
    meta = meta_src[cols_meta].set_index("Cadastro")
    if area_col:
        meta = meta.rename(columns={area_col: "ÁreaNorm"})
    else:
        meta["ÁreaNorm"] = ""

    # ordem compatível com o heatmap
    try:
        cad_M, cad_E = calcular_ordem_heatmap(dfp)
    except Exception:
        mask_M = meta["Cargo"].astype(str).eq("M")
        mask_E = meta["Cargo"].astype(str).eq("E")
        cad_M = list(meta.index[mask_M])
        cad_E = list(meta.index[mask_E])

    cad_M = [str(c) for c in cad_M]
    cad_E = [str(c) for c in cad_E]

    if not cad_M or not cad_E:
        return _final(fig)

    # garante presença de todas as colunas
    days = pd.date_range(dfp["Data"].min().normalize(), dfp["Data"].max().normalize(), freq="D")
    pres = (dfp.groupby(["Data", "Cadastro"])["Contagem Presença"]
            .max().unstack("Cadastro").reindex(days).fillna(0.0))
    pres.columns = pres.columns.astype(str)

    A = pres.reindex(columns=cad_M).fillna(0.0).to_numpy(float)
    B = pres.reindex(columns=cad_E).fillna(0.0).to_numpy(float)

    # --- matriz de encontros (% relativo ao mensalista) ---
    if validar_mesma_planta and ("Planta" in dfp.columns):
        def _to_set(x):
            if x is None:
                return frozenset()
            if isinstance(x, float) and pd.isna(x):
                return frozenset()
            if isinstance(x, (set, frozenset)):
                return frozenset(str(t).strip().lower() for t in x)
            if isinstance(x, (list, tuple)):
                return frozenset(str(t).strip().lower() for t in x)
            s = str(x).strip()
            if not s:
                return frozenset()
            for sep in ("|", ";", ",", "/", "\\"):
                if sep in s:
                    return frozenset(t.strip().lower() for t in s.split(sep))
            return frozenset([s.lower()])

        encontros = np.zeros((len(cad_M), len(cad_E)), dtype=float)
        denom = np.zeros((len(cad_M),), dtype=float)

        df_day = dfp[["Data", "Cadastro", "Planta"]].copy()
        df_day["Data"] = pd.to_datetime(df_day["Data"], errors="coerce").dt.normalize()
        df_day = df_day.dropna(subset=["Data", "Cadastro"])

        for _, g in df_day.groupby("Data"):
            g = g.sort_values("Cadastro").drop_duplicates(subset=["Cadastro"], keep="last")
            s = g.set_index("Cadastro")["Planta"]
            pM = [ _to_set(s.get(m, None)) for m in cad_M ]
            pE = [ _to_set(s.get(e, None)) for e in cad_E ]

            presM = np.array([len(ms) > 0 for ms in pM], dtype=bool)
            presE = np.array([len(es) > 0 for es in pE], dtype=bool)

            same = np.array([[ (len(pM[i]) > 0 and len(pE[j]) > 0 and not pM[i].isdisjoint(pE[j]))
                               for j in range(len(cad_E)) ]
                             for i in range(len(cad_M)) ], dtype=bool)

            valid = same & presM[:, None] & presE[None, :]
            encontros += valid.astype(float)
            denom += presM.astype(float)
    else:
        encontros = A.T @ B
        denom = A.sum(axis=0)

    def _rel(am, ae):
        if pd.isna(am) or pd.isna(ae):
            return False
        sm = str(am).replace("/", "-").lower().strip()
        se = str(ae).replace("/", "-").lower().strip()
        return bool(se) and sm.startswith(se)

    # Áreas dos Mensalistas e Executivos (para restringir a comparação à mesma área / herança)
    try:
        # Garante que o índice de meta é string, compatível com cad_M / cad_E
        meta_idx = meta.copy()
        meta_idx.index = meta_idx.index.astype(str)
    except Exception:
        meta_idx = meta

    def _get_area(cad):
        try:
            return str(meta_idx.loc[cad, "ÁreaNorm"])
        except Exception:
            return ""

    areas_M = [_get_area(m) for m in cad_M]
    areas_E = [_get_area(e) for e in cad_E]

    allowed = np.array(
        [[_rel(areas_M[i], areas_E[j]) for j in range(len(cad_E))]
         for i in range(len(cad_M))],
        dtype=bool
    )

    with np.errstate(invalid="ignore", divide="ignore"):
        percent = np.divide(encontros,
                            denom[:, None],
                            out=np.full_like(encontros, np.nan),
                            where=(denom[:, None] > 0) & allowed) * 100.0

    # ---------- construção da rede ----------
    nM, nE = len(cad_M), len(cad_E)
    yM = np.linspace(0, 1, nM) if nM > 1 else np.array([0.5])
    yE = np.linspace(0, 1, nE) if nE > 1 else np.array([0.5])
    pos_M = {cad_M[i]: (0.0, float(yM[i])) for i in range(nM)}
    pos_E = {cad_E[j]: (1.0, float(yE[j])) for j in range(nE)}

    def _width(p):
        if np.isnan(p):
            return 0.0
        if p < 10.0:
            return 0.8
        if p < 25.0:
            return 2.0
        return 3.6

    # coletar todas as arestas para poder destacar TOP 3
    
    # cores das arestas conforme intensidade (vermelho, amarelo, verde)
    def _edge_color(p):
        if np.isnan(p) or p <= 0:
            return "rgba(160,160,160,0.30)"   # cinza, não deve aparecer pois filtramos antes
        if p < 33.0:
            return "rgba(220,53,69,0.75)"     # vermelha (mais fraco)
        if p < 66.0:
            return "rgba(255,193,7,0.75)"     # amarela (médio)
        return "rgba(40,167,69,0.75)"         # verde (mais forte)

    # agrupa as arestas por cor para reduzir quantidade de traces
    edges_by_color = {}

    for i, mid in enumerate(cad_M):
        for j, eid in enumerate(cad_E):
            p = percent[i, j]
            if np.isnan(p) or p <= 0:
                continue
            x0, y0 = pos_M[mid]
            x1, y1 = pos_E[eid]
            nomeM = meta.loc[mid, "Nome"] if mid in meta.index else mid
            nomeE = meta.loc[eid, "Nome"] if eid in meta.index else eid
            txt = f"{_abbr_nome_vw(nomeM)} × {_abbr_nome_vw(nomeE)} — {p:.1f}%"
            w = _width(p)
            if w <= 0:
                continue
            c = _edge_color(p)
            bucket = edges_by_color.setdefault(c, {"x": [], "y": [], "text": [], "widths": []})
            bucket["x"] += [x0, x1, None]
            bucket["y"] += [y0, y1, None]
            bucket["text"].append(txt)
            bucket["widths"].append(w)

    # desenha as arestas por cor
    for color, data in edges_by_color.items():
        if not data["x"]:
            continue
        avg_w = float(np.mean(data["widths"])) if data["widths"] else 1.5
        fig.add_trace(go.Scatter(
            x=data["x"], y=data["y"],
            mode="lines",
            line=dict(width=avg_w, color=color),
            hoverinfo="text",
            hovertext=data["text"],
            showlegend=False
        ))

    nomes_M = [meta.loc[m, "Nome"] if m in meta.index else m for m in cad_M]
    nomes_E = [meta.loc[e, "Nome"] if e in meta.index else e for e in cad_E]
    fig.add_trace(go.Scatter(
        x=[pos_M[c][0] for c in cad_M],
        y=[pos_M[c][1] for c in cad_M],
        mode="markers+text",
        text=[_abbr_nome_vw(n) for n in nomes_M],
        textposition="middle right",
        marker=dict(size=12, line=dict(width=1, color="white")),
        hoverinfo="text",
        hovertext=[f"{n} ({c})" for n, c in zip(nomes_M, cad_M)],
        showlegend=False,
        name="Mensalistas"
    ))
    fig.add_trace(go.Scatter(
        x=[pos_E[c][0] for c in cad_E],
        y=[pos_E[c][1] for c in cad_E],
        mode="markers+text",
        text=[_abbr_nome_vw(n) for n in nomes_E],
        textposition="middle left",
        marker=dict(size=12, line=dict(width=1, color="white")),
        hoverinfo="text",
        hovertext=[f"{n} ({c})" for n, c in zip(nomes_E, cad_E)],
        showlegend=False,
        name="Executivos"
    ))

    linhas = max(len(cad_M), len(cad_E))
    altura = max(460, 40 + 24 * max(linhas, 10))
    fig.update_layout(
        title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
        showlegend=False,
        height=altura,
        xaxis=dict(visible=False, range=[-0.1, 1.1]),
        yaxis=dict(visible=False, range=[-0.1, 1.1]),
        margin=dict(l=120, r=120, t=70, b=60),
        font=dict(family='VW Text')
    )
    return fig


def grafico_rede_encontros_exec(df_full, titulo, validar_mesma_planta: bool = True):
    fig = go.Figure()

    def _final(_fig):
        _fig.update_layout(
            title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
            height=420,
            font={'family': 'VW Text'}
        )
        return _fig

    if df_full is None or getattr(df_full, "empty", True):
        return _final(fig)

    dfp = df_full.copy()

    # Sanitização defensiva de chaves (evita: unhashable type: 'numpy.ndarray')
    for _c in ["Cadastro", "Cargo", "Nome", "Planta", "Área", "Área", "Departamento", "Setor"]:
        if _c in dfp.columns:
            dfp[_c] = dfp[_c].map(_safe_hashable)
    if "Cadastro" in dfp.columns:
        dfp["Cadastro"] = dfp["Cadastro"].astype(str)
    if "Cargo" in dfp.columns:
        dfp["Cargo"] = dfp["Cargo"].astype(str)

    dfp["Data"] = pd.to_datetime(dfp["Data"], errors="coerce")
    dfp = dfp.dropna(subset=["Data"])
    if dfp.empty:
        return _final(fig)

    dfE = dfp[dfp["Cargo"].astype(str).eq("E")].copy()
    if dfE.empty:
        return _final(fig)

    # metadados
    meta_src = dfE.sort_values("Data").drop_duplicates("Cadastro", keep="last")
    meta = meta_src[["Cadastro", "Nome", "Cargo"]].set_index("Cadastro")

    # eixo de dias e matriz de presença
    days = pd.date_range(dfp["Data"].min().normalize(), dfp["Data"].max().normalize(), freq="D")
    pres = (dfE.groupby(["Data", "Cadastro"])["Contagem Presença"]
            .max().unstack("Cadastro").reindex(days).fillna(0.0))
    pres.columns = pres.columns.astype(str)

    cads = list(pres.columns)
    if not cads:
        return _final(fig)

    arr = pres.to_numpy(float)

    if validar_mesma_planta and ("Planta" in dfE.columns):
        def _to_set(x):
            if x is None:
                return frozenset()
            if isinstance(x, float) and pd.isna(x):
                return frozenset()
            if isinstance(x, (set, frozenset)):
                return frozenset(str(t).strip().lower() for t in x)
            if isinstance(x, (list, tuple)):
                return frozenset(str(t).strip().lower() for t in x)
            s = str(x).strip()
            if not s:
                return frozenset()
            for sep in ("|", ";", ",", "/", "\\"):
                if sep in s:
                    return frozenset(t.strip().lower() for t in s.split(sep))
            return frozenset([s.lower()])

        encontros = np.zeros((len(cads), len(cads)), dtype=float)
        dias_pres = np.zeros((len(cads),), dtype=float)

        df_day = dfE[["Data", "Cadastro", "Planta"]].copy()
        df_day["Data"] = pd.to_datetime(df_day["Data"], errors="coerce").dt.normalize()
        df_day = df_day.dropna(subset=["Data", "Cadastro"])

        for _, g in df_day.groupby("Data"):
            g = g.sort_values("Cadastro").drop_duplicates(subset=["Cadastro"], keep="last")
            s = g.set_index("Cadastro")["Planta"]
            p = [ _to_set(s.get(e, None)) for e in cads ]
            presE = np.array([len(es) > 0 for es in p], dtype=bool)

            same = np.array([[ (len(p[i]) > 0 and len(p[j]) > 0 and not p[i].isdisjoint(p[j]))
                               for j in range(len(cads)) ]
                             for i in range(len(cads)) ], dtype=bool)
            valid = same & presE[:, None] & presE[None, :]
            encontros += valid.astype(float)
            dias_pres += presE.astype(float)
    else:
        encontros = arr.T @ arr
        dias_pres = arr.sum(axis=0)

    with np.errstate(invalid="ignore", divide="ignore"):
        perc_dir = np.divide(encontros,
                             dias_pres[:, None],
                             out=np.full_like(encontros, np.nan),
                             where=(dias_pres[:, None] > 0))
    percent = np.nanmean(np.stack([perc_dir, perc_dir.T], axis=0), axis=0) * 100.0

    # layout circular
    n = len(cads)
    if n == 0:
        return _final(fig)

    angs = np.linspace(0, 2*np.pi, n, endpoint=False)
    pos = {cads[i]: (float(np.cos(angs[i])), float(np.sin(angs[i]))) for i in range(n)}

    def _width(p):
        if np.isnan(p):
            return 0.0
        if p < 10.0:
            return 0.8
        if p < 25.0:
            return 2.0
        return 3.6

    # coletar arestas para TOP 3
    all_edges = []
    for i in range(n):
        for j in range(i + 1, n):
            p = percent[i, j]
            if np.isnan(p) or p <= 0:
                continue
            all_edges.append((float(p), i, j))

    all_edges.sort(key=lambda t: t[0], reverse=True)
    top_edges = all_edges[:3]
    highlight_idx = {(i, j) for (_, i, j) in top_edges}

    edges_by_w = {}
    hi_x, hi_y, hi_text = [], [], []

    for i in range(n):
        for j in range(i + 1, n):
            p = percent[i, j]
            if np.isnan(p) or p <= 0:
                continue
            x0, y0 = pos[cads[i]]
            x1, y1 = pos[cads[j]]
            nome_i = meta.loc[cads[i], "Nome"] if cads[i] in meta.index else cads[i]
            nome_j = meta.loc[cads[j], "Nome"] if cads[j] in meta.index else cads[j]
            txt = f"{_abbr_nome_vw(nome_i)} × {_abbr_nome_vw(nome_j)} — {p:.1f}%"
            if (i, j) in highlight_idx:
                hi_x += [x0, x1, None]
                hi_y += [y0, y1, None]
                hi_text.append(txt)
            else:
                w = _width(p)
                if w <= 0:
                    continue
                bucket = edges_by_w.setdefault(w, {"x": [], "y": [], "text": []})
                bucket["x"] += [x0, x1, None]
                bucket["y"] += [y0, y1, None]
                bucket["text"].append(txt)

    for w, data in edges_by_w.items():
        fig.add_trace(go.Scatter(
            x=data["x"], y=data["y"],
            mode="lines",
            line=dict(width=w, color="rgba(80,80,80,0.45)"),
            hoverinfo="text",
            hovertext=data["text"],
            showlegend=False
        ))

    if hi_x:
        max_w = max(_width(p) for (p, _, _) in top_edges) if top_edges else 3.6
        fig.add_trace(go.Scatter(
            x=hi_x, y=hi_y,
            mode="lines",
            line=dict(width=max_w + 0.8, color="rgba(10,163,122,0.95)"),
            hoverinfo="text",
            hovertext=hi_text,
            showlegend=False
        ))

    nomes = [meta.loc[c, "Nome"] if c in meta.index else c for c in cads]
    fig.add_trace(go.Scatter(
        x=[pos[c][0] for c in cads],
        y=[pos[c][1] for c in cads],
        mode="markers+text",
        text=[_abbr_nome_vw(n) for n in nomes],
        textposition="top center",
        marker=dict(size=12, line=dict(width=1, color="white")),
        hoverinfo="text",
        hovertext=[f"{n} ({c})" for n, c in zip(nomes, cads)],
        showlegend=False
    ))

    fig.update_layout(
        title={'text': titulo, 'font': {'family': 'VW Text', 'size': 22}},
        showlegend=False,
        height=max(460, 420),
        xaxis=dict(visible=False, range=[-1.2, 1.2]),
        yaxis=dict(visible=False, range=[-1.2, 1.2]),
        margin=dict(l=80, r=80, t=70, b=60),
        font=dict(family='VW Text')
    )
    return fig


'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Relacionamento ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
def _hex_to_rgb(h):
    h = h.strip()
    if not h.startswith("#"): h = f"#{h}"
    h = h.lstrip("#")
    if len(h)==3:
        h = "".join([c*2 for c in h])
    return tuple(int(h[i:i+2], 16) for i in (0,2,4))


def _luminancia(hex_color) -> float:
    r, g, b = _hex_to_rgb(hex_color)
    def _lin(c):
        c = c / 255.0
        return c/12.92 if c <= 0.03928 else ((c+0.055)/1.055)**2.4
    R, G, B = _lin(r), _lin(g), _lin(b)
    return 0.2126*R + 0.7152*G + 0.0722*B


def tag_chip(texto: str, cor_hex: str):
    cor_hex = cor_hex.strip()
    if not cor_hex.startswith("#"): cor_hex = f"#{cor_hex}"
    txt_color = "#000000" if _luminancia(cor_hex) > 0.5 else "#ffffff"
    return html.Span(
        texto,
        className="tag-chip",  # classe neutra (sem cor base!)
        style={
            "backgroundColor": cor_hex,
            "color": txt_color,
            "border": f"1px solid {cor_hex}",
            "borderRadius": "12px",
            "padding": "3px 8px",
            "display": "inline-block",
            "marginRight": "6px",
            "marginBottom": "6px",
            "fontWeight": "600",
            "fontSize": "12px",
            "lineHeight": "1",
            "whiteSpace": "nowrap"
        }
    )


def add_calendar_stack(
    fig, semanas,
    week_font_size=11,
    month_font_size=12,
    year_font_size=12,
    month_as_number=True,
    month_y=-0.18,
    year_y=-0.30,
    bottom_margin=70,
    yshift_mes=-20,
    yshift_ano=-28
):
    if not semanas:
        return fig

    idx = pd.to_datetime(pd.Series(semanas)).sort_values().reset_index(drop=True)
    df = pd.DataFrame({"x": idx})
    iso = df['x'].dt.isocalendar()
    df['ano'] = iso.year.astype(int)
    df['sem_iso'] = iso.week.astype(int)
    df['mes'] = df['x'].dt.month

    # Ticks semanais (1..N reiniciando por ano)
    week_labels = []
    for _, g in df.groupby('ano', sort=True):
        g = g.sort_values('x').copy()
        seq_map = {w: i+1 for i, w in enumerate(g['sem_iso'].tolist())}
        week_labels.extend([str(seq_map[w]) for w in g['sem_iso']])

    fig.update_xaxes(
        type='date',
        tickmode='array',
        tickvals=df['x'],
        ticktext=week_labels,
        tickangle=0,
        tickfont=dict(size=week_font_size),
        ticks='outside',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.06)'
    )

    x_first = df['x'].iloc[0]

    # MÊS
    for (_, m), g in df.groupby(['ano', 'mes'], sort=True):
        g = g.sort_values('x')
        x0, x1 = g['x'].iloc[0], g['x'].iloc[-1]
        xc = x0 + (x1 - x0) / 2
        mes_label = (str(m) if month_as_number else g['x'].dt.strftime('%b').iloc[0])

        fig.add_annotation(
            x=xc, y=month_y, yref='paper', yshift=yshift_mes,
            text=mes_label,
            showarrow=False,
            font=dict(size=month_font_size),
            xanchor='center', align='center'
        )
        if x0 != x_first:
            fig.add_vline(x=x0, line_color='rgba(0,0,0,0.20)', line_width=1.5)

    # ANO — rótulo central + linha pontilhada NA FRONTEIRA entre anos
    grupos = [(a, g.sort_values('x')) for a, g in df.groupby('ano', sort=True)]

    # 1) Rótulos centrados em cada ano
    for a, g in grupos:
        x0, x1 = g['x'].iloc[0], g['x'].iloc[-1]
        xc = x0 + (x1 - x0) / 2
        fig.add_annotation(
            x=xc, y=year_y, yref='paper', yshift=yshift_ano,
            text=str(a),
            showarrow=False,
            font=dict(size=year_font_size),
            xanchor='center', align='center'
        )

    # margem inferior para caber as anotações
    fig.update_layout(
        margin=dict(
            t=(fig.layout.margin.t if fig.layout.margin and fig.layout.margin.t else 110),
            r=20, b=bottom_margin, l=60
        )
    )
    return fig


def _collapse_areas_selection(sel_list: list[str], her_df: pd.DataFrame) -> list[str]:
    """
    Remove subáreas quando a área macro (mãe) já estiver na seleção.
    Considera colunas 'Área' e 'Heranca' se disponíveis, caso contrário, usa prefixo.
    """
    sel_list = [str(a).strip() for a in (sel_list or [])]
    if not sel_list:
        return []

    sel_set = set(sel_list)
    to_remove = set()

    # Caso 1: não há DataFrame de herança válido ⭢ só compara prefixos
    if not isinstance(her_df, pd.DataFrame) or her_df.empty or not {'Área','Heranca'}.issubset(her_df.columns):
        for a in sel_set:
            for b in sel_set:
                if a != b and str(b).startswith(str(a)):
                    to_remove.add(b)
        return sorted(sel_set - to_remove)

    # Caso 2: há herança — respeita cadeia mãe-filha
    tmp = her_df[['Área','Heranca']].dropna().drop_duplicates()
    H = dict(zip(tmp['Área'].astype(str), tmp['Heranca'].astype(str)))

    for a in sel_set:
        for b in sel_set:
            if a == b:
                continue

            # É filha direta ou descendente da mesma herança
            is_prefix = str(b).startswith(str(a))
            mesma_linhagem = (H.get(b) == H.get(a)) or (H.get(b) == a)
            if is_prefix and (mesma_linhagem or a in H.get(b, '')):
                to_remove.add(b)

    # Se mãe foi selecionada, remove filhas
    # (ex: 'Comercial' selecionada ⭢ remove 'Comercial - SP', 'Comercial - RJ')
    return sorted(sel_set - to_remove)


def _gptw_rows_apenas(area_label: str | None, anos=(2023, 2024, 2025), incluir_macro=True):
    td = {'padding':'4px 8px','border':'1px solid #bbb','textAlign':'center','fontFamily':'VW Text'}
    td_first = {**td, 'fontWeight':'bold', 'fontSize':'20px'}
    ARROW_UP, ARROW_DOWN, ARROW_NONE = '▲', '▼', '—'
    color_up, color_down, color_none = '#0a7f3f', '#b00020', '#999'
    arrow_style_base = {'marginLeft':'6px', 'fontSize':'12px', 'opacity':0.9}

    anos_ord = list(anos)
    anos_str_ord = [str(a) for a in anos_ord]

    def _fmt(v):
        if pd.isna(v): return '—'
        try:
            v = float(v); return str(int(v)) if v.is_integer() else f'{v:.1f}'
        except Exception:
            return str(v)

    def _arrow(curr, prox):
        if curr is None or prox is None or pd.isna(curr) or pd.isna(prox):
            return (ARROW_NONE, color_none)
        try:
            c = float(curr); p = float(prox)
        except Exception:
            return (ARROW_NONE, color_none)
        if c > p:   return (ARROW_UP, color_up)
        if c < p:   return (ARROW_DOWN, color_down)
        return (ARROW_NONE, color_none)

    if not area_label:
        return []

    df = globals().get('df_gptw', None)
    if df is None or getattr(df, 'empty', True):
        return [html.Tr([html.Td(area_label, style={**td_first, 'textAlign':'left'})] +
                        [html.Td('—', style=td_first) for _ in anos_ord])]

    df = df.copy()
    area_col = 'Área' if 'Área' in df.columns else ('Área' if 'Área' in df.columns else None)
    if area_col is None:
        return [html.Tr([html.Td("df_gptw sem coluna Área/Área", style=td_first)])]

    is_largo = set(anos_str_ord).issubset(set(df.columns))
    if is_largo:
        cols = [area_col] + anos_str_ord
        tmp = df[cols].copy()
        for c in anos_str_ord: tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
        piv = (tmp.groupby(area_col)[anos_str_ord].mean())
        piv.columns = [int(c) for c in piv.columns]
        piv = piv.reindex(columns=anos_ord)
    else:
        cand_nota = ['Nota','nota','Score','Pontuação','Pontuacao','Índice','Indice','Valor','Value']
        nota_col = next((c for c in cand_nota if c in df.columns), None)
        if 'Ano' not in df.columns or nota_col is None:
            return [html.Tr([html.Td("df_gptw precisa de 'Ano' e 'Nota/Score/Pontuação'", style=td_first)])]
        df['Ano'] = pd.to_numeric(df['Ano'], errors='coerce')
        df[nota_col] = pd.to_numeric(df[nota_col], errors='coerce')
        piv = (df.groupby([area_col, 'Ano'])[nota_col].mean().unstack('Ano'))
        piv = piv.reindex(columns=anos_ord)

    piv.index = piv.index.astype(str).str.strip()
    alvo = str(area_label).strip()

    # subáreas por herança (se houver) ou por prefixo
    subareas = []
    her_df = globals().get('her', None)
    if isinstance(her_df, pd.DataFrame) and {'Área','Heranca'}.issubset(her_df.columns):
        her_df = her_df.copy()
        her_df['Área'] = her_df['Área'].astype(str).str.strip()
        herancas = her_df.loc[her_df['Área'].eq(alvo), 'Heranca'].dropna().unique()
        cand = her_df[her_df['Área'].str.startswith(alvo) & her_df['Área'].ne(alvo)]
        if len(herancas) > 0:
            cand = cand[cand['Heranca'].isin(herancas)]
        subareas = cand['Área'].dropna().unique().tolist()
    if not subareas:
        all_areas = piv.index.tolist()
        subareas = [a for a in all_areas if a != alvo and a.startswith(alvo)]

    def _akey(a):
        toks = [t.strip() for t in str(a).replace('/', '-').split('-') if t.strip()]
        return (len(toks), tuple(toks))

    linhas = []
    if incluir_macro: linhas.append(alvo)
    linhas.extend(sorted(subareas, key=_akey))
    if not linhas: linhas = [alvo]
    
    body_rows = []
    for idx, area_nome in enumerate(linhas):
        row = piv.loc[area_nome] if area_nome in piv.index else pd.Series(index=anos_ord, dtype=float)
        td_this = td_first if idx == 0 else td
        tds = [html.Td(area_nome, style={**td_this, 'textAlign':'left'})]
        for j, a in enumerate(anos_ord):
            val = row.get(a) if isinstance(row, pd.Series) else None
            prox_val = row.get(anos_ord[j-1]) if j > 0 else None
            val_txt = _fmt(val)
            sym, color = _arrow(val, prox_val)
            kids = [html.Span(val_txt)]
            if val_txt != '—':
                kids.append(html.Span(sym, style={**arrow_style_base, 'color': color}))
            tds.append(html.Td(kids, style=td_this))
        body_rows.append(html.Tr(tds))
    return body_rows


def gptw_tabela_empilhada(areas: list[str] | None, anos=(2023, 2024, 2025)):
    if not areas:
        return html.Div()

    her_df = globals().get('her', pd.DataFrame())
    areas_final = _collapse_areas_selection(areas, her_df)

    # agrega linhas de todas as áreas (macro + subáreas)
    all_rows = []
    for a in areas_final:
        all_rows.extend(_gptw_rows_apenas(a, anos=anos, incluir_macro=True))

    # cabeçalho (título) uma única vez
    anos_ord = list(anos)
    th = {'padding':'4px 8px','border':'1px solid #bbb','backgroundColor':'#f5f5f5',
          'fontWeight':'bold','textAlign':'center','fontFamily':'VW Text', 'fontSize':'30px'}
    thead = html.Thead(html.Tr([html.Th('Área', style=th)] + [html.Th(str(a), style=th) for a in anos_ord]))

    tabela = html.Table([thead, html.Tbody(all_rows)], style={
        'borderCollapse':'collapse','fontSize':'13px',
        'boxShadow':'0px 2px 6px rgba(0,0,0,0.10)',
        'maxWidth':'720px','width':'100%'
    })
    return html.Div(tabela, style={'display':'flex','justifyContent':'center'})


def aplicar_tags(fig, semanas, tags, dash='dot', width=4):
    """
        Desenha vlines pontilhadas e títulos no topo (paper y=1) para cada tag.
        - Encaixa a data da tag na semana mais próxima (SemIni), pra manter alinhamento.
    """
    if not semanas or not tags:
        return fig

    xs = pd.to_datetime(pd.Series(semanas)).sort_values().reset_index(drop=True)

    for t in (tags or []):
        data_str = (t or {}).get('data')
        nome     = (t or {}).get('nome') or ''
        cor_raw  = (t or {}).get('cor')  or '#3221cc'
        cor      = _fmt_hex(cor_raw) or '#3221cc'  # <— garante #RRGGBB

        try:
            dt = pd.to_datetime(data_str)
        except Exception:
            continue

        # "encaixa" na semana mais próxima
        pos = (xs - dt).abs().argmin()
        xline = xs.iloc[int(pos)]

        fig.add_vline(x=xline, line_color=cor, line_dash=dash, line_width=width)
        if nome:
            fig.add_annotation(
                x=xline, y=1.04,   # <— sobe um pouco acima do eixo/paper
                yref='paper', xref='x',
                text=nome, showarrow=False,
                xanchor='left', yanchor='bottom',
                font=dict(color=cor, size=12),
                bgcolor='rgba(255,255,255,0.7)'
            )
    return fig


def mini_tabelas_gptw_multi(areas: list[str] | None):
    if not areas:
        return html.Div()  # nada selecionado => não mostra

    # uma tabela por área, em coluna única e centralizada
    blocos = [html.Div(mini_tabela_gptw(a), style={'margin':'8px 0'}) for a in areas]
    return html.Div(
        blocos,
        style={
            'display':'flex',
            'flexDirection':'column',
            'alignItems':'center',     # CENTRALIZA
            'justifyContent':'center',
            'gap':'8px'
        }
    )


def _fmt_hex(c):
    """Normaliza para #RRGGBB; retorna None se inválido."""
    if c is None or str(c).strip() == "":
        return None
    s = str(c).strip().lower()
    if s.startswith("#"):
        s = s[1:]
    if len(s) == 3 and all(ch in "0123456789abcdef" for ch in s):
        s = "".join(ch*2 for ch in s)  # ex: #abc -> #aabbcc
    if len(s) == 6 and all(ch in "0123456789abcdef" for ch in s):
        return "#" + s
    # tenta rgb(a)
    if s.startswith("rgb"):
        try:
            nums = "".join(ch if ch.isdigit() or ch in ",." else " " for ch in s)
            r,g,b = [int(float(x)) for x in nums.split(",")[:3]]
            r,g,b = max(0,min(255,r)), max(0,min(255,g)), max(0,min(255,b))
            return f"#{r:02x}{g:02x}{b:02x}"
        except Exception:
            return None
    return None


def render_tags(df_tags):
    if df_tags is None or df_tags.empty:
        return html.Span("Sem tags")
    items = []
    for _, row in df_tags.iterrows():
        items.append(tag_chip(str(row.get("tag", "")), row.get("cor") or "#3221cc"))
    return html.Div(items, style={"display": "flex", "flexWrap": "wrap", "gap": "6px"})


def build_dataframe_almoco_pessoas(cad_list, df_almoco_local, dt_ini=None, dt_fim=None):
    """
    Retorna um DF com uma linha por (Data, Cadastro) contendo o conjunto de clusters de almoço do dia.
    Usado para calcular encontros de almoço entre pessoas selecionadas.
    """
    def empty():
        return pd.DataFrame(columns=['Data','Cadastro','Clusters'])

    if not cad_list or df_almoco_local is None or getattr(df_almoco_local, 'empty', True):
        return empty()

    cad_set = {str(c) for c in cad_list}
    
    df = df_almoco_local.copy()
    req_cols = {'Cadastro','Data','Hora','Dir'}
    if not req_cols.issubset(df.columns):
        return empty()

    df['Cadastro'] = df['Cadastro'].astype(str)
    df = df[df['Cadastro'].isin(cad_set)]
    if df.empty:
        return empty()

    # datas/horas
    df['Data'] = pd.to_datetime(df['Data'], errors='coerce')
    horas_raw = df['Hora'].astype(str).str.strip()
    h1 = pd.to_datetime(horas_raw, format='%H:%M:%S', errors='coerce')
    h2 = pd.to_datetime(horas_raw, format='%H:%M', errors='coerce')
    df['Hora'] = h1.fillna(h2)
    df = df.dropna(subset=['Data','Hora','Dir'])
    df['Data_norm'] = df['Data'].dt.normalize()

    # janela de almoço
    h_ini = pd.to_datetime('11:00:00', format='%H:%M:%S').time()
    h_fim = pd.to_datetime('14:00:00', format='%H:%M:%S').time()
    df = df[(df['Hora'].dt.time >= h_ini) & (df['Hora'].dt.time <= h_fim)]
    df = df[df['Dir'].isin(['E','S'])]

    if dt_ini is not None and dt_fim is not None:
        df = df[(df['Data'] >= pd.Timestamp(dt_ini).normalize()) &
                (df['Data'] <= pd.Timestamp(dt_fim).normalize())]
    if df.empty:
        return empty()

    def normalizar_coletor(desc: str):
        if desc is None: return 'na'
        d = str(desc).lower().strip()
        if d in ('', 'nan'): return 'na'
        if 'refe' in d: return 'refeitorio'   # refeitório
        if 'rua'  in d: return 'rua'
        if 'port' in d: return 'portaria'
        return d

    #  cluster sempre alinhado por índice (evita length mismatch)
    def cluster_por_janela(g, chave_prefix, janela_min=3):
        if g.empty:
            return g.assign(_cluster=pd.Series([], dtype=object))
        g = g.sort_values('Hora').copy()
        tmins = (g['Hora'].dt.hour * 60 + g['Hora'].dt.minute + g['Hora'].dt.second/60.0).astype(float)
        delta = tmins.diff()
        # garante que a primeira linha sempre inicia um cluster
        delta.iloc[0] = janela_min + 1
        novo = (delta >= janela_min).astype(int).cumsum()
        # alinhar pelo índice:
        novo = pd.Series(novo.values, index=g.index)
        g['_cluster'] = (
            g['Data_norm'].dt.strftime('%Y-%m-%d') + '|' + chave_prefix + '|' + novo.astype(str)
        )
        return g

    parts = []
    if 'Descricao Coletor' in df.columns:
        df['Descricao Agrupada'] = df['Descricao Coletor'].apply(normalizar_coletor)
        # ENTRADAS – cluster 1min para refeitório, 3min outros
        for (_, _, col_norm), g in df[df['Dir']=='E'].groupby(['Data_norm','Dir','Descricao Agrupada'], dropna=False):
            chave = f"E|{str(col_norm) if pd.notna(col_norm) else 'NA'}"
            janela_min = 1 if str(col_norm).startswith('ref') else 3
            parts.append(cluster_por_janela(g, chave, janela_min))
    else:
        # sem coletor, usa 1min para entradas
        for (_, _), g in df[df['Dir']=='E'].groupby(['Data_norm','Dir']):
            parts.append(cluster_por_janela(g, "E", 1))

    # SAÍDAS – 3min
    for (_, _), g in df[df['Dir']=='S'].groupby(['Data_norm','Dir']):
        parts.append(cluster_por_janela(g, "S", 3))

    if not parts:
        return empty()

    dfc = pd.concat(parts, ignore_index=True)

    # um conjunto de clusters por dia/pessoa
    clust = (
        dfc.groupby(['Data_norm','Cadastro'])['_cluster']
           .apply(lambda s: frozenset(s.dropna().unique().tolist()))
           .reset_index()
    )
    clust.rename(columns={'Data_norm':'Data', '_cluster':'Clusters'}, inplace=True)

    # mantém apenas dias com 2+ pessoas (onde encontro é possível)
    pres_por_dia = clust.groupby('Data')['Cadastro'].nunique().reset_index(name='qtde_pessoas')
    dias_ok = set(pres_por_dia.loc[pres_por_dia['qtde_pessoas'] > 1, 'Data'])
    clust = clust[clust['Data'].isin(dias_ok)]

    return clust[['Data','Cadastro','Clusters']]


def _semana_inicio(s):
    """
    Recebe uma Series de datas e retorna uma Series com o 1º dia (segunda) da semana ISO de cada data.
    Mantém o mesmo índice e mesmo comprimento da entrada.
    """
    s = pd.to_datetime(s, errors='coerce')
    iso = s.dt.isocalendar()  # tem colunas 'year', 'week', 'day'
    y = iso['year'].astype(int).astype(str)
    w = iso['week'].astype(int).astype(str).str.zfill(2)
    out = pd.to_datetime(y + w + '1', format='%G%V%u', errors='coerce')
    # preserva o índice 1:1 para evitar length mismatch em atribuições
    out.index = s.index
    return out


# '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ App e Layout ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
her = (
    df_semana_merged[['Área', 'Departamento', 'Heranca']]
    .drop_duplicates()
    .sort_values('Área')
)


dropdown_options = [{'label': 'Geral (todas as áreas)', 'value': 'GERAL'}] + [
    {'label': f"{row['Área']} {row['Departamento']}", 'value': row['Área']}
    for _, row in her.iterrows()
]


app = Dash(
    __name__,
    suppress_callback_exceptions=True,
    prevent_initial_callbacks='initial_duplicate'
)


# Injeta CSS global no <head> sem depender de html.Style (compatível com versões antigas do Dash)
try:
    app.index_string = f'''<!DOCTYPE html>
<html>
    <head>
        {{%metas%}}
        <title>{{%title%}}</title>
        {{%favicon%}}
        {{%css%}}
        <style>{GLOBAL_CSS}</style>
    </head>
    <body>
        {{%app_entry%}}
        <footer>
            {{%config%}}
            {{%scripts%}}
            {{%renderer%}}
        </footer>
    </body>
</html>'''
except Exception:
    pass


cache = Cache(app.server, config={
    "CACHE_TYPE": "SimpleCache",
    "CACHE_DEFAULT_TIMEOUT": 900
})


cache.init_app(app.server)


app.layout = html.Div([
    dcc.Location(id='url', refresh=False),


    # '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Geral ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
    html.Div([
        html.Div(id='pagina-geral', children=[

            # ====== Título ======
            html.Div([
                html.H1([
                    "Relatório Presencial Geral de toda a Volkswagen (Mensalistas/Executivos)",
                    html.Br(),
                    html.Span(id='periodo-texto')
                ], style={'textAlign': 'center', 'flex': '1', 'fontFamily': 'VW Text'})
            ], style={
                'display': 'flex',
                'justifyContent': 'flex-end',
                'alignItems': 'center',
                'marginBottom': '20px',
                'fontFamily': 'VW Text'
            }),

            # ================== BLOCO DE FILTROS ==================
            html.Div([
                html.Div([
                    # --- Bloco A: 2 RadioItems ---
                    html.Div([
                        html.Label("Mostrar as pessoas desligadas?"),
                        dcc.RadioItems(
                            id='desligados-toggle',
                            options=[
                                {'label': 'Sim', 'value': 'left'},
                                {'label': 'Não', 'value': 'right'}
                            ],
                            value='left',
                            labelStyle={'display': 'inline-block', 'marginRight': '10px', 'fontFamily': 'VW Text'}
                        ),
                    ], style={'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center', 'fontFamily': 'VW Text'}),

                    html.Div([
                        html.Label("Mostrar as áreas que viajam?"),
                        dcc.RadioItems(
                            id='viajam-toggle',
                            options=[
                                {'label': 'Sim', 'value': 'left'},
                                {'label': 'Não', 'value': 'right'}
                            ],
                            value='left',
                            labelStyle={'display': 'inline-block', 'marginRight': '10px', 'fontFamily': 'VW Text'}
                        ),
                    ], style={'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center', 'fontFamily': 'VW Text'}),
                ], style={'display': 'flex', 'gap': '30px', 'alignItems': 'center', 'fontFamily': 'VW Text'}),

                # --- Bloco B: Linha do tempo ---
                html.Div([
                    html.Div([
                        dcc.RadioItems(
                            id='cenario-toggle',
                            options=[
                                {'label': 'Jan/2024 a Abr/2025', 'value': 'c1'},
                                {'label': 'Mai/2025 em diante', 'value': 'c2'}
                            ],
                            value='c1',
                            labelStyle={'display': 'inline-block', 'fontFamily': 'VW Text'}
                        )
                    ], style={'textAlign': 'center', 'marginBottom': '6px', 'fontFamily': 'VW Text'}),

                    dcc.RangeSlider(
                        id='periodo-slider',
                        min=0,
                        max=len(LT_MONTHS)-1,
                        value=[LT_C1_MIN, LT_C1_MAX],
                        marks=LT_MARKS,
                        allowCross=False,
                        step=1,
                        pushable=1
                    )
                ], style={'flex': '0 0 560px', 'fontFamily': 'VW Text'}),

                # --- Bloco C: Dropdown ---
                html.Div([
                    dcc.Dropdown(
                        id='dropdown-area',
                        options=dropdown_options,
                        value='GERAL',
                        clearable=False,
                        style={'width': '270px', 'fontFamily': 'VW Text'}
                    ),
                    html.Div([
                        dcc.Dropdown(
                            id='dropdown-subarea',
                            options=[],
                            value=None,
                            placeholder='Selecione uma subárea',
                            clearable=True,
                            style={'width': '270px', 'marginTop': '10px', 'display': 'none', 'fontFamily': 'VW Text'}
                        )
                    ], id='subarea-container')
                ], style={'marginLeft': 'auto', 'fontFamily': 'VW Text'})
            ], style={
                'flex': '1',
                'padding': '20px',
                'boxSizing': 'border-box',
                'backgroundColor': "#e0e0e0",
                'borderRadius': '20px',
                'height': 'fit-content',
                'marginTop': '30px',
                                'marginRight': '10px',
                'boxShadow': '0px 3px 6px rgba(0,0,0,0.1)',
                'display': 'flex',
                'flexWrap': 'wrap',
                'alignItems': 'flex-start',
                'marginBottom': '30px',
                'fontFamily': 'VW Text'
            }),
            # ======================================================

            # ====== Heatmap + Info ======
            html.Div([
                html.Div([
                    dcc.Loading(
                        dcc.Graph(id='grafico-heatmap', config={'displayModeBar': False},
                                    style={'height': '250px', 'width': '100%', 'fontFamily': 'VW Text'})
                    ),
                    html.Button(
                        "↻",
                        id="botao-resetar",
                        style={
                            'fontFamily': 'VW Text',
                            'position': 'absolute',
                            'top': '30px',
                            'right': '20px',
                            'zIndex': '1',
                            'backgroundColor': '#e0e0e0',
                            'border': '1px solid #ccc',
                            'padding': '5px 10px',
                            'borderRadius': '4px',
                            'cursor': 'pointer',
                            'fontWeight': 'bold',
                            'fontSize': '18px'
                        }
                    )
                ], style={
                    'position': 'relative',
                    'flex': '1',
                    'minWidth': '400px',
                    'maxWidth': '100%',
                    'fontFamily': 'VW Text',
                }),

                html.Div(id='tabela-informacoes-colaboradores', style={
                    'flex': '1',
                    'minWidth': '280px',
                    'maxWidth': '400px',
                    'padding': '20px',
                    'boxSizing': 'border-box',
                    'backgroundColor': "#e0e0e0",
                    'borderRadius': '20px',
                    'height': 'fit-content',
                    'marginTop': '30px',
                    'marginLeft': '240px',
                    'marginRight': '130px',
                    'boxShadow': '0px 3px 6px rgba(0,0,0,0.1)',
                    'fontFamily': 'VW Text'
                })
            ], style={
                'display': 'flex',
                'flexWrap': 'wrap',
                'alignItems': 'flex-start',
                'marginBottom': '30px',
                'fontFamily': 'VW Text'
            }),

            # ====== Dispersão + Cargo ======
            html.Div([
                html.Div([
                    dcc.Loading(dcc.Graph(id='grafico-dispersao',
                                            style={'height': '550px', 'width': '100%', 'fontFamily': 'VW Text'}))
                ], style={'flex': '1', 'marginRight': '20px'}),
                html.Div([
                    dcc.Loading(dcc.Graph(id='grafico-presenca-cargo',
                                            style={'height': '550px', 'width': '100%', 'fontFamily': 'VW Text'}))
                ], style={'flex': '1'})
            ], style={'display': 'flex', 'flexWrap': 'wrap'}),

            # ====== Boxplot ======
            html.Div([
                html.Div([
                    dcc.Loading(dcc.Graph(id='grafico-boxplot',
                                            style={'height': '600px', 'width': '100%', 'fontFamily': 'VW Text'}))
                ], style={'flex': '2'})
            ], style={'display': 'flex', 'flexWrap': 'wrap'}),

            # ====== Resumo ======
            html.Div([
                # Botão de toggle ►
                html.Button("►", id="toggle-resumo-btn", n_clicks=0, style={
                    'fontSize': '18px',
                                        'background': 'none',
                    'border': 'none',
                    'cursor': 'pointer',
                    'verticalAlign': 'middle',
                    'fontFamily': 'VW Text'
                }),

                # Título principal
                html.Span("Resumo Específico por Área (Mensalistas/Executivos)", style={
                    'fontWeight': 'bold',
                    'fontSize': '18px',
                    'color': '#000000',
                    'flex': '1',
                    'textAlign': 'left',
                    'fontFamily': 'VW Text'
                }),

                # Novo botão/link para a página de relacionamento
                dcc.Link(
                    "Relacionamento entre Colaboradores ⭢",
                    href="/relacionamento",
                    style={
                        'fontFamily': 'VW Text',
                        'fontSize': '15px',
                        'fontWeight': 'bold',
                        'color': '#000000',
                        'textDecoration': 'none',
                        'marginLeft': 'auto',
                        'marginRight': '20px',
                        'padding': '6px 12px',
                        'border': '1px solid #000000',
                        'borderRadius': '6px',
                        'backgroundColor': "#e0e0e0",
                        'boxSizing': 'border-box',
                        'cursor': 'pointer'
                    }
                )
            ], style={
                'display': 'flex',
                'alignItems': 'center',
                'justifyContent': 'space-between',
                'marginBottom': '8px',
                'fontFamily': 'VW Text'
            }),

            html.Div(id='resumo-container', style={'marginBottom': '20px', 'display': 'none', 'fontFamily': 'VW Text'}),

            # ====== Lista Pessoas ======
            html.Div([
                html.Div([
                    html.Button(
                        "►", id="toggle-lista-btn", n_clicks=0,
                        style={
                            'fontSize': '18px',
                                                        'background': 'none',
                            'border': 'none',
                            'cursor': 'pointer',
                            'verticalAlign': 'middle',
                            'fontFamily': 'VW Text'
                        }
                    ),
                    html.Span("Presença dos Colaboradores", style={
                        'fontWeight': 'bold',
                        'fontSize': '18px',
                        'color': '#000000',
                        'flex': '1',
                        'textAlign': 'left',
                        'fontFamily': 'VW Text'
                    })
                ], style={'marginBottom': '8px'}),

                html.Div(
                    id='lista-pessoas-container',
                    children=[
                        dcc.Dropdown(
                            id='dropdown-chapa',
                            options=[],
                            value=None,
                            placeholder='Buscar colaborador pela chapa...',
                            clearable=True,
                            searchable=True,
                            style={'width': '300px', 'marginTop': '10px', 'marginBottom': '10px', 'fontFamily': 'VW Text'}
                        ),
                        html.Div(id='tabela-detalhada-colaboradores')
                    ],
                    style={'display': 'none'}
                )
            ],
            style={'marginTop': '15px', 'fontFamily': 'VW Text'}),

            # ====== Stores ======
            dcc.Store(id='faixa-selecionada', data=None),
            dcc.Store(id='area-selecionada', data='GERAL'),
            dcc.Store(id='subarea-selecionada', data=None),
            dcc.Store(id='ultimo-clique', data={'faixa': None, 'timestamp': 0}),
            dcc.Store(id='periodo-indices', data=[LT_C1_MIN, LT_C1_MAX])
        ], style={'padding': '20px', 'backgroundColor': 'white', 'fontFamily': 'VW Text'})
    ]),

    # '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Individual ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
    dcc.Loading(
        id="loading-individual",
        type="circle",
        fullscreen=True,
        children=html.Div(
            id='conteudo-individual',
            children=[
                html.Button(
                    " ↩ ",
                    id='botao-voltar',
                    n_clicks=0,
                    style={
                        'color': 'Black',
                        'fontFamily': 'VW Text',
                        'fontWeight': 'bold',
                        'marginTop': '20px',
                        'display': 'block',
                                                'marginRight': 'auto',
                        'padding': '5px 10px',
                        'fontSize': '20px',
                        'backgroundColor': '#e0e0e0',
                        'border': 'none',
                        'borderRadius': '5px',
                        'cursor': 'pointer',
                        'boxShadow': '0px 2px 6px rgba(0, 0, 0, 0.25)'
                    }
                )
            ],
            style={'display': 'none', 'backgroundColor': 'white'}
        )
    ), 
    
    # '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Áreas ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
    html.Div(
        id='conteudo-analitico-area',
        children=[
            html.Button(
                " ↩ ",
                id='botao-voltar-area',
                n_clicks=0,
                style={
                    'color':'Black','fontFamily':'VW Text','fontWeight':'bold',
                    'marginTop':'20px','display':'block','marginLeft':'10px',
                    'marginRight':'auto','padding':'5px 10px','fontSize':'20px',
                    'backgroundColor':'#e0e0e0','border':'none','borderRadius':'5px',
                    'cursor':'pointer','boxShadow':'0px 2px 6px rgba(0,0,0,0.25)'
                }
            )
        ],
        style={'display':'none','backgroundColor':'white'}
    ),

    # '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Relacionamento ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
    html.Div(
        id='conteudo-relacionamento',
        children=[
            html.Button(
                " ↩ ",
                id='botao-voltar-relacionamento',
                n_clicks=0,
                style={
                    'color':'Black','fontFamily':'VW Text','fontWeight':'bold',
                    'marginTop':'20px','display':'block','marginLeft':'10px',
                    'marginRight':'auto','padding':'5px 10px','fontSize':'20px',
                    'backgroundColor':'#e0e0e0','border':'none','borderRadius':'5px',
                    'cursor':'pointer','boxShadow':'0px 2px 6px rgba(0,0,0,0.25)'
                }
            )
        ],
        style={'display':'none','backgroundColor':'white'}
    )
])


# '''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Callbacks ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Geral ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
@app.callback(
    Output('resumo-container', 'style'),
    Output('toggle-resumo-btn', 'children'),
    Input('toggle-resumo-btn', 'n_clicks')
)
def toggle_lista_areas(n):
    aberto = (n or 0) % 2 == 1
    return ({'display': 'block'}, "▼") if aberto else ({'display': 'none'}, "►")


@app.callback(
    Output('lista-pessoas-container', 'style'),
    Output('toggle-lista-btn', 'children'),
    Input('toggle-lista-btn', 'n_clicks')
)
def toggle_lista_pessoas(n):
    aberto = (n or 0) % 2 == 1
    return ({'display': 'block'}, "▼") if aberto else ({'display': 'none'}, "►")  

@app.callback(
    [
        # Subárea
        Output('dropdown-subarea', 'options'),
        Output('dropdown-subarea', 'style'),
        Output('dropdown-subarea', 'value'),

        # Slider período
        Output('periodo-slider', 'min'),
        Output('periodo-slider', 'max'),
        Output('periodo-slider', 'value'),

        # Stores + texto
        Output('area-selecionada', 'data'),
        Output('subarea-selecionada', 'data'),
        Output('periodo-indices', 'data'),
        Output('periodo-texto', 'children'),
    ],
    [
        Input('dropdown-area', 'value'),
        Input('cenario-toggle', 'value'),
        Input('periodo-slider', 'value'),
    ],
    State('dropdown-subarea', 'value')
)
def orquestrador(area, cenario, slider_val, sub_val_atual):
    # ------ Área/Subárea ------
    if not area or area == 'GERAL':
        sub_opts, sub_style, sub_val = [], {'display': 'none'}, None
    else:
        try:
            heranca = her[her['Área'] == area]['Heranca'].unique()
            subareas_df = her[
                (her['Área'].str.startswith(area)) &
                (her['Área'] != area) &
                (her['Heranca'].isin(heranca))
            ]
            if subareas_df.empty:
                sub_opts, sub_style, sub_val = [], {'display': 'none'}, None
            else:
                sub_opts = [
                    {'label': f"{row['Área']} {row.get('Departamento','')}", 'value': row['Área']}
                    for _, row in subareas_df.iterrows()
                ]
                # preserva a seleção anterior se ainda existir nas opções
                valores = {o['value'] for o in sub_opts}
                sub_val = sub_val_atual if sub_val_atual in valores else None
                sub_style = {'width': '270px', 'marginTop': '10px', 'display': 'block'}
        except Exception:
            sub_opts, sub_style, sub_val = [], {'display': 'none'}, None

    # ------ Travar período pelo cenário ------
    if cenario == 'c1':
        min_i, max_i = LT_C1_MIN, LT_C1_MAX
    else:
        min_i, max_i = LT_C2_MIN, LT_C2_MAX

    v0, v1 = (slider_val if slider_val else [min_i, max_i])
    v0 = min(max(v0, min_i), max_i)
    v1 = min(max(v1, min_i), max_i)
    if v0 > v1:
        v0, v1 = min_i, max_i
    slider_min, slider_max, slider_out = min_i, max_i, [v0, v1]

    # ------ Texto do período (com fallback de locale) ------
    def fmt_mes_ano(dt: pd.Timestamp) -> str:
        try:
            return f"{dt.month_name(locale='pt_BR').capitalize()}/{dt.year}"
        except Exception:
            return dt.strftime("%b/%Y").capitalize()

    i0, i1 = slider_out
    d0 = pd.Timestamp(LT_MONTHS[i0])
    d1 = (pd.Timestamp(LT_MONTHS[i1]) + pd.offsets.MonthEnd(1))
    periodo_txt = f"Período selecionado: {fmt_mes_ano(d0)} a {fmt_mes_ano(d1)}"

    # ------ Stores ------
    area_store = area
    subarea_store = sub_val
    periodo_store = [i0, i1]

    return (
        sub_opts, sub_style, sub_val,
        slider_min, slider_max, slider_out,
        area_store, subarea_store, periodo_store, periodo_txt
    )


@app.callback(
    Output('faixa-selecionada', 'data'),
    Output('ultimo-clique', 'data'),
    Input('grafico-heatmap', 'clickData'),
    Input('botao-resetar', 'n_clicks'),
    State('faixa-selecionada', 'data'),
    State('ultimo-clique', 'data'),
    prevent_initial_call=True
)
def atualizar_faixa(clickData, n_reset, faixa_atual, ultimo_clique):
    ctx = callback_context
    trigger = ctx.triggered_id

    faixa_atual = faixa_atual or []
    ultimo_clique = ultimo_clique or {'faixa': None, 'timestamp': 0}

    if trigger == 'grafico-heatmap' and clickData:
        faixa = clickData['points'][0]['x']
        nova_lista = [f for f in faixa_atual if f != faixa] if faixa in faixa_atual else faixa_atual + [faixa]
        return nova_lista, {'faixa': faixa, 'timestamp': time.time()}

    if trigger == 'botao-resetar':
        return [], {'faixa': None, 'timestamp': 0}

    return faixa_atual, ultimo_clique


@app.callback(
    Output('tabela-detalhada-colaboradores', 'children'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('dropdown-chapa', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_tabela(desligados, viajam, chapa_selecionada, faixas, area_selecionada, subarea_selecionada, periodo_idx):
    if not faixas:
        faixas = []
    elif isinstance(faixas, str):
        faixas = [faixas]

    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas)

    media_colab = (
        df_filtrado
        .groupby(['Área', 'Departamento', 'Cadastro', 'Nome', 'Cargo Especifico'])['Contagem Presença']
        .mean()
        .reset_index()
    )
    media_colab.columns = ['Área', 'Departamento', 'Chapa', 'Nome', 'Cargo', 'Média Dia']
    media_colab['Média Dia'] = media_colab['Média Dia'].round(2)

    if chapa_selecionada:
        media_colab = media_colab[media_colab['Chapa'].astype(str) == str(chapa_selecionada)]

    media_colab = aplicar_filtro_faixas(media_colab, 'Média Dia', faixas)
    media_colab['Média %'] = (((media_colab['Média Dia'] / 2) * 100) - 100).round(2)
    media_colab = media_colab.sort_values(by=['Área', 'Média %'], ascending=True)
    media_colab['Média %'] = media_colab['Média %'].astype(str) + ' %'

    tabela = html.Div([
        html.Table([
            html.Thead([
                html.Tr([
                    html.Th(col, style={
                        'textAlign': 'left',
                        'backgroundColor': '#f0f0f0',
                        'padding': '4px',
                        'fontWeight': 'bold',
                        'borderBottom': '2px solid #ccc',
                        'borderRight': '1px solid #ccc'
                    }) for col in media_colab.columns
                ])
            ]),
            html.Tbody([
                html.Tr([
                    html.Td(
                        dcc.Link(str(valor), href=f"/pessoa/{linha[2]}", style={
                            'display': 'block',
                            'width': '100%',
                            'height': '100%',
                            'color': (
                                'green' if (
                                    (isinstance(linha[5], (int, float)) and linha[5] >= 2) or
                                    (isinstance(linha[6], str) and '%' in linha[6] and float(linha[6].replace('%', '').strip()) >= 0)
                                ) else 'red'
                            ),
                            'textDecoration': 'none',
                        }),
                        style={
                            'textAlign': 'left',
                            'padding': '8px',
                            'borderBottom': '1px solid #eee',
                            'borderRight': '1px solid #eee' if i < len(media_colab.columns) - 1 else 'none',
                            'backgroundColor': 'white'
                        }
                    ) for i, valor in enumerate(linha)
                ]) for linha in media_colab.values
            ])
        ], style={
            'borderCollapse': 'separate',
            'borderSpacing': '0',
            'width': '100%',
            'fontSize': '15px',
            'boxShadow': '0 4px 12px rgba(0, 0, 0, 0.05)',
            'borderRadius': '12px',
            'overflow': 'hidden'
        })
    ])

    return tabela


@app.callback(
    Output('dropdown-chapa', 'options'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_opcoes_chapa(desligados, viajam, faixas, area_selecionada, subarea_selecionada, periodo_idx):
    if not faixas:
        faixas = []
    elif isinstance(faixas, str):
        faixas = [faixas]

    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas)

    media_presenca = df_filtrado.groupby(['Cadastro', 'Nome'])['Contagem Presença'].mean().reset_index()
    media_presenca.columns = ['Chapa', 'Nome', 'Média']
    media_presenca = aplicar_filtro_faixas(media_presenca, 'Média', faixas)
    media_presenca = media_presenca.sort_values(by='Nome')
    opcoes = [{'label': f"{row['Chapa']} - {row['Nome']}", 'value': str(row['Chapa'])} for _, row in media_presenca.iterrows()]

    return opcoes


@app.callback(
    Output('grafico-heatmap', 'figure'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_heatmap(desligados, viajam, faixas, area_selecionada, subarea_selecionada, periodo_idx):
    if not faixas:
        faixas = []
    elif isinstance(faixas, str):
        faixas = [faixas]

    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas)

    media_presenca_detalhada = df_filtrado.groupby('Cadastro')['Contagem Presença'].mean().reset_index()
    media_presenca_detalhada.columns = ['Cadastro', 'Média Presença']

    media_por_pessoa = media_presenca_detalhada[['Cadastro', 'Média Presença']].copy()
    media_por_pessoa['Média Presença'] = media_por_pessoa['Média Presença'].clip(lower=0, upper=7)

    bins = list(range(0, 8))
    labels_bins = [f"{i} a {i+1}" for i in range(7)]

    media_por_pessoa['Faixa'] = pd.cut(
        media_por_pessoa['Média Presença'],
        bins=bins,
        labels=labels_bins,
        right=False,
        include_lowest=True
    )

    if faixas:
        dados_faixa = media_por_pessoa[media_por_pessoa['Faixa'].isin(faixas)]
    else:
        dados_faixa = media_por_pessoa.copy()

    contagem_faixa = dados_faixa['Faixa'].value_counts().reindex(labels_bins, fill_value=0)
    x_faixas = contagem_faixa.index.tolist()
    y_faixas = contagem_faixa.values.tolist()

    num_faixas = max(1, len(x_faixas))
    colorscale = [None] * num_faixas

    quant_vermelho, indices_vermelho = [], []
    quant_amarelo, indices_amarelo = [], []
    quant_verde, indices_verde = [], []

    for i, faixa in enumerate(x_faixas):
        fmin = int(faixa.split(" ")[0])
        qtd = y_faixas[i]
        if fmin < 2:
            quant_vermelho.append(qtd)
            indices_vermelho.append(i)
        elif fmin > 4:
            quant_amarelo.append(qtd)
            indices_amarelo.append(i)
        else:
            quant_verde.append(qtd)
            indices_verde.append(i)

    min_verm, max_verm = (min(quant_vermelho), max(quant_vermelho)) if quant_vermelho else (0, 1)
    min_am, max_am = (min(quant_amarelo), max(quant_amarelo)) if quant_amarelo else (0, 1)
    min_vd, max_vd = (min(quant_verde), max(quant_verde)) if quant_verde else (0, 1)

    for idx in indices_vermelho:
        qtd = y_faixas[idx]
        t = (qtd - min_verm) / (max_verm - min_verm) if max_verm > min_verm else 0
        pos = idx / (num_faixas - 1) if num_faixas > 1 else 0
        colorscale[idx] = [pos, interpolar_cor_hex("#fc9898", "#ef6464", t)]

    for idx in indices_amarelo:
        qtd = y_faixas[idx]
        t = (qtd - min_am) / (max_am - min_am) if max_am > min_am else 0
        pos = idx / (num_faixas - 1) if num_faixas > 1 else 0
        colorscale[idx] = [pos, interpolar_cor_hex("#fff9a9", "#ffef09", t)]

    for idx in indices_verde:
        qtd = y_faixas[idx]
        t = (qtd - min_vd) / (max_vd - min_vd) if max_vd > min_vd else 0
        pos = idx / (num_faixas - 1) if num_faixas > 1 else 0
        colorscale[idx] = [pos, interpolar_cor_hex("#9effd0", "#3fbe81", t)]

    for i in range(num_faixas):
        if colorscale[i] is None:
            pos = i / (num_faixas - 1) if num_faixas > 1 else 0
            colorscale[i] = [pos, "#cccccc"]

    fig_heat = go.Figure(data=go.Heatmap(
        z=[list(range(num_faixas))],
        x=x_faixas,
        y=["Distribuição"],
        colorscale=colorscale,
        hovertemplate="Faixa: %{x}<br>Qtd Pessoas: %{customdata}<extra></extra>",
        customdata=[y_faixas] if y_faixas else [[]],
        showscale=False
    ))

    annotations = [
        dict(x=x_faixas[i], y="Distribuição", text=str(val), showarrow=False,
             font=dict(color="black", size=15, family="VW Text"),
             xanchor="center", yanchor="middle")
        for i, val in enumerate(y_faixas)
    ]

    fig_heat.update_layout(
        title={'text': "Distribuição de Colaboradores por Faixa", 'font': {'family': 'VW Text', 'color': '#000000', 'size': 24}},
        xaxis_title="Faixa de Média (dias/semana)",
        yaxis_title="",
        xaxis=dict(type='category'),
        yaxis=dict(showticklabels=False),
        height=250,
        showlegend=False,
        annotations=annotations
    )

    return fig_heat


@app.callback(
    Output('grafico-dispersao', 'figure'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_dispersao(desligados, viajam, faixas, area_selecionada, subarea_selecionada, periodo_idx):
    if not faixas:
        faixas = []
    elif isinstance(faixas, str):
        faixas = [faixas]

    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas)

    media_presenca_detalhada = df_filtrado.groupby('Cadastro')['Contagem Presença'].mean().reset_index()
    media_presenca_detalhada.columns = ['Cadastro', 'Média Presença']

    media_presenca_detalhada = aplicar_filtro_faixas(media_presenca_detalhada, 'Média Presença', faixas)
    media_presenca_detalhada['Média Presença Arred'] = media_presenca_detalhada['Média Presença'].round(1)

    dados_disp = media_presenca_detalhada.groupby('Média Presença Arred')['Cadastro'].count().reset_index()
    dados_disp.columns = ['Média Presença', 'Qtd Pessoas']

    fig_dispersao = go.Figure()
    fig_dispersao.add_trace(go.Scatter(
        x=dados_disp['Média Presença'],
        y=dados_disp['Qtd Pessoas'],
        mode='markers+text',
        marker=dict(size=12, color="#0003b4"),
        hovertemplate="Qtd Pessoas: %{y}<br>Média Presença: %{x:.1f}<extra></extra>"
    ))

    fig_dispersao.update_layout(
        title={'text': f"Dispersão Geral ({', '.join(faixas) if faixas else 'Todos'})", 'font': {'family': 'VW Text', 'color': '#000000', 'size': 24}},
        xaxis_title="Média de Presença Semanal",
        yaxis_title="Qtd de Pessoas",
        height=500
    )

    fig_dispersao = aplicar_tema_pb(fig_dispersao)
    return fig_dispersao


@app.callback(
    Output('grafico-presenca-cargo', 'figure'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_presenca_cargo(desligados, viajam, faixas, area_selecionada, subarea_selecionada, periodo_idx):
    if not faixas:
        faixas = []
    elif isinstance(faixas, str):
        faixas = [faixas]

    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas)

    media_presenca_detalhada = df_filtrado.groupby('Cadastro')['Contagem Presença'].mean().reset_index()
    media_presenca_detalhada.columns = ['Cadastro', 'Média Presença']
    media_presenca_detalhada = media_presenca_detalhada.merge(
        df_filtrado[['Cadastro', 'Cargo']].drop_duplicates(),
        on='Cadastro',
        how='left'
    )
    media_presenca_detalhada = aplicar_filtro_faixas(media_presenca_detalhada, 'Média Presença', faixas)

    fig_cargo = go.Figure()
    for cod, nome, cor in [('M', 'Mensalistas', '#7cc3e9'), ('E', 'Executivos', "#1574b3")]:
        df_aux = media_presenca_detalhada[media_presenca_detalhada['Cargo'] == cod].copy()
        if df_aux.empty:
            continue
        df_aux['Média Presença Arred'] = df_aux['Média Presença'].round(1)
        df_cargo = df_aux.groupby('Média Presença Arred')['Cadastro'].count().reset_index()
        df_cargo.columns = ['Média Presença', 'Qtd Pessoas']
        fig_cargo.add_trace(go.Scatter(
            x=df_cargo['Média Presença'],
            y=df_cargo['Qtd Pessoas'],
            mode='markers+text',
            name=nome,
            marker=dict(size=12, color=cor),
            hovertemplate=(f"{nome}<br>Qtd Pessoas: %{{y}}<br>Média Presença: %{{x:.1f}}<extra></extra>")
        ))

    fig_cargo.update_layout(
        title={'text': f"Presença por Cargo ({', '.join(faixas) if faixas else 'Todos'})", 'font': {'family': 'VW Text', 'color': '#000000', 'size': 24}},
        xaxis_title="Média de Presença Semanal",
        yaxis_title="Qtd de Pessoas",
        height=500,
        legend=dict(x=1, y=0.877, orientation='v', xanchor='right', yanchor='bottom',
                    font=dict(size=12), bgcolor='#ffffff')
    )

    fig_cargo = aplicar_tema_pb(fig_cargo)
    return fig_cargo
@app.callback(
    Output('grafico-boxplot', 'figure'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_boxplot(desligados, viajam, faixas, area_selecionada, subarea_selecionada, periodo_idx):
    # Se nenhuma área específica foi selecionada, não carrega o gráfico
    if not area_selecionada or area_selecionada == "GERAL":
        # pode retornar um gráfico vazio ou simplesmente não atualizar
        return figura_vazia_linha("Selecione uma área no dropdown para visualizar o Boxplot.")
    if not faixas:
        faixas = []
    elif isinstance(faixas, str):
        faixas = [faixas]

    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas)

    medias_individuais = (
        df_filtrado.groupby(['Cadastro', 'Nome', 'Área'])['Contagem Presença']
        .mean().reset_index().round(2)
    )
    medias_individuais.columns = ['Chapa', 'Nome', 'Área', 'Média Presença']

    if area_selecionada and area_selecionada != 'GERAL':
        if subarea_selecionada:
            medias_individuais = medias_individuais[medias_individuais['Área'] == subarea_selecionada]
        else:
            medias_individuais = medias_individuais[medias_individuais['Área'].str.startswith(area_selecionada)]

    medias_individuais = aplicar_filtro_faixas(medias_individuais, 'Média Presença', faixas)

    fig_boxplot = go.Figure()
    AZUL_LINHA = "#3299dd"
    AZUL_FILL = 'rgba(21,116,179,0.30)'
    VERM_LINHA = "#FF0000"
    VERM_FILL = 'rgba(211,47,47,0.30)'

    for area_nome, dados_area in medias_individuais.groupby('Área'):
        s = dados_area['Média Presença'].dropna()
        tem_abaixo = (s.min() < 2)
        cor_linha = VERM_LINHA if tem_abaixo else AZUL_LINHA
        cor_fill = VERM_FILL if tem_abaixo else AZUL_FILL

        if not s.empty:
            fig_boxplot.add_trace(go.Box(
                y=s,
                name=area_nome,
                boxpoints='outliers',
                line=dict(color=cor_linha, width=2),
                marker=dict(color=cor_linha, size=4),
                fillcolor=cor_fill,
                hovertemplate=area_nome + "<br>Média: %{y:.2f}<extra></extra>"
            ))

    titulo_bp = f"Boxplot de Presença por Área ({', '.join(faixas) if faixas else 'Todos'})"
    fig_boxplot.update_layout(
        title={'text': titulo_bp, 'font': {'family': 'VW Text', 'size': 24}},
        yaxis_title="Média de Presença Semanal",
        xaxis=dict(
            title="",
            showline=True,
            zeroline=True,
            rangeslider=dict(
                visible=True,
                bgcolor="white",
                bordercolor="black",
                borderwidth=1
            )
        ),
        boxmode='group',
        height=600,
        shapes=[dict(
            type="line", xref="paper", yref="y", x0=0, x1=1, y0=2, y1=2,
            line=dict(color="#A10000", width=3, dash="solid")
        )]
    )

    fig_boxplot = aplicar_tema_pb(fig_boxplot, so_y_boxplot=True)
    return fig_boxplot


@app.callback(
    Output('tabela-informacoes-colaboradores', 'children'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_info(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx):
    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, [])

    total_colaboradores = df_filtrado['Cadastro'].nunique()
    total_mensalistas = df_filtrado[df_filtrado['Cargo'] == 'M']['Cadastro'].nunique() if 'Cargo' in df_filtrado.columns else 0
    total_executivos = df_filtrado[df_filtrado['Cargo'] == 'E']['Cadastro'].nunique() if 'Cargo' in df_filtrado.columns else 0
    media_presenca_geral = round(df_filtrado['Contagem Presença'].mean(), 2) if not df_filtrado.empty else 0.0
    media_presenca_mensalistas = round(df_filtrado[df_filtrado['Cargo'] == 'M']['Contagem Presença'].mean(), 2) if 'Cargo' in df_filtrado.columns else 0.0
    media_presenca_executivos = round(df_filtrado[df_filtrado['Cargo'] == 'E']['Contagem Presença'].mean(), 2) if 'Cargo' in df_filtrado.columns else 0.0

    tabela_info = html.Table([
        html.Thead([html.Tr([html.Th("Informações de Colaboradores", colSpan=2)], style={'fontSize': '20px'})]),
        html.Tbody([
            html.Tr([html.Td("Total de Colaboradores:"), html.Td(total_colaboradores)]),
            html.Tr([html.Td("Mensalistas:"), html.Td(total_mensalistas)]),
            html.Tr([html.Td("Executivos:"), html.Td(total_executivos)]),
            html.Tr([html.Td(html.Br()), html.Td()]),
            html.Tr([html.Th("Média(%) de Presença por Semana", colSpan=2)], style={'fontSize': '20px'}),
            html.Tr([html.Td("Total:"), html.Td(f"{media_presenca_geral}%")]),
            html.Tr([html.Td("Mensalistas:"), html.Td(f"{media_presenca_mensalistas}%")]),
            html.Tr([html.Td("Executivos:"), html.Td(f"{media_presenca_executivos}%")])
        ])
    ], style={'fontSize': '16px', 'width': '100%'})

    return tabela_info


@app.callback(
    Output('resumo-container', 'children'),
    Input('desligados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data'),
    Input('periodo-indices', 'data')
)
def atualizar_resumo(desligados, viajam, faixas, area_selecionada, subarea_selecionada, periodo_idx):
    if not faixas:
        faixas = []
    elif isinstance(faixas, str):
        faixas = [faixas]

    df_filtrado = filtrar_base(desligados, viajam, area_selecionada, subarea_selecionada, periodo_idx, faixas)

    media_colab_resumo = (
        df_filtrado.groupby(['Área', 'Departamento', 'Cadastro', 'Nome'])['Contagem Presença']
        .mean().reset_index()
    )
    media_colab_resumo.columns = ['Área', 'Departamento', 'Chapa', 'Nome', 'Média Dia']
    media_colab_resumo['Média Dia'] = media_colab_resumo['Média Dia'].round(2)
    media_colab_resumo = media_colab_resumo.merge(
        df_filtrado[['Cadastro', 'Cargo']].drop_duplicates(),
        how='left', left_on='Chapa', right_on='Cadastro'
    )

    resumo_area = media_colab_resumo.groupby(['Área', 'Departamento']).agg(
        Qtd_Pessoas=('Chapa', 'count'),
        Qtd_Mensalistas=('Cargo', lambda x: (x == 'M').sum()),
        Qtd_Executivos=('Cargo', lambda x: (x != 'M').sum()),
        Media_Dia=('Média Dia', 'mean')
    ).reset_index()
    resumo_area['Média Dia'] = resumo_area['Media_Dia'].round(2)
    resumo_area['Média %'] = ((resumo_area['Média Dia'] / 2) * 100 - 100).round(2)
    resumo_area = aplicar_filtro_faixas(resumo_area, 'Média Dia', faixas)

    def _td_link_area(texto, area, extra_style=None):
        base_style = {
            'display': 'block', 'width': '100%', 'height': '100%',
            'color': 'inherit', 'textDecoration': 'none', 'padding': '8px'
        }
        if extra_style:
            base_style.update(extra_style)
        return dcc.Link(str(texto), href=f"/area/{slug(area)}", style=base_style)

    def td_style(align='left', width='auto'):
        return {
            'borderBottom': '1px solid #eee',
            'borderRight': '1px solid #eee',
            'cursor': 'pointer',
            'textAlign': align,
            'width': width,
            'padding': '5px 10px'
        }

    def th_style(align='left', width='auto'):
        return {
            'padding': '10px 10px',
            'backgroundColor': '#f0f0f0',
            'textAlign': align,
            'fontWeight': 'bold',
            'borderBottom': '2px solid #ccc',
            'borderRight': '1px solid #ccc',
            'width': width
        }

    tabela_resumo_area = html.Div([
        html.Div(children=[
            html.Table([
                html.Thead([
                    html.Tr([
                        html.Th('Área', style=th_style('left', '0%')),
                        html.Th('Departamento', style=th_style('left', '0%')),
                        html.Th('Mensalistas', style=th_style('left', '0%')),
                        html.Th('Executivos', style=th_style('left', '0%')),
                        html.Th('Total', style=th_style('left', '0%')),
                        html.Th('Média Dia', style=th_style('left', '0%')),
                        html.Th('Média %', style=th_style('left', '0%')),
                    ])
                ]),
                html.Tbody([
                    html.Tr([
                        html.Td(_td_link_area(row['Área'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Departamento'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Qtd_Mensalistas'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Qtd_Executivos'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Qtd_Pessoas'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Média Dia'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(f"{row['Média %']} %", row['Área']),
                                style=td_style('left', '0%')),
                    ],
                    style={'color': 'red'} if row['Média Dia'] < 2 else {'color': 'green'})
                    for _, row in resumo_area.iterrows()
                ])
            ], style={
                'borderCollapse': 'collapse',
                'width': '100%',
                'fontSize': '15px',
                'boxShadow': '0 4px 12px rgba(0, 0, 0, 0.05)',
                'borderRadius': '12px',
                'overflow': 'hidden'
            })
        ])
    ])

    return tabela_resumo_area


'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Individual ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
@app.callback(
    Output('conteudo-individual', 'children'),
    Output('conteudo-individual', 'style'),
    Output('pagina-geral', 'style', allow_duplicate=True),
    Output('url', 'pathname', allow_duplicate=True),
    Input('url', 'pathname'),
    Input('botao-voltar', 'n_clicks'),
    prevent_initial_call=True
)
def exibir_conteudo_individual(pathname, n_clicks):
    # Voltar
    if n_clicks is not None and n_clicks > 0:
        return None, estilo_oculto, estilo_exibir, "/"

    # Roteamento
    if not pathname or '/pessoa/' not in pathname:
        return None, estilo_oculto, no_update, no_update

    chapa_input = pathname.split('/pessoa/')[1]
    if not chapa_input:
        return None, estilo_oculto, estilo_exibir

    # Filtra dados da pessoa
    df_chapa_semana = df_semana_merged[df_semana_merged['Cadastro'] == str(chapa_input)].copy()
    if df_chapa_semana.empty:
        return None, estilo_oculto, estilo_exibir

    nome_input = df_chapa_semana['Nome'].iloc[0].title()

    # Semana-Ano
    df_chapa_semana['Semana_Ano'] = (
        df_chapa_semana['Ano'].astype(str) + "-" + df_chapa_semana['Semana por Ano'].astype(str).str.zfill(2)
    )

    # Presenças por semana
    dias_presenca_por_semana = df_chapa_semana.groupby('Semana_Ano')['Contagem Presença'].mean()
    media_presenca_semana = round(dias_presenca_por_semana.mean(), 2)

    # Regra mínima por semana (mudança a partir de Maio/2025: mínimo 3 dias)
    min_req_por_semana = dias_presenca_por_semana.index.to_series().apply(min_presencial_exigido)
    media_min_req = round(min_req_por_semana.mean(), 2) if len(min_req_por_semana) else MIN_PRESENCIAL_APOS_MUDANCA
    if pd.isna(media_min_req) or media_min_req <= 0:
        media_min_req = MIN_PRESENCIAL_APOS_MUDANCA

    percentual_da_media = round(((media_presenca_semana / media_min_req) * 100) - 100, 2)
    percentuais_presenca = calcular_percentual_presenca(dias_presenca_por_semana)

    # Distribuição de presenças (0..7)
    df_filtrado_presenca = df_semana_merged[
        df_semana_merged['Cadastro'].astype(str) == str(chapa_input)
    ]
    distribuicao_presencas = (
        df_filtrado_presenca['Contagem Presença'].value_counts().sort_index()
        if not df_filtrado_presenca.empty
        else df_chapa_semana['Contagem Presença'].value_counts().sort_index()
    )
    x = sorted(set(distribuicao_presencas.index))
    y_presencas = distribuicao_presencas.reindex(x, fill_value=0)

    # Série temporal (ordenada)
    dias_presenca_por_semana_ordenado = dias_presenca_por_semana.sort_index()
    x_semanas = list(range(1, len(dias_presenca_por_semana_ordenado) + 1))
    y_presenca_por_semana = dias_presenca_por_semana_ordenado.values

    # Datas do heatmap geral
    df_chapa_data = df_datas[df_datas['Cadastro'] == str(chapa_input)][['Data', 'Cadastro']].copy()
    df_chapa_data['Data'] = pd.to_datetime(df_chapa_data['Data'])

    # Datas reais da chapa em df_datas
    df_chapa_data = df_datas[df_datas['Cadastro'] == str(chapa_input)][['Data', 'Cadastro']].copy()
    df_chapa_data['Data'] = pd.to_datetime(df_chapa_data['Data'], errors='coerce')

    # Se quiser limitar por admissão/demissão, usa o range dentro desse intervalo:
    # garanta que são Timestamps
    min_real = pd.Timestamp(df_chapa_data['Data'].min())
    max_real = pd.Timestamp(df_chapa_data['Data'].max())

    m0 = min_real.month_name(locale='pt_BR').capitalize()
    m1 = max_real.month_name(locale='pt_BR').capitalize()

    mes_ano_inicio = f"{m0} de {min_real.year}"
    mes_ano_fim    = f"{m1} de {max_real.year}"

    # cria rótulos dinâmicos
    periodo_label_completo = f"{mes_ano_inicio} a {mes_ano_fim}"

    mes_ano_inicio = min_real.strftime("%b %Y").capitalize()
    mes_ano_fim    = max_real.strftime("%b %Y").capitalize()
    periodo_label_curto = f"{mes_ano_inicio} a {mes_ano_fim}"

    # Guard contra base vazia
    if pd.isna(min_real) or pd.isna(max_real):
        # Sem datas -> não há como montar o heatmap
        return None, estilo_oculto, estilo_exibir

    # Se quiser cruzar com adm/dem, aplique aqui:
    data_adm, data_dem = obter_periodo_admissao(chapa_input)
    if data_adm is not None:
        min_real = max(min_real.normalize(), pd.to_datetime(data_adm).normalize())
    if data_dem is not None:
        max_real = min(max_real.normalize(), pd.to_datetime(data_dem).normalize())

    # Gera TODAS as datas no período [min_real .. max_real]
    todas_datas = pd.date_range(min_real.normalize(), max_real.normalize(), freq='D')

    # Monta dataframe de referência (com rótulos ISO semanais e dias em PT)
    _dias_pt = {0:'Segunda',1:'Terça',2:'Quarta',3:'Quinta',4:'Sexta',5:'Sábado',6:'Domingo'}
    inicio_sem = todas_datas - pd.to_timedelta(todas_datas.weekday, unit='D')
    fim_sem = inicio_sem + pd.to_timedelta(6, unit='D')

    df_todas_datas = pd.DataFrame({
        "Data": todas_datas,
        "Ano": todas_datas.isocalendar().year.astype(int),
        "Semana_Num": todas_datas.isocalendar().week.astype(int),
        "Início da Semana": inicio_sem,
        "Fim da Semana": fim_sem,
    })
    df_todas_datas["Semana_Label"] = (
        df_todas_datas["Ano"].astype(str) + "-" + df_todas_datas["Semana_Num"].astype(str).str.zfill(2)
        + " | " + df_todas_datas["Início da Semana"].dt.strftime('%d/%m/%Y')
        + "-" + df_todas_datas["Fim da Semana"].dt.strftime('%d/%m/%Y')
    )
    df_todas_datas["Dia da Semana"] = df_todas_datas["Data"].dt.weekday.map(_dias_pt)

    # Merge de presença (1 se existe linha para a Data daquela chapa)
    df_merged = df_todas_datas.merge(df_chapa_data[["Data", "Cadastro"]], on="Data", how="left")
    df_merged["Presente"] = df_merged["Cadastro"].notna().astype(int)

    tabela_heatmap = (
        df_merged
        .pivot_table(index="Semana_Label", columns="Dia da Semana", values="Presente", aggfunc="sum", fill_value=0)
        .reindex(columns=ordenacao_dias)
        .sort_index()
    )

    df_todas_datas["Semana_Num"] = df_todas_datas["Data"].dt.isocalendar().week
    df_todas_datas["Início da Semana"] = df_todas_datas["Data"] - pd.to_timedelta(df_todas_datas["Data"].dt.weekday, unit="d")
    df_todas_datas["Fim da Semana"] = df_todas_datas["Início da Semana"] + pd.to_timedelta(6, unit="d")
    df_todas_datas["Semana_Label"] = (
        df_todas_datas["Ano"].astype(str) + "-" + df_todas_datas["Semana_Num"].astype(str).str.zfill(2)
        + " | " + df_todas_datas["Início da Semana"].dt.strftime('%d/%m/%Y')
        + "-" + df_todas_datas["Fim da Semana"].dt.strftime('%d/%m/%Y')
    )
    df_todas_datas["Dia da Semana"] = df_todas_datas["Data"].dt.day_name().map(dias_portugues).fillna(
        df_todas_datas["Data"].dt.day_name()
    )

    df_merged = df_todas_datas.merge(df_chapa_data[["Data", "Cadastro"]], how="left", on="Data")
    tabela_heatmap = df_merged.pivot_table(
        index="Semana_Label",
        columns="Dia da Semana",
        values="Cadastro",
        aggfunc="count",
        fill_value=0
    ).reindex(columns=ordenacao_dias)

    # ======= Figura (Matplotlib) =======
    fig = plt.figure(figsize=(28, 45))
    gs = gridspec.GridSpec(3, 2, height_ratios=[0.7, 0.5, 3.0], width_ratios=[1.9, 1.9])
    gs_heatmaps = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[2, 0],
                                                   height_ratios=[max(1, tabela_heatmap.shape[0]), 1], hspace=0.02)

    # Título e linha de topo
    fig.suptitle(
        f"{nome_input} — Chapa {chapa_input}",
        x=0.01, y=1.012, fontsize=30, fontweight='bold', ha='left'
    )
    fig.lines.append(
        Line2D([0.01, 0.99], [0.992, 0.992], color='gray', linestyle='dashed', linewidth=1.5,
            alpha=0.6, transform=fig.transFigure, figure=fig)
    )
    fig.text(0.5, 0.953, f"Relatório de Frequência e Regime de Trabalho no Período de {periodo_label_completo}\n\n\n",
             fontsize=28, ha='center')

    # ~~~~~~~~~~~~~~~~~~~~~ Subplot 1 - Gauge ~~~~~~~~~~~~~~~~~~~~~
    ax_clock = fig.add_subplot(gs[0, 0], polar=True)
    ax_clock.set_theta_offset(np.pi)
    ax_clock.set_theta_direction(-1)
    current_val = float(media_presenca_semana)
    colors = ['#ffb3b3', '#f4cccc', '#d9ead3', '#b6e6c5', '#96ddb1', '#6ccf9a', '#3fbe81']
    bounds = [0, 1, 2, 3, 4, 5, 6, 7]
    ax_clock.set_yticklabels([]); ax_clock.set_xticklabels([]); ax_clock.set_axis_off()
    for i in range(len(colors)):
        start = np.pi * (bounds[i] / 7)
        end = np.pi * (bounds[i + 1] / 7)
        ax_clock.barh(1, end - start, left=start, height=0.4, color=colors[i], edgecolor='black')
    tick_vals = [0, 1, 2, 3, 4, 5, 6, 7]
    tick_angles = [np.pi * (val / 7) for val in tick_vals]
    for ang, label in zip(tick_angles, [f"{val:.1f}" for val in tick_vals]):
        ax_clock.text(ang, 1.3, label, ha='center', va='center', fontsize=17, color='gray')
    angle = np.pi * (min(current_val, 7) / 7)
    ax_clock.arrow(angle, 0, 0, 0.95, width=0.03, head_width=0.08, head_length=0.15, fc='black', ec='black', zorder=5)
    ax_clock.text(0, 0, f"\n\n\n{current_val:.2f} dias/semana", fontsize=20, ha='center', va='center', color='#444')
    rect = Rectangle((0.264, 0.385), 0.48, 0.07, linewidth=2, edgecolor='black', facecolor='none',
                     transform=ax_clock.transAxes, zorder=10)
    ax_clock.add_patch(rect)
    ax_clock.set_title(f"\n\nFrequência da Contagem de Presencial\npor Semana ({periodo_label_curto})\n", va='bottom', fontsize=20, pad=35)
    if current_val > media_min_req:
        ax_clock.text(0.01, 0, f"\n\n\n\n\n Acima do necessário! \n+{percentual_da_media}%",
                      ha='center', va='center', fontsize=34, color='green', fontweight='bold',
                      path_effects=[patheffects.withStroke(linewidth=3, foreground='white')])
    elif current_val < media_min_req:
        ax_clock.text(0.01, 0, f"\n\n\n\n\n Abaixo do esperado! \n{percentual_da_media}%",
                      ha='center', va='center', fontsize=34, color='red', fontweight='bold',
                      path_effects=[patheffects.withStroke(linewidth=3, foreground='white')])
    ax_clock.patch.set_facecolor('white'); ax_clock.patch.set_alpha(0)
    ax_clock.text(-0.15, -0.16,
                  'Indicador de frequência presencial semanal de "0" a "7" dias úteis por semana.\n'
                  'Esse gráfico ajuda a saber a média que o colaborador está presente na planta.',
                  ha='left', va='center', fontsize=14, wrap=True, transform=ax_clock.transAxes)

    # ~~~~~~~~~~~~~~~~~~~~~ Subplot 2 - Distribuição por contagem semanal ~~~~~~~~~~~~~~~~~~~~~
    ax_line = fig.add_axes([0.618, 0.8195, 0.35, 0.120])
    ax_line.plot(x, y_presencas, color='#3fbe81', linewidth=4, label='Presencial')
    ax_line.fill_between(x, y_presencas, color='#96ddb1', alpha=0.5)
    for xi, yi in zip(x, y_presencas):
        ax_line.text(xi, yi + 0.14, f"{yi}", ha='center', fontsize=17)
    ax_line.set_xticks(x)
    ax_line.set_xlabel('Qtd. de Presencial na Semana', fontsize=20)
    ax_line.set_ylabel('Número de Ocorrências', fontsize=20)
    ax_line.set_title(f'Distribuição da Contagem de Presencial\npor Semana ({periodo_label_curto})', fontsize=20, pad=25)
    ax_line.grid(True, linestyle='--')
    ax_line.tick_params(axis='x', labelsize=17)
    ax_line.tick_params(axis='y', labelsize=17)
    ax_line.text(0, -0.28,
                 "Quantidade de semanas para uma certa quantidade de presença.\n"
                 "Esse gráfico ajuda a visualizar picos de frequência, transições de regime.",
                 ha='left', va='center', fontsize=14, wrap=True, transform=ax_line.transAxes)

    # ~~~~~~~~~~~~~~~~~~~~~ Subplot 3 - Série temporal + tendência ~~~~~~~~~~~~~~~~~~~~~
    x_np = np.array(sorted(x_semanas))
    y_np = np.array([y for _, y in sorted(zip(x_semanas, y_presenca_por_semana))])
    ax_line2 = fig.add_axes([0.204, 0.626, 0.680, 0.119])
    ax_line2.plot(x_np, y_np, color='#3fbe81', linewidth=4, label='Presencial')
    if len(x_np) >= 2:
        deg = 1 if len(x_np) == 2 else 2
        coef_poly = np.polyfit(x_np, y_np, deg=deg)
        poly_fn = np.poly1d(coef_poly)
        x_pred = np.linspace(x_np[0], x_np[-1], 500)
        fim = poly_fn(x_pred[-1])
        cor_tendencia = 'red' if fim < media_min_req else '#256b2b'
        ax_line2.plot(x_pred, poly_fn(x_pred), color=cor_tendencia, linestyle='--', linewidth=2, label='Tendência')
    ax_line2.set_xlim(x_np[0], x_np[-1] if len(x_np) else 1)
    anos = [label.split("-")[0] for label in dias_presenca_por_semana_ordenado.index]
    semanas = [int(label.split("-")[1]) for label in dias_presenca_por_semana_ordenado.index]
    ax_line2.set_xticks(x_semanas)
    ax_line2.set_xticklabels(semanas, fontsize=16)
    semanas_por_ano = defaultdict(list)
    for xv, ano in zip(x_semanas, anos):
        semanas_por_ano[ano].append(xv)
    for ano, semanas_ano in semanas_por_ano.items():
        centro = np.mean(semanas_ano)
        ax_line2.text(centro, -0.8, ano, transform=ax_line2.transData, ha='center', va='center', fontsize=18, color='gray')
    for i in range(1, len(anos)):
        if anos[i] != anos[i - 1]:
            x_val = x_semanas[i] - 0.5
            x_fig, _ = ax_line2.transData.transform((x_val, 0))
            x_fig_norm = fig.transFigure.inverted().transform((x_fig, 0))[0]
            bbox = ax_line2.get_position()
            y_fig = bbox.y0 - 0.015
            fig.lines.append(Line2D([x_fig_norm, x_fig_norm], [y_fig, y_fig + 0.015],
                                    transform=fig.transFigure, linestyle='--', color='gray', linewidth=2, alpha=0.7))
    ax_line2.set_xlabel('', fontsize=20)
    ax_line2.set_ylabel('Quantidade de Presença', fontsize=20)
    ax_line2.set_title(f'Distribuição da Contagem de Presencial em cada Semana ({periodo_label_curto})', fontsize=20, pad=25)
    ax_line2.grid(True, linestyle='--')
    ax_line2.legend(fontsize=16, loc='upper right')
    ax_line2.tick_params(axis='x', labelsize=17)
    ax_line2.tick_params(axis='y', labelsize=17)
    ax_line2.text(0, -0.31,
                  "Número de presença em cada semana de todo esse período, esse gráfico ajuda a visualizar picos de frequência, "
                  "transições de regime e a tendência do colaborador.\n\n\n\n\n\n",
                  ha='left', va='center', fontsize=14, wrap=True, transform=ax_line2.transAxes)

    # ~~~~~~~~~~~~~~~~~~~~~ Subplot 4 - Heatmap (semanas x dias) ~~~~~~~~~~~~~~~~~~~~~
    ax_heatmap = fig.add_subplot(gs_heatmaps[0])
    if tabela_heatmap.empty:
        ax_heatmap.axis('off')
    else:
        sns.heatmap(
            tabela_heatmap,
            cmap=ListedColormap(['#d9ead3']),
            linewidths=1, linecolor='gray',
            annot=True, annot_kws={"size": 17},
            cbar=False, ax=ax_heatmap
        )

    # Coloração por presença baixa
    for y_idx, semana_label in enumerate(tabela_heatmap.index):
        semana_codigo = semana_label.split(" |")[0].replace(" ", "-")
        if semana_codigo in dias_presenca_por_semana.index:
            qtd_presenca = dias_presenca_por_semana.loc[semana_codigo]
            req_min = min_presencial_exigido(semana_codigo)
            if qtd_presenca >= req_min:
                cor_destaque = "#d9ead3"
            else:
                cor_destaque = "#ffb3b3" if qtd_presenca == 0 else "#f4cccc"
            for col_idx in range(tabela_heatmap.shape[1]):
                ax_heatmap.add_patch(
                    plt.Rectangle((col_idx, y_idx), 1, 1, facecolor=cor_destaque,
                                  edgecolor='gray', linewidth=2, zorder=1)
                )
    ax_heatmap.set_title(f'\n\n\n\n\n\n\nDistribuição da Contagem de Presencial por\nAno e Semana para cada Dia da Semana ({periodo_label_curto})\n',
                         fontsize=20, pad=40)
    ax_heatmap.xaxis.set_label_position('top')
    ax_heatmap.xaxis.tick_top()
    ax_heatmap.tick_params(axis='x', bottom=False, labelbottom=False)
    ax_heatmap.set_ylabel(''); ax_heatmap.set_xlabel('')
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), fontsize=20)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), fontsize=20)

    # ~~~~~~~~~~~~~~~~~~~~~ Subplot 5 - Texto + mini heatmap de semanas baixas ~~~~~~~~~~~~~~~~~~~~~
    ax_report = fig.add_subplot(gs[2, 1])
    relatorio_formatado = gerar_relatorio_formatado(nome_input, chapa_input, media_presenca_semana, percentuais_presenca, periodo_label_completo)
    ax_report.text(0, 1.058, relatorio_formatado, fontsize=20, ha='left', va='top', wrap=True, usetex=False)
    ax_report.axis('off')
    semanas_baixa = [
        sem for sem, qtd in dias_presenca_por_semana.items()
        if (0.0 if pd.isna(qtd) else float(qtd)) < float(min_presencial_exigido(sem))
    ]
    tabela_heatmap_baixa = tabela_heatmap[
        tabela_heatmap.index.str.split(r" \|").str[0].isin(semanas_baixa)
    ]

    if not tabela_heatmap_baixa.empty:
        left = 0.69; width = 0.30
        num_linhas = tabela_heatmap_baixa.shape[0]
        altura_base = 0.28; linhas_base = 36
        height = (num_linhas / linhas_base) * altura_base
        top = 0.410; bottom = top - height
        ax_heatmap3 = fig.add_axes([left, bottom, width, height])
        sns.heatmap(tabela_heatmap_baixa, cmap=ListedColormap(['#ffb3b3', '#f4cccc']),
                    linewidths=1, linecolor='gray', annot=True, annot_kws={'size': 14},
                    cbar=False, ax=ax_heatmap3)
        for y_idx, semana_label in enumerate(tabela_heatmap_baixa.index):
            semana_codigo = semana_label.split(" |")[0].replace(" ", "-")
            if semana_codigo in dias_presenca_por_semana.index:
                qtd_presenca = dias_presenca_por_semana.loc[semana_codigo]
                cor_destaque = "#ffb3b3" if qtd_presenca == 0 else "#f4cccc"
                for col_idx in range(tabela_heatmap_baixa.shape[1]):
                    ax_heatmap3.add_patch(
                        plt.Rectangle((col_idx, y_idx), 1, 1, facecolor=cor_destaque,
                                      edgecolor='gray', linewidth=2, zorder=1)
                    )
        ax_heatmap3.set_ylabel(''); ax_heatmap3.set_xlabel('')
        ax_heatmap3.set_xticklabels(ax_heatmap3.get_xticklabels(), fontsize=14)
        ax_heatmap3.set_yticklabels(ax_heatmap3.get_yticklabels(), fontsize=14, rotation=0)
    else:
        pass

    # ~~~~~~~~~~~~~~~~~~~~~ Subplot 6 - Heatmap TOTAL do colaborador (dias) ~~~~~~~~~~~~~~~~~~~~~
    ax_heatmap2 = fig.add_subplot(gs_heatmaps[1])

    # Fonte diária de presença
    df_chapa_data = df_datas[df_datas['Cadastro'].astype(str) == str(chapa_input)][['Data']].copy()
    df_chapa_data['Data'] = pd.to_datetime(df_chapa_data['Data'], errors='coerce')
    df_chapa_data = df_chapa_data.dropna(subset=['Data'])

    if df_chapa_data.empty:
        ax_heatmap2.axis('off')
    else:
        # Dia da semana em PT (mantém PT se já estiver)
        _dias_pt = {
            'Monday':'Segunda','Tuesday':'Terça','Wednesday':'Quarta',
            'Thursday':'Quinta','Friday':'Sexta','Saturday':'Sábado','Sunday':'Domingo'
        }
        nomes_en = df_chapa_data['Data'].dt.day_name()
        df_chapa_data['Dia Semana'] = nomes_en.map(_dias_pt).fillna(nomes_en)

        # Contagem por dia da semana, na ordem desejada
        counts = (df_chapa_data['Dia Semana']
                .value_counts()
                .reindex(ordenacao_dias, fill_value=0)
                .astype(int))

        # 1 linha (Total) x 7 colunas (dias)
        heatmap_data = pd.DataFrame(
            [counts.values.tolist()],
            columns=ordenacao_dias,
            index=['Total']
        )

        cmap_personalizado = ListedColormap(["#ffb3b3", "#f4cccc", "#e3dacd", "#dae5d1", "#d9ead3"])
        sns.heatmap(
            heatmap_data,
            annot=True, fmt='g',
            cmap=cmap_personalizado,
            linewidths=1, linecolor='gray',
            annot_kws={'color': 'black', 'size': 17},
            cbar=False, ax=ax_heatmap2
        )

        # Ticks centralizados nas células
        ax_heatmap2.set_xticks(np.arange(heatmap_data.shape[1]) + 0.5)
        ax_heatmap2.set_xticklabels(list(heatmap_data.columns), fontsize=20, rotation=0)

        ax_heatmap2.set_yticks(np.arange(heatmap_data.shape[0]) + 0.5)
        ax_heatmap2.set_yticklabels(list(heatmap_data.index), fontsize=20, rotation=0)

        ax_heatmap2.set_xlabel('')
        ax_heatmap2.set_ylabel('')
        ax_heatmap2.tick_params(axis='y', left=True)

    # Layout final
    plt.subplots_adjust(wspace=1.4)
    fig.tight_layout()
    images_base64 = fig_to_base64(fig)

    # Gera binário
    png_bytes = fig_to_bytes(fig, fmt="png")
    pdf_bytes = fig_to_bytes(fig, fmt="pdf")

    # Converte em strings base64 (isso o Dash aceita)
    png_b64 = "data:image/png;base64," + base64.b64encode(png_bytes).decode()
    pdf_b64 = "data:application/pdf;base64," + base64.b64encode(pdf_bytes).decode()

    conteudo = html.Div([
        html.Button(" ↩ ", id='botao-voltar', n_clicks=0, style={
            'color': 'Black', 'fontFamily': 'VW Text', 'fontWeight': 'bold',
            'display': 'block', 'marginTop': '10px', 'marginLeft': '10px', 'marginRight': 'auto',
            'padding': '5px 10px', 'fontSize': '20px', 'backgroundColor': '#e0e0e0', 'border': 'none',
            'borderRadius': '5px', 'cursor': 'pointer', 'boxShadow': '0px 2px 6px rgba(0, 0, 0, 0.25)'}),

        # Botões de download
        html.Div([
            html.A("Baixar PNG", href=png_b64, download=f"relatorio_{chapa_input}.png",
                style={"marginRight": "20px", "fontSize": "16px",
                        "padding": "8px 14px", "backgroundColor": "#e0e0e0",
                        "color": "Black", "fontFamily": "VW Text", "borderRadius": "6px",
                        "textDecoration": "none",}),
            html.A("Baixar PDF", href=pdf_b64, download=f"relatorio_{chapa_input}.pdf",
                style={"fontSize": "16px", "padding": "8px 14px",
                        "backgroundColor": "#ff0000", "color": "white",
                        "fontFamily": "VW Text", "borderRadius": "6px",
                        "textDecoration": "none",}),
        ], style={"textAlign": "center", "marginTop": "10px"}),

        # Imagem inline
        html.Img(src=images_base64, style={
            'width': '100%', 'maxWidth': '1400px', 'marginTop': '20px', 'background': 'white',
            'display': 'block', 'marginLeft': 'auto', 'marginRight': 'auto'
        })
    ])

    # >>> devolvemos o conteúdo individual e escondemos a página geral
    return conteudo, {'display': 'block'}, {'display': 'none'}, no_update


'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Áreas ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
@app.callback(
    [
        # Subárea
        Output('dropdown-subarea-area', 'options'),
        Output('dropdown-subarea-area', 'value'),

        # Slider período
        Output('periodo-slider-area', 'min'),
        Output('periodo-slider-area', 'max'),
        Output('periodo-slider-area', 'value'),

        # Store do período + texto
        Output('periodo-indices-area', 'data'),
        Output('periodo-texto-area', 'children'),

        # Gráficos
        Output('grafico-area-heatmap-men', 'figure'),
        Output('grafico-area-heatmap-exec', 'figure'),
    ],
    [
        # Inputs
        Input('area-atual-area', 'data'),
        Input('dropdown-subarea-area', 'value'),
        Input('cenario-toggle-area', 'value'),
        Input('desligados-toggle-area', 'value'),
        Input('viajam-toggle-area', 'value'),
        Input('periodo-slider-area', 'value'),
        Input('url', 'search'),
        Input('areas_vis_state', 'data'),
    ]
)
def callback_unificado(area_base, subarea_sel, cenario, desligados_area, viajam_area, slider_val_state, search, areas_vis_state):
    qs = parse_qs((search or '').lstrip('?'))
    modo = (qs.get('modo', ['geral'])[0] or 'geral').lower()
    is_almoco = (modo == 'almoco')
    
    ctx = callback_context
    trig = (ctx.triggered_id or 'initial') if ctx else 'initial'

    # --- 1) limites baseados no cenário ---
    if   cenario == 'c1':
        min_i, max_i = LT_C1_MIN, LT_C1_MAX
    elif cenario == 'c2':
        min_i, max_i = LT_C2_MIN, LT_C2_MAX
    else:  # 'all'
        # Em "Tudo", não extrapolar até "hoje". O máximo deve ser o último mês
        # efetivamente disponível na base (LT_FIM_ABS já é calculado a partir do dado).
        min_i = 0
        try:
            max_i = lt_idx_of(LT_FIM_ABS.year, LT_FIM_ABS.month)
        except Exception:
            max_i = len(LT_MONTHS) - 1

    # valor atual do slider, também “clampado”
    if slider_val_state:
        v0, v1 = slider_val_state
    else:
        v0, v1 = min_i, max_i

    v0 = min(max(v0, min_i), max_i)
    v1 = min(max(v1, min_i), max_i)
    
    if v0 > v1:
        v0, v1 = min_i, max_i

    slider_min_out, slider_max_out, slider_val_out = min_i, max_i, [v0, v1]

    # --- 3) período calculado corretamente (AGORA com texto!) ---
    # inicializa variáveis para evitar erro de escopo
    i0 = i1 = None

    if is_almoco:
        # 🔹 Calcula período real com base no df_almoco
        if 'df_almoco' in globals() and isinstance(df_almoco, pd.DataFrame) and not df_almoco.empty:
            datas_validas = pd.to_datetime(df_almoco['Data'], errors='coerce').dropna()
            if not datas_validas.empty:
                dt_ini = datas_validas.min()
                dt_fim = datas_validas.max()
            else:
                dt_ini = datas_validas.min()
                dt_fim = datas_validas.max()
        else:
            dt_ini = datas_validas.min()
            dt_fim = datas_validas.max()
    else:
        # 🔹 Mantém lógica normal (usando o slider)
        i0, i1 = slider_val_out
        dt_ini, dt_fim = periodo_limites([i0, i1])

    # --- Criação do texto do período ---
    m0, y0 = dt_ini.month_name('pt_BR').capitalize(), dt_ini.year
    m1, y1 = dt_fim.month_name('pt_BR').capitalize(), dt_fim.year
    periodo_txt_out = f""
    periodo = f"{m0}/{y0} a {m1}/{y1}"
    periodo_data_out = [i0, i1] if not is_almoco else None

    # -------------------- 2) SUBÁREA (options + manter seleção) -------------
    sub_opts_out = []
    sub_val_out = subarea_sel  # default: manter
    if not area_base:
        sub_opts_out, sub_val_out = [], None
    else:
        try:
            heranca = her[her['Área'] == area_base]['Heranca'].unique()
            subareas_df = her[
                (her['Área'].str.startswith(area_base)) &
                (her['Área'] != area_base) &
                (her['Heranca'].isin(heranca))
            ]
            if subareas_df.empty:
                sub_opts_out = []
            else:
                sub_opts_out = [
                    {'label': f"{row['Área']} {row.get('Departamento','')}", 'value': row['Área']}
                    for _, row in subareas_df.iterrows()
                ]
        except Exception:
            sub_opts_out = []

        # manter seleção se ainda válida
        if sub_val_out and not is_valid(sub_val_out, sub_opts_out):
            sub_val_out = None
        # se a área mudou, pode limpar a subárea inválida
        if trig == 'area-atual-area' and (not is_valid(sub_val_out, sub_opts_out)):
            sub_val_out = None

    # --- 3) DF FULL (recalcular só quando precisa) ---
    mudou_area_ou_sub = trig in ('area-atual-area', 'dropdown-subarea-area')
    mudou_periodo     = trig in ('periodo-slider-area', 'cenario-toggle-area')
    mudou_filtros     = trig in ('desligados-toggle-area', 'viajam-toggle-area')

    df_key = cache_key(area_base, sub_val_out, desligados_area, viajam_area, i0, i1)

    if not hasattr(callback_unificado, "_df_cache"):
        callback_unificado._df_cache = {}
    cache_map = callback_unificado._df_cache

    precisa_df = bool(area_base) and (mudou_area_ou_sub or mudou_periodo or mudou_filtros or (trig == 'initial') or (df_key not in cache_map))

    if precisa_df:
        if is_almoco:
            df_full = build_dataframe_almoco_area(area_base, sub_val_out, her, df_almoco, df_ativos)
        else:
            df_full = build_dataframe_diario_area(area_base, sub_val_out, her, df_datas, df_ativos)

        # filtros opcionais (desligados/viajam) – mantenha como já estava, se fizer sentido para almoço
        if (desligados_area == 'right' and not df_full.empty and
            'Cadastro' in df_full.columns and 'Desligado' in getattr(globals().get('df_semana_merged', pd.DataFrame()), 'columns', [])):
            afast = df_semana_merged[['Cadastro','Desligado']].drop_duplicates()
            afast['Cadastro'] = afast['Cadastro'].astype(str)
            df_full = df_full.merge(afast, on='Cadastro', how='left')
            df_full = df_full[df_full['Desligado'] != 'Sim'].drop(columns=['Desligado'])

        if (viajam_area == 'right' and not df_full.empty and 'Cadastro' in df_full.columns):
            df_sm = globals().get('df_semana_merged', pd.DataFrame())
            viaja_col = None
            for c in ['Viaja','Viajando','Em Viagem','Viajam']:
                if c in getattr(df_sm, 'columns', []): viaja_col = c; break
            if viaja_col is not None:
                vj = df_sm[['Cadastro', viaja_col]].drop_duplicates()
                vj['Cadastro'] = vj['Cadastro'].astype(str)
                df_full = df_full.merge(vj, on='Cadastro', how='left')
                df_full = df_full[df_full[viaja_col] != 'Sim'].drop(columns=[viaja_col])

        cache_map[df_key] = df_full
    else:
        df_full = cache_map.get(df_key, pd.DataFrame())

    
    # --- TÍTULO amigável por modo ---
    periodo = f"{m0}/{y0} a {m1}/{y1}"
    titulo_men = (
        f"Almoços e Saídas — Mensalistas × Executivos ({periodo})"
        if is_almoco else
        f"Presença Percentual dos Mensalistas com os seus Executivos ({periodo})"
    )
    titulo_exec = (
        f"Almoços e Saídas — Executivos × Executivos ({periodo})"
        if is_almoco else
        f"Presença Percentual entre Executivos ({periodo})"
    )

    # --- Evita piscar o gráfico sem design ---
    fig_men_out = no_update
    fig_exec_out = no_update

    # --- Decide modo de visualização (Heatmap ou Teia/Network) ---
    vis_mode = areas_vis_state or "heatmap"
    if isinstance(vis_mode, str):
        vis_mode = vis_mode.lower()
    else:
        vis_mode = "heatmap"

    if not df_full.empty:


        if vis_mode == "network":
            # Modo TEIA – Pentágono (Mensalistas×Executivos) + Rede (Exec×Exec)
            if not hasattr(callback_unificado, "_fig_cache"):
                callback_unificado._fig_cache = {}
            fig_cache = callback_unificado._fig_cache
            fig_key = (df_key, "network_radar_v3")

            if fig_key in fig_cache:
                fig_men_full, fig_exec_full = fig_cache[fig_key]
            else:
                df_net = df_full.copy()

                # Sanitiza objetos para evitar erros de hashing em pandas/plotly
                try:
                    df_net = _sanitize_object_columns(df_net)
                except Exception:
                    pass

                # Normaliza Cargo (aceita códigos 'E'/'M' e texto como 'Executivo'/'Mensalista')
                if "Cargo" in df_net.columns:
                    try:
                        c = df_net["Cargo"].astype(str).str.strip()
                        cu = c.str.upper()
                        df_net["Cargo"] = np.where(
                            cu.str.startswith("E") | cu.str.contains("EXEC"),
                            "E",
                            np.where(
                                cu.str.startswith("M") | cu.str.contains("MENS"),
                                "M",
                                c
                            )
                        )
                    except Exception:
                        df_net["Cargo"] = df_net["Cargo"].astype(str)

                # Colunas mínimas esperadas pelos gráficos
                if "Cadastro" in df_net.columns:
                    df_net["Cadastro"] = df_net["Cadastro"].astype(str)
                if "Nome" not in df_net.columns and "Cadastro" in df_net.columns:
                    df_net["Nome"] = df_net["Cadastro"]
                if "Data" in df_net.columns:
                    df_net["Data"] = pd.to_datetime(df_net["Data"], errors="coerce")
                elif "Semana" in df_net.columns:
                    df_net["Data"] = pd.to_datetime(df_net["Semana"], errors="coerce")
                if "Contagem Presença" in df_net.columns:
                    df_net["Contagem Presença"] = pd.to_numeric(df_net["Contagem Presença"], errors="coerce").fillna(0.0)
                else:
                    df_net["Contagem Presença"] = 1.0

                # 1º gráfico: Radar por executivo (eixos = mensalistas comparáveis pela Área — mesma lógica do heatmap)
                try:
                    titulo_men_teia = f"Pentágonos de relacionamento (Executivos × Mensalistas) ({periodo})"
                    fig_men_full = grafico_pentagonos_rel_exec_multiplos(
                        df_net,
                        titulo=titulo_men_teia,
                        max_exec=12,
                        max_eixos=None,
                        range_mode="global",
                        validar_mesma_planta=True,
                        df_hierarquia=globals().get("df_ativos", None),
                        is_almoco=is_almoco,
                        strict=True,
                    )
                except Exception as e:
                    fig_men_full = go.Figure()
                    fig_men_full.update_layout(
                        title={'text': f"{titulo_men} — erro ao gerar TEIA", 'font': {'family': 'VW Text', 'size': 22}},
                        height=420,
                        font={'family': 'VW Text'}
                    )
                    fig_men_full.add_annotation(
                        text=str(e)[:200],
                        x=0.5, y=0.5, xref='paper', yref='paper',
                        showarrow=False
                    )

                # 2º gráfico: Radar Exec×Exec (eixos = executivos comparáveis pela Área)
                try:
                    titulo_exec_teia = f"Pentágonos de relacionamento (Executivos × Executivos) ({periodo})"
                    fig_exec_full = grafico_pentagonos_exec_exec_multiplos(
                        df_net,
                        titulo=titulo_exec_teia,
                        max_exec=12,
                        max_eixos=None,
                        range_mode="global",
                        validar_mesma_planta=True,
                        df_hierarquia=globals().get("df_ativos", None),
                        is_almoco=is_almoco,
                        strict=True,
                    )
                except Exception as e:
                    fig_exec_full = go.Figure()
                    fig_exec_full.update_layout(
                        title={'text': f"{titulo_exec} — erro ao gerar TEIA", 'font': {'family': 'VW Text', 'size': 22}},
                        height=420,
                        font={'family': 'VW Text'}
                    )
                    fig_exec_full.add_annotation(
                        text=str(e)[:200],
                        x=0.5, y=0.5, xref='paper', yref='paper',
                        showarrow=False
                    )

                # Cache (para alternar Heatmap⇄Teia sem recomputar)
                fig_cache[fig_key] = (fig_men_full, fig_exec_full)
        else:
            # Modo padrão – HEATMAP
            if not hasattr(callback_unificado, "_fig_cache"):
                callback_unificado._fig_cache = {}
            fig_cache = callback_unificado._fig_cache
            fig_key = (df_key, "heatmap")

            if fig_key in fig_cache:
                fig_men_full, fig_exec_full = fig_cache[fig_key]
            else:
                fig_men_full = aplicar_tema_pb(
                    grafico_heatmap_encontros_men(
                        df_full, titulo_men, validar_mesma_planta=True
                    ),
                    so_y_boxplot=False,
                    grade=True,
                )
                fig_exec_full = aplicar_tema_pb(
                    grafico_heatmap_encontros_exec(
                        df_full, titulo_exec, validar_mesma_planta=True
                    ),
                    so_y_boxplot=False,
                    grade=False,
                    clean=True,
                )
                fig_cache[fig_key] = (fig_men_full, fig_exec_full)

        # Preserva zoom/pan sempre que possível
        fig_men_full.update_layout(uirevision="area-vis-men")
        fig_exec_full.update_layout(uirevision="area-vis-exec")

        fig_men_out = fig_men_full
        fig_exec_out = fig_exec_full

    # retorna tudo normalmente
    return (
        sub_opts_out, sub_val_out,
        slider_min_out, slider_max_out, slider_val_out,
        periodo_data_out, periodo_txt_out,
        fig_men_out, fig_exec_out
    )


@app.callback(
    Output('conteudo-analitico-area','children'),
    Output('conteudo-analitico-area','style'),
    Output('pagina-geral','style', allow_duplicate=True),
    Output('url','pathname', allow_duplicate=True),
    Input('url','pathname'),
    Input('botao-voltar-area','n_clicks'),
    Input('url','search')
)
def exibir_conteudo_area(pathname, n_clicks, search):
    # Ação de voltar
    if n_clicks and n_clicks > 0:
        return None, estilo_oculto, estilo_exibir, "/"

    # Rota inválida
    if not pathname or "/area/" not in pathname:
        return dash.no_update, estilo_oculto, dash.no_update, dash.no_update

    # Área/subárea a partir do path
    area, subarea = get_area(pathname)
    if not area:
        return None, estilo_oculto, estilo_oculto, dash.no_update

    # --- detectar modo pelo querystring (?modo=geral|almoco) ---
    qs = parse_qs((search or '').lstrip('?'))
    modo = (qs.get('modo', ['geral'])[0] or 'geral').lower()
    is_almoco = (modo == 'almoco')

    # Botão de alternância (topo direito): muda label e destino conforme o modo atual
    if is_almoco:
        texto_toggle = "Relacionamento Geral da Área ⭢"
        href_toggle = f"/area/{slug(subarea or area)}?modo=geral"
    else:
        texto_toggle = "Relacionamento do Almoço da Área ⭢"
        href_toggle = f"/area/{slug(subarea or area)}?modo=almoco"

    # ======= CONSTRÓI O RELATÓRIO (FIGURAS INICIAIS) =======
    fig_heatmap_men = None
    fig_heatmap_exec = None

    # ======= BARRA DE FILTROS (mesmo conteúdo, mas pode ficar oculta no modo almoço) =======
    filter_block_style = {
        'flex':'1','padding':'16px','boxSizing':'border-box','backgroundColor':"#e0e0e0",
        'borderRadius':'16px','margin':'10px','boxShadow':'0px 3px 6px rgba(0,0,0,0.1)',
        'display':'flex','flexWrap':'wrap','alignItems':'flex-start','gap':'24px'
    }
    if is_almoco:
        # Esconder completamente no modo almoço (mantém IDs e callbacks)
        filter_block_style['display'] = 'none'
        
    filtros_div = html.Div([
        # desligados / Viajam
        html.Div([
            html.Div([
                html.Label("Mostrar as pessoas desligadas?"),
                dcc.RadioItems(
                    id='desligados-toggle-area',
                    options=[{'label': 'Sim', 'value': 'left'}, {'label': 'Não', 'value': 'right'}],
                    value='left',
                    labelStyle={'display': 'inline-block', 'marginRight': '10px'}
                ),
            ], style={'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'}),

            html.Div([
                html.Label("Mostrar as áreas que viajam?"),
                dcc.RadioItems(
                    id='viajam-toggle-area',
                    options=[{'label': 'Sim', 'value': 'left'}, {'label': 'Não', 'value': 'right'}],
                    value='left',
                    labelStyle={'display': 'inline-block', 'marginRight': '10px'}
                ),
            ], style={'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'}),
        ], style={'display':'flex','gap':'30px','alignItems':'center'}),

        # Cenário + RangeSlider
        html.Div([
            html.Div([
                dcc.RadioItems(
                    id='cenario-toggle-area',
                    options=[
                        {'label': 'Tudo',                 'value': 'all'},
                        {'label': 'Jan/2024 a Abr/2025',  'value': 'c1'},
                        {'label': 'Mai/2025 em diante',   'value': 'c2'}
                    ],
                    value='all',
                    labelStyle={'display': 'inline-block'}
                )
            ], style={'textAlign': 'center', 'marginBottom': '6px'}),

            dcc.RangeSlider(
                id='periodo-slider-area',
                min=0, max=len(LT_MONTHS)-1,
                value=[0, len(LT_MONTHS)-1],
                marks=LT_MARKS, allowCross=False, step=1, pushable=1
            )
        ], style={'flex':'0 0 540px'}),

        # Subárea local (dentro da mesma herança)
        html.Div([
            dcc.Dropdown(
                id='dropdown-subarea-area',
                options=[],
                value=subarea,  # já entra com a subárea atual, se houver
                placeholder='Selecione uma subárea',
                clearable=True,
                style={'width': '270px', 'marginTop': '6px'}
            ),
        ], style={'marginLeft':'auto'})
    ], style=filter_block_style)

    # ======= Função local para CENTRALIZAR as duas tabelas =======
    def painel_turnover_gptw_centralizado(tbl_turn, tbl_gptw):
        card_style = {
            'backgroundColor': '#fff',
            'border': '1px solid #dcdcdc',
            'borderRadius': '6px',
            'padding': '6px',
            'boxShadow': '0 2px 2px rgba(0,0,0,0.08)',
            'flex': '0 1 270px',
            'maxWidth': '560px',
            'fontFamily': 'VW Text'
        }

        mk_title = lambda t: html.H4(t, style={'textAlign': 'center', 'margin': '0 0 24px', 'fontFamily': 'VW Text'})

        def center(child):
            return html.Div(
                child,
                style={
                    'margin': '0 auto',
                    'width': 'fit-content',
                    'maxWidth': '100%',
                    'overflowX': 'auto',
                    'fontFamily': 'VW Text'
                }
            )

        return html.Div([
            html.Div([mk_title("Headcount & Turnover"), tbl_turn], style=card_style),
            html.Div([mk_title("GPTW"),                 tbl_gptw],  style=card_style),
            # Toggle Heatmap ↔ Teia (fora do fluxo; não causa reflow)
            html.Div(
                html.Div(areas_toggle_ui(), style={'pointerEvents': 'auto'}),
                style={
                    'position': 'absolute',
                    'right': '10px',
                    'top': '10px',
                    'zIndex': 20,
                    'pointerEvents': 'auto'
                }
            ),
        ] , style={
            'position': 'relative',
            'display': 'flex',
            'justifyContent': 'center',
            'alignItems': 'flex-start',
            'gap': '24px',
            'flexWrap': 'wrap',
            'width': '100%',
            'maxWidth': '100%',
            'margin': '8px 0 16px',
            'padding': '10px',
            'boxSizing': 'border-box',
            'fontFamily': 'VW Text'
        })
    # ======= Monta as duas tabelas =======
    def _safe_build_turnover(a, s):
        chave = (s or a)
        try:
            return tabela_padrao_turnover(chave, anos=(2023, 2024, 2025), incluir_macro=True)
        except Exception:
            pass

        for fname in ('montar_tabela_turnover', 'tabela_turnover_area', 'render_tabela_turnover_area'):
            fn = globals().get(fname)
            if callable(fn):
                try:
                    return fn(a, s)
                except Exception:
                    continue

        anos = (2023, 2024, 2025)
        header = [html.Th("Área", style=PAD_STY['th'])]
        for ano in anos:
            header.append(html.Th(str(ano), style=PAD_STY['th'] | {"textAlign": "center"}))
            header.append(html.Th("E",        style=PAD_STY['th']))
            header.append(html.Th("S",        style=PAD_STY['th']))

        return html.Table([
            html.Thead(html.Tr(header)),
            html.Tbody([
                html.Tr([
                    html.Td(s or a or '—', style=PAD_STY['td_first'])
                ] + [
                    html.Td("—", style=PAD_STY['td'])
                    for _ in range(len(anos) * 3)
                ])
            ])
        ], style=PAD_STY['table'])

    def _safe_build_gptw(a, s):
        try:
            return tabela_padrao_gptw(s or a, anos=(2023, 2024, 2025), incluir_macro=True)
        except Exception:
            pass

        for fname in ('montar_tabela_gptw', 'tabela_gptw_area'):
            fn = globals().get(fname)
            if callable(fn):
                try:
                    return fn(a, s)
                except Exception:
                    pass

        try:
            return mini_tabela_gptw(s or a)
        except Exception:
            anos = (2023, 2024, 2025)
            thead = html.Thead(html.Tr([html.Th('Área', style=PAD_STY['th'])] +
                                    [html.Th(str(y), style=PAD_STY['th']) for y in anos]))
            tbody = html.Tbody([html.Tr([html.Td(s or a or '—', style=PAD_STY['td_first'])] +
                                        [html.Td('—', style=PAD_STY['td']) for _ in anos])])
            return html.Table([thead, tbody], style=PAD_STY['table'])

    tbl_turn = _safe_build_turnover(area, subarea)
    tbl_gptw = _safe_build_gptw(area, subarea)
    bloco_duplo = painel_turnover_gptw_centralizado(tbl_turn, tbl_gptw)

    # ======= CONTEÚDO DA PÁGINA =======
    conteudo = html.Div([
        # Sequenciador de loading da aba Áreas
        dcc.Store(id='area-load-step', data=0),
        dcc.Interval(id='area-load-interval', interval=500, n_intervals=0, disabled=False, max_intervals=2),

        # ===== CABEÇALHO =====
        html.Div([
            # --- Definição ocupando toda a largura, acima de tudo ---
            html.Div(
                html.Span(
                    "O que essa página mostra?\n"
                    "Entradas no refeitório e saídas da planta durante o horário de almoço, "
                    "considerando diferenças de até 1 minuto para o refeitório e 3 minutos "
                    "para as saídas da fábrica, além de estar sendo analisado apenas os últimos 3 meses.",
                    style={
                        'fontFamily': 'VW Text',
                        'fontSize': '14px',
                        'fontWeight': 'bold',
                        'padding': '6px 20px',
                        'border': '1px solid #000',
                        'borderRadius': '6px',
                        'backgroundColor': '#fff',
                        'display': 'block',
                        'textAlign': 'justify',
                        'whiteSpace': 'pre-line',
                        'width': '100%',
                        'margin': '20px 10px 0 10px',
                        'lineHeight': '1.3',
                        'boxSizing': 'border-box'
                    }
                ),
                id='definicao-site',
                style={
                    'display': 'flex',
                    'justifyContent': 'center',
                    'marginBottom': '8px'
                }
            ) if is_almoco else html.Div(),
            # --- Cabeçalho principal: esquerda (voltar) / direita (imprimir + toggle-ALMOÇO) / título ---
            html.Div([
                # Coluna da esquerda (botão Voltar)
                html.Div(
                    [
                        html.Button(
                            " ↩ ",
                            id='botao-voltar-area',
                            n_clicks=0,
                            style={
                                'display': 'inline-flex',
                                'alignItems': 'center',
                                'color': 'Black',
                                'fontFamily': 'VW Text',
                                'fontWeight': 'bold',
                                'padding': '5px 20px',
                                'fontSize': '20px',
                                'backgroundColor': '#e0e0e0',
                                'border': 'none',
                                'borderRadius': '5px',
                                'cursor': 'pointer',
                                'boxShadow': '0px 2px 6px rgba(0,0,0,0.25)',
                                'textAlign': 'center',
                                'textDecoration': 'none',
                                'transition': '0.2s ease',
                            }
                        ),
                    ],
                    style={
                        'display': 'flex',
                        'flexDirection': 'row',
                        'alignItems': 'center',
                        'justifyContent': 'flex-start',
                        'gap': '12px',
                                                'flexWrap': 'nowrap',
                    }
                ),

                # Coluna da direita (Imprimir + alternância Almoço/Geral já existente)
                html.Div([
                    html.Button(
                        "🖨️ Imprimir",
                        id='btn-print-area',
                        n_clicks=0,
                        style={
                            'backgroundColor': 'transparent',
                            'border': 'none',
                            'color': 'Black',
                            'fontFamily': 'VW Text',
                            'fontWeight': 'bold',
                            'fontSize': '16px',
                            'cursor': 'pointer',
                            'padding': '10px 6px',
                            'display': 'inline-flex',
                            'alignItems': 'center',
                            'justifyContent': 'center'
                        }
                    ),
                    dcc.Link(
                        texto_toggle,
                        href=href_toggle,
                        style={
                            'display': 'inline-block',
                            'color': 'Black',
                            'fontFamily': 'VW Text',
                            'fontWeight': 'bold',
                            'padding': '10px',
                            'fontSize': '14px',
                            'backgroundColor': '#e0e0e0',
                            'border': 'none',
                            'borderRadius': '5px',
                            'cursor': 'pointer',
                            'boxShadow': '0px 2px 6px rgba(0, 0, 0, 0.25)',
                            'width': 'auto',
                            'textAlign': 'center',
                            'textDecoration': 'none',
                            'transition': '0.2s ease'
                        }
                    )
                ], style={
                    'marginLeft': 'auto',
                    'display': 'flex',
                    'alignItems': 'center',
                    'justifyContent': 'flex-end',
                    'gap': '16px',
                    'flexWrap': 'nowrap'
                }),

                # Título central absoluto
                html.H2(
                    f"Análise Específica — {subarea or area}",
                    style={
                        'position': 'absolute',
                        'left': '50%',
                        'transform': 'translateX(-50%)',
                        'margin': '0',
                        'fontFamily': 'VW Text',
                        'fontWeight': 'bold',
                        'fontSize': '22px',
                        'whiteSpace': 'nowrap',
                        'overflow': 'hidden',
                        'textOverflow': 'ellipsis',
                        'textAlign': 'center',
                        'zIndex': 1,
                    }
                )
            ], style={
                'display': 'flex',
                'alignItems': 'center',
                'justifyContent': 'space-between',
                'flexWrap': 'wrap',
                'gap': '16px',
                'margin': '4px 0 0',
                'padding': '0 10px',
                'width': '100%',
                'boxSizing': 'border-box',
                'position': 'relative'
            })
        ]),

        # Div escondida para o callback de impressão
        html.Div(id='dummy-print-area', style={'display': 'none'}),

        # ===================== FILTROS E GRÁFICOS =====================
        filtros_div,

        html.Div(
            id='periodo-texto-area',
            style={
                'textAlign': 'center',
                'marginTop': '4px',
                'marginBottom': '10px'
            }
        ),

        # ===================== BLOCO: Tabelas centralizadas =====================
        bloco_duplo,

        dcc.Loading(
            id='loading-area-men',
            type='default',
            children=dcc.Graph(figure=fig_heatmap_men, id='grafico-area-heatmap-men', style={'height':'auto'})
        ),
        dcc.Loading(
            id='loading-area-exec',
            type='default',
            children=dcc.Graph(figure=fig_heatmap_exec, id='grafico-area-heatmap-exec', style={'height':'auto'})
        ),
# ===================== STORES =====================
        dcc.Store(id='periodo-indices-area', data=[LT_C1_MIN, LT_C1_MAX]),
        dcc.Store(id='area-atual-area', data=area)

    ],
    style={
        'padding': '0px',
        'backgroundColor': 'white'
    })

    return conteudo, {'display':'block'}, {'display':'none'}, dash.no_update


app.clientside_callback(
    """
    function(n_clicks, current) {
        current = current || "heatmap";

        // alterna estado baseado no estado atual (não depende de latência do servidor)
        var next_state = current;
        if (n_clicks && n_clicks > 0) {
            next_state = (current === "heatmap") ? "network" : "heatmap";
        }

        var btn_base = {
            position: "relative",
            width: "92px",
            height: "42px",
            borderRadius: "999px",
            border: "1px solid rgba(0,0,0,.14)",
            background: "linear-gradient(#f7f7f7,#e7e7e7)",
            boxShadow: "0 1px 2px rgba(0,0,0,.12) inset, 0 1px 2px rgba(0,0,0,.05)",
            cursor: "pointer",
            transition: "all .18s ease-in-out",
            padding: "0",
            fontFamily: "VW Text"
        };
        var btn_on = {
            background: "linear-gradient(#181818,#2a2a2a)",
            border: "1px solid rgba(255,255,255,.12)"
        };

        var knob_base = {
            position: "absolute",
            top: "3px",
            left: "3px",
            width: "36px",
            height: "36px",
            borderRadius: "50%",
            background: "#ffffff",
            boxShadow: "0 1px 2px rgba(0,0,0,.28), 0 0 0 1px rgba(0,0,0,.06) inset",
            transform: "translateX(0)",
            transition: "transform .18s ease-in-out, background .18s ease"
        };
        var knob_on = {
            transform: "translateX(50px)",
            background: "#1e1e1e",
            boxShadow: "0 1px 2px rgba(0,0,0,.28), 0 0 0 1px rgba(255,255,255,.08) inset"
        };

        var ico_base = {
            position: "absolute",
            top: "50%",
            transform: "translateY(-50%)",
            fontSize: "18px",
            lineHeight: "18px",
            fontFamily: "VW Text",
            userSelect: "none"
        };
        var ico_heat_color = { color: "#0057b8", left: "14px" };
        var ico_net_color  = { color: "#00a37a", right: "14px" };
        var ico_dim = { opacity: 0.28 };
        var ico_on_color = { color: "#f3f3f3" };

        var knob_ico_base = {
            position: "absolute",
            inset: "0",
            display: "flex",
            alignItems: "center",
            justifyContent: "center",
            fontSize: "20px",
            fontFamily: "VW Text",
            transform: "translateX(-25px)"
        };

        var btn_style = Object.assign({}, btn_base);
        var knob_style = Object.assign({}, knob_base);
        var heat_style = Object.assign({}, ico_base, ico_heat_color);
        var net_style  = Object.assign({}, ico_base, ico_net_color);
        var knob_ico_style = Object.assign({}, knob_ico_base);
        var knob_ico_txt = "▦";

        if (next_state === "network") {
            Object.assign(btn_style, btn_on);
            Object.assign(knob_style, knob_on);
            Object.assign(heat_style, ico_dim);
            Object.assign(net_style, ico_on_color);
            knob_ico_txt = "🕸️";
            Object.assign(knob_ico_style, { color: "#f3f3f3", transform: "translateX(25px)" });
        } else {
            Object.assign(net_style, ico_dim);
            Object.assign(knob_ico_style, { color: "#0057b8", transform: "translateX(-25px)" });
        }

        return [next_state, btn_style, knob_style, heat_style, net_style, knob_ico_txt, knob_ico_style];
    }
    """,
    Output('areas_vis_state', 'data'),
    Output('areas_vis_btn', 'style'),
    Output('knob', 'style'),
    Output('ico_heat', 'style'),
    Output('ico_net', 'style'),
    Output('knob_ico', 'children'),
    Output('knob_ico', 'style'),
    Input('areas_vis_btn', 'n_clicks'),
    State('areas_vis_state', 'data'),
)


app.clientside_callback(
    """
    function(n_clicks) {
        if (!n_clicks) {
            return "";
        }
        try {
            window.print();
        } catch (e) {
            if (console && console.error) {
                console.error("Erro ao chamar window.print()", e);
            }
        }
        return "";
    }
    """,
    Output('dummy-print-area', 'children'),
    Input('btn-print-area', 'n_clicks'),
)


'''~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Relacionamento ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'''
@app.callback(
    Output('tabela-gptw-rel', 'children'),
    Input('dropdown-area-gptw', 'value')
)
def renderizar_tabela_gptw(areas):
    # quando nada selecionado, retorna vazio (Div)
    return gptw_tabela_empilhada(areas)


@app.callback(
    Output("dropdown_gptw", "options"),
    Output("dropdown_gptw", "value"),
    Input("dropdown_gptw", "value"),
    prevent_initial_call=True
)
def atualizar_dropdown_gptw(valor_atual):
    df = df_gptw.copy()
    todas_areas = sorted(df["Área"].dropna().unique().tolist())

    # normaliza valor atual
    if not valor_atual:
        valor_atual = []
    elif isinstance(valor_atual, str):
        valor_atual = [valor_atual]
    valor_atual = [v.strip() for v in valor_atual]

    # mapeamento área ⭢ subáreas, se existir coluna
    if "Subarea" in df.columns:
        mapa = (
            df[["Área", "Subarea"]]
            .dropna()
            .groupby("Área")["Subarea"]
            .apply(set)
            .to_dict()
        )
    else:
        # fallback: considera prefixo (ex: B-I ⭢ B-IA, B-IE, ...)
        mapa = {a: {b for b in todas_areas if b.startswith(a) and b != a}
                for a in todas_areas}

    # subáreas a remover
    ocultar = set()
    for area_sel in valor_atual:
        ocultar.update(mapa.get(area_sel, set()))

    # remove as subáreas da lista de opções
    visiveis = [a for a in todas_areas if a not in ocultar]
    opcoes = [{"label": a, "value": a} for a in visiveis]

    # também limpa do value qualquer item que tenha sido ocultado
    valor_filtrado = [v for v in valor_atual if v in visiveis]

    return opcoes, valor_filtrado


@app.callback(
    Output('rel-tags', 'data', allow_duplicate=True),
    Input('tag-tabela', 'active_cell'),
    State('tag-tabela', 'data'),
    State('rel-tags', 'data'),
    prevent_initial_call=True
)
def _delete_row(ac, rows, tags):
    if not ac or ac.get('column_id') != 'del':
        raise PreventUpdate
    idx = ac.get('row')
    if idx is None or idx < 0 or idx >= len(rows or []):
        raise PreventUpdate

    rid = rows[idx].get('id')
    novos = [t for t in (tags or []) if t.get('id') != rid]
    return novos


@app.callback(
    Output('tag-tabela', 'data'),
    Input('rel-tags', 'data')
)
def _sync_table_from_store(tags):
    rows = []
    for t in (tags or []):
        nome   = t.get('nome', '')
        data   = t.get('data', '')
        corhex = t.get('cor', '#3221cc')

        # 🧹 Garante que a cor seja válida
        if not isinstance(corhex, str) or not corhex.startswith('#') or len(corhex) not in (4, 7):
            corhex = '#3221cc'  # cor padrão de fallback

        # 🗓️ converte data para DD/MM/YYYY
        try:
            data_br = pd.to_datetime(data).strftime('%d/%m/%Y')
        except Exception:
            data_br = ''

        # ✅ HTML seguro
        swatch = (
            f"<div style='display:inline-block;width:12px;height:12px;"
            f"border:1px solid #888;margin-left:6px;"
            f"background-color:{corhex};vertical-align:middle;'></div>"
        )

        rows.append({
            'id': t.get('id'),
            'nome_md': nome,
            'data_br': data_br,
            'cor': swatch,
            'del': "🗑️"
        })

    return rows


@app.callback(
    Output('rel-tags', 'data', allow_duplicate=True),
    Output('tag-nome', 'value'),
    Output('tag-data', 'date'),
    Output('tag-cor',  'value'),
    Input('tag-add', 'n_clicks'),
    State('tag-nome', 'value'),
    State('tag-data', 'date'),
    State('tag-cor',  'value'),
    State('rel-tags', 'data'),
    prevent_initial_call=True
)
def _add_tag(n, nome, data_iso, cor, tags):
    tags = list(tags or [])
    if not data_iso:
        raise PreventUpdate

    novo = {
        'id':   str(time.time()),      # <— id único
        'nome': (nome or 'Evento'),
        'data': str(data_iso),         # 'YYYY-MM-DD'
        'cor':  (_fmt_hex(cor) or '#3221cc')
    }
    tags.append(novo)
    return tags, "", None, "#3221cc"

@app.callback(
    Output('rel-tags', 'data', allow_duplicate=True),
    Input('tag-tabela', 'data'),
    State('rel-tags', 'data'),
    prevent_initial_call=True
)
def _sync_store_from_table(rows, tags):
    # dicionário id -> tag (cópia mutável)
    by_id = {t.get("id"): dict(t) for t in (tags or []) if t and t.get("id")}

    def _extract_hex_from_cell(cell):
        """Extrai #RGB/#RRGGBB do HTML do swatch ou normaliza strings tipo 'abc', 'rgb(...)'."""
        if isinstance(cell, str):
            m = re.search(r"background-color\s*:\s*(#[0-9a-fA-F]{3,6})", cell)
            if m:
                return _fmt_hex(m.group(1))
            return _fmt_hex(cell)
        return None

    for r in (rows or []):
        rid = r.get("id")
        if not rid or rid not in by_id:
            continue

        # Data (DD/MM/YYYY -> YYYY-MM-DD)
        data_iso = None
        if r.get("data_br"):
            try:
                data_iso = pd.to_datetime(r["data_br"], dayfirst=True).strftime("%Y-%m-%d")
            except Exception:
                data_iso = None
        if data_iso is not None:
            by_id[rid]["data"] = data_iso

        # Cor: extrai do HTML da célula ou normaliza, SEM nunca gravar HTML no store
        hex_new = _extract_hex_from_cell(r.get("cor"))
        if hex_new:
            by_id[rid]["cor"] = hex_new

        # Nome (limpa formatações)
        nome_md = r.get("nome_md") or ""
        nome_limpo = nome_md.replace("**", "").strip()
        if nome_limpo:
            by_id[rid]["nome"] = nome_limpo

    # devolve TODAS as tags
    return list(by_id.values())


@app.callback(
    Output('dropdown-pessoas-rel', 'options'),
    Input('url', 'pathname')
)
def carregar_opcoes_pessoas(_):
    df_p = (df_semana_merged[['Cadastro','Nome']]
            .dropna()
            .drop_duplicates())
    df_p['Cadastro'] = df_p['Cadastro'].astype(str)
    df_p = df_p.sort_values('Nome')

    # Exclusivo de pessoas, com rótulo "CHAPA - NOME"
    return [
        {'label': f"{row['Cadastro']} - {row['Nome']}",
         'value': f"PESSOA::{row['Cadastro']}"}
        for _, row in df_p.iterrows()
    ]


@app.callback(
    Output('cores-pessoas-container', 'children'),
    Input('dropdown-pessoas-rel', 'value')
)
def gerar_inputs_cor(selecionados):
    if not selecionados:
        return []
    pessoas_ids = [v for v in selecionados if isinstance(v, str) and v.startswith("PESSOA::")]
    if not pessoas_ids:
        return []

    cad_to_nome = (df_semana_merged[['Cadastro','Nome']]
                   .drop_duplicates()
                   .assign(Cadastro=lambda d: d['Cadastro'].astype(str))
                   .set_index('Cadastro')['Nome']
                   .to_dict())

    def label_pessoa(pid):
        cad = pid.split("PESSOA::",1)[1]
        nome = cad_to_nome.get(cad, cad)
        return f"{nome} ({cad})"

    return [
        html.Div([
            html.Label(label_pessoa(pid)),
            dcc.Input(
                type='color',
                id={'type':'cor-pessoa','index':pid},
                value=f"#{np.random.randint(0, 0xFFFFFF):06x}",
                debounce=True,
                style={'marginLeft':'10px','width':'60px'}
            )
        ], style={'display':'flex','alignItems':'center','justifyContent':'space-between', 'gap':'8px'})
        for pid in pessoas_ids
    ]


@app.callback(
    Output('grafico-relacionamento-linhas', 'figure'),
    Output('grafico-mini-media-movel', 'figure'),
    Input('dropdown-pessoas-rel', 'value'),
    Input({'type':'cor-pessoa','index':dash.ALL}, 'value'),
    Input('rel-tags', 'data'),
    Input('modo-rel', 'data'),
)
def atualizar_graficos(selecionados, cores, tags, modo_rel):
    def fig_msg(msg):
        f = go.Figure()
        f.update_layout(template="plotly_white", title=msg)
        return f

    # --- tags válidas ---
    def sanitize_tags(tag_list):
        fixed = []
        for t in (tag_list or []):
            cor = _fmt_hex(t.get("cor")) or "#3221cc"
            fixed.append({
                "id":   t.get("id"),
                "nome": t.get("nome", "Evento"),
                "data": t.get("data"),
                "cor":  cor,
            })
        return fixed
    tags = sanitize_tags(tags)

    # --- seleção ---
    pessoas_ids = [v for v in (selecionados or []) if isinstance(v, str) and v.startswith("PESSOA::")]
    cad_list = [pid.split("PESSOA::",1)[1] for pid in pessoas_ids]
    if not cad_list:
        return fig_msg("Selecione pelo menos 1 pessoa"), fig_msg("—")

    # nomes
    cad_to_nome = (df_semana_merged[['Cadastro','Nome']]
                   .drop_duplicates()
                   .assign(Cadastro=lambda d: d['Cadastro'].astype(str))
                   .set_index('Cadastro')['Nome']
                   .to_dict())

    # cores
    cores_map_by_pid = {}
    if isinstance(cores, list):
        for i, pid in enumerate(pessoas_ids):
            c = cores[i] if i < len(cores) else None
            if isinstance(c, str) and c.strip():
                cores_map_by_pid[pid] = c.strip()

    # ==================== 1º GRÁFICO: SEMPRE GERAL ====================
    if 'df_datas' not in globals() or not isinstance(globals()['df_datas'], pd.DataFrame):
        return fig_msg("Base df_datas ausente"), fig_msg("—")

    dfp = globals()['df_datas'].copy()
    if 'Data' not in dfp.columns:
        return fig_msg("df_datas sem coluna 'Data'"), fig_msg("—")

    dfp['Cadastro'] = dfp['Cadastro'].astype(str)
    dfp['Data'] = pd.to_datetime(dfp['Data'], errors='coerce').dt.normalize()
    dfp = dfp.dropna(subset=['Data'])
    dfp = dfp[dfp['Cadastro'].isin(cad_list)]
    if dfp.empty:
        return fig_msg("Sem registros de presença para as pessoas selecionadas"), fig_msg("—")

    # SemIni robusto
    dfp['SemIni'] = _semana_inicio(dfp['Data'])
    semanas = sorted(dfp['SemIni'].dropna().unique())

    # contagem por semana
    presenca_semana = {}
    for cad in cad_list:
        s = (dfp[dfp['Cadastro']==cad]
             .groupby('SemIni')['Data'].nunique()
             .reindex(semanas, fill_value=0)
             .astype(float))
        presenca_semana[cad] = s

    fig_pres = go.Figure()
    for cad in cad_list:
        pid  = f"PESSOA::{cad}"
        nome = cad_to_nome.get(cad, cad)
        cor  = cores_map_by_pid.get(pid, None)
        hover = (nome + "<br>Semana: %{x|%Y-%m-%d}<br>Dias presentes: %{y:.0f}<extra></extra>")
        fig_pres.add_trace(go.Scatter(
            x=semanas, y=presenca_semana[cad].values,
            mode='lines+markers', name=nome,
            line=(dict(width=2, **({'color': cor} if cor else {}))),
            hovertemplate=hover
        ))
    fig_pres.update_layout(
        template="plotly_white", title="Presenças semanais (dias)",
        xaxis_title="", yaxis_title="Dias presentes",
        showlegend=False, margin=dict(t=110, r=20, b=60, l=60),
        hovermode='x unified'
    )

    # ==================== 2º GRÁFICO: DEPENDE DO MODO ====================
    modo_rel = (modo_rel or 'geral').lower()

    if modo_rel == 'almoco':
        # Mini ALMOÇO
        df_almoco_local = globals().get('df_almoco', None)
        if df_almoco_local is None or getattr(df_almoco_local, 'empty', True):
            fig_mini_movel = fig_msg("Sem base de almoço (df_almoco)")
        else:
            dfal = build_dataframe_almoco_pessoas(cad_list, df_almoco_local)
            if dfal is None or dfal.empty:
                fig_mini_movel = fig_msg("Sem dados de almoço para as pessoas selecionadas")
            else:
                dfal['SemIni'] = _semana_inicio(dfal['Data'])
                dfal = dfal.dropna(subset=['SemIni'])
                sems = pd.Index(semanas)

                # mapa semana -> {cad: {data: set(cluster_ids)}}
                mapa_sem = {}
                for sem, gsem in dfal.groupby('SemIni'):
                    m = {}
                    for cad, gg in gsem.groupby('Cadastro'):
                        m[str(cad)] = {d: cset for d, cset in zip(gg['Data'], gg['Clusters'])}
                    mapa_sem[sem] = m

                ref = cad_list[0]; outros = cad_list[1:]
                pct_medias = []
                for sem in sems:
                    m = mapa_sem.get(sem, {})
                    datas_ref = m.get(str(ref), {})
                    pcts = []
                    for cad in outros:
                        dias_outro = m.get(str(cad), {})
                        denom = len(dias_outro)
                        if denom > 0:
                            num = 0
                            for d, cset in dias_outro.items():
                                if d in datas_ref and len(cset & datas_ref[d]) > 0:
                                    num += 1
                            pcts.append(100.0 * num / denom)
                    pct_medias.append(float(np.nanmean(pcts)) if pcts else float('nan'))

                serie_pct = pd.Series(pct_medias, index=sems).astype(float)
                mm3_pct = serie_pct.rolling(window=3, min_periods=1).mean().round(2)

                fig_mini_movel = go.Figure()
                fig_mini_movel.add_trace(go.Scatter(
                    x=sems, y=list(mm3_pct.clip(lower=0, upper=100).values),
                    mode='lines+markers', line_shape='spline', line=dict(width=2),
                    marker=dict(size=6),
                    name='Média móvel (3) — % encontros no almoço',
                    hovertemplate="Semana: %{x|%Y-%m-%d}<br>Média móvel (3): %{y:.1f}%<extra></extra>"
                ))
                # Linha de referência (50%)
                fig_mini_movel.add_shape(
                    type='line', xref='paper', x0=0, x1=1,
                    yref='y', y0=50, y1=50,
                    line=dict(dash='dot', width=1, color='rgba(0,0,0,0.35)')
                )
                fig_mini_movel.update_layout(
                    template="plotly_white",
                    title="Encontros no almoço — média móvel (3 semanas)",
                    xaxis_title="", yaxis_title="Percentual de coincidência",
                    yaxis=dict(ticksuffix="%", range=[0, 100]),
                    showlegend=False, hovermode='x unified',
                    margin=dict(t=110, r=20, b=60, l=60)
                )
    else:
        # Mini GERAL — CORRIGIDO (agora com o append que faltava)
        ref = cad_list[0]; outros = cad_list[1:]

        def mapa_datas_por_semana(df):
            out = {}
            for sem, g in df.groupby('SemIni'):
                g = g[['Cadastro','Data']].dropna()
                out[sem] = {cad: set(gc['Data'].tolist()) for cad, gc in g.groupby('Cadastro')}
            return out

        mapa = mapa_datas_por_semana(dfp)
        semanas_mapa = sorted(mapa.keys())
        pct_medias = []
        for sem in semanas_mapa:
            d = mapa.get(sem, {})
            datas_ref = d.get(ref, set())
            pcts = []
            for cad in outros:
                datas_outro = d.get(cad, set())
                denom = len(datas_outro)
                if denom > 0:
                    num = len(datas_ref & datas_outro)
                    pcts.append(100.0 * num / denom)
            # >>> ESTA LINHA FALTAVA <<<
            pct_medias.append(float(np.nanmean(pcts)) if pcts else float('nan'))

        serie_pct = pd.Series(pct_medias, index=semanas_mapa).astype(float).round(2)
        mm3_pct = serie_pct.rolling(window=3, min_periods=1).mean().round(2)
        mm3_pct = mm3_pct.reindex(semanas)

        fig_mini_movel = go.Figure()
        fig_mini_movel.add_trace(go.Scatter(
            x=semanas, y=list(mm3_pct.clip(lower=0, upper=100).values),
            mode='lines+markers', line_shape='spline', line=dict(width=2),
            marker=dict(size=6),
            name='Média móvel (3) — % encontros vs referência',
            hovertemplate="Semana: %{x|%Y-%m-%d}<br>Média móvel (3): %{y:.1f}%<extra></extra>"
        ))
        # Linha de referência (50%)
        fig_mini_movel.add_shape(
            type='line', xref='paper', x0=0, x1=1,
            yref='y', y0=50, y1=50,
            line=dict(dash='dot', width=1, color='rgba(0,0,0,0.35)')
        )
        fig_mini_movel.update_layout(
            template="plotly_white",
            title="Relacionamento com a referência — média móvel (3 semanas)",
            xaxis_title="", yaxis_title="Percentual de coincidência",
            yaxis=dict(ticksuffix="%", range=[0, 100]),
            showlegend=False, hovermode='x unified',
            margin=dict(t=110, r=20, b=60, l=60)
        )

    # =================== final comum (calendário + tags) ===================
    if len(semanas):
        rng = [semanas[0], semanas[-1]]
        fig_pres.update_xaxes(type='date', range=rng, showgrid=True, gridcolor='rgba(0,0,0,0.05)')
        fig_mini_movel.update_xaxes(type='date', range=rng, showgrid=True, gridcolor='rgba(0,0,0,0.05)')

    fig_pres = add_calendar_stack(
        fig_pres, semanas,
        week_font_size=8, month_font_size=12, year_font_size=14,
        month_as_number=True, month_y=-0.18, year_y=-0.30,
        bottom_margin=100, yshift_mes=-8, yshift_ano=-15
    )
    fig_mini_movel = add_calendar_stack(
        fig_mini_movel, semanas,
        week_font_size=8, month_font_size=10, year_font_size=12,
        month_as_number=True, month_y=-0.28, year_y=-0.42,
        bottom_margin=90, yshift_mes=-18, yshift_ano=-25
    )

    fig_pres       = aplicar_tags(fig_pres, semanas, tags, dash='dot', width=4)
    fig_mini_movel = aplicar_tags(fig_mini_movel, semanas, tags, dash='dot', width=4)

    return fig_pres, fig_mini_movel


@app.callback(
    Output('dropdown-area-gptw', 'options'),
    Output('dropdown-area-gptw', 'value'),
    Input('dropdown-area-gptw', 'value'),
    Input('url', 'pathname'),
    Input('rel-load-interval', 'n_intervals'),
    prevent_initial_call=True
)
def atualizar_opcoes_e_valor_areas(selecionadas, pathname, _n_intervals):
    if not pathname or '/relacionamento' not in pathname:
        raise PreventUpdate

    # --- base de áreas (normalizada) ---
    if 'Área' in df_ativos.columns:
        base = df_ativos['Área']
    else:
        base = df_semana_merged['Área']

    todas = sorted({str(a).strip() for a in base.dropna().unique().tolist()})

    # --- seleção atual (normalizada) ---
    sel = [str(a).strip() for a in (selecionadas or [])]

    # --- herança normalizada (se existir) ---
    her_df = globals().get('her', pd.DataFrame())
    tem_her = (
        isinstance(her_df, pd.DataFrame)
        and not her_df.empty
        and {'Área', 'Heranca'}.issubset(her_df.columns)
    )

    if tem_her:
        h = her_df[['Área', 'Heranca']].copy()
        h['Área'] = h['Área'].astype(str).str.strip()
        h['Heranca'] = h['Heranca'].astype(str).str.strip()

        # mapa mãe -> filhos diretos
        filhos_map = {}
        for mae in todas:
            filhos_map[mae] = set()
        for _, row in h.iterrows():
            mae = row['Heranca']
            fil = row['Área']
            if mae in filhos_map:
                filhos_map[mae].add(fil)

        def descendentes(mae: str) -> set:
            """Todos os descendentes via herança + fallback por prefixo."""
            vistos = set()
            dq = deque(filhos_map.get(mae, set()))
            while dq:
                c = dq.popleft()
                if c in vistos:
                    continue
                vistos.add(c)
                dq.extend(filhos_map.get(c, set()))
            # une com fallback por prefixo (garantia extra)
            vistos |= {a for a in todas if a != mae and a.startswith(mae)}
            return vistos
    else:
        def descendentes(mae: str) -> set:
            return {a for a in todas if a != mae and a.startswith(mae)}

    # --- calcula o que ocultar (união de descendentes de cada mãe selecionada) ---
    ocultar = set()
    for mae in sel:
        ocultar |= descendentes(mae)

    # --- options visíveis (sem subáreas) ---
    visiveis = [a for a in todas if a not in ocultar]
    options = [{'label': a, 'value': a} for a in visiveis]

    # --- value filtrado (remove qualquer seleção que saiu das options) ---
    novo_valor = [v for v in sel if v in visiveis]

    return options, novo_valor


@app.callback(
    Output('url', 'pathname', allow_duplicate=True),
    Input('botao-voltar-relacionamento', 'n_clicks'),
    prevent_initial_call=True
)
def voltar_home(n_clicks):
    if n_clicks:
        return '/'
    raise PreventUpdate


@app.callback(
    Output('modo-rel', 'data'),
    Input('btn-modo-rel', 'n_clicks'),
    State('modo-rel', 'data'),
    prevent_initial_call=True
)
def _toggle_modo(n, modo_atual):
    modo_atual = (modo_atual or 'geral').lower()
    return 'almoco' if modo_atual == 'geral' else 'geral'


@app.callback(
    Output('btn-modo-rel', 'children'),
    Output('rel-obs-almoco', 'style'),
    Output('rel-obs-almoco', 'open'),
    Input('modo-rel', 'data')
)
def _render_ui_modo(modo):
    base_style = {
        'marginTop': '8px',
        'background': '#f8f9fa',
        'padding': '10px 12px',
        'borderRadius': '6px',
        'border': '1px solid #e0e0e0'
    }
    modo = (str(modo or 'geral')).lower()
    if modo == 'almoco':
        # Mostra a observação, mas deixa FECHADA (open=False)
        return "↩ Ver Geral", {**base_style, 'display': 'block'}, False
    else:
        # Oculta completamente no modo Geral
        return "🍽️ Almoço", {**base_style, 'display': 'none'}, False


@app.callback(
    Output('conteudo-relacionamento', 'children'),
    Output('conteudo-relacionamento', 'style'),
    Output('pagina-geral', 'style', allow_duplicate=True),
    Input('url', 'pathname'),
    prevent_initial_call=True
)
def exibir_relacionamento(pathname):
    if pathname and '/relacionamento' in pathname:
        conteudo = html.Div([
            # Sequenciador de loading da aba Relacionamento
            dcc.Store(id='rel-load-step', data=0),
            dcc.Store(id='modo-rel', data='geral'),  # << modo inicial
            dcc.Interval(id='rel-load-interval', interval=500, n_intervals=0, disabled=False, max_intervals=2),

            # Cabeçalho
            html.Div([
                html.Button(" ↩ ", id='botao-voltar-relacionamento', n_clicks=0,
                            style={'color':'Black','fontFamily':'VW Text','fontWeight':'bold','margin':'0',
                                   'display':'inline-block','padding':'5px 10px','fontSize':'20px',
                                   'backgroundColor':'#e0e0e0','border':'none','borderRadius':'5px',
                                   'cursor':'pointer','boxShadow':'0px 2px 6px rgba(0,0,0,0.25)'}),
                html.H2("Relacionamento entre Colaboradores",
                        style={'margin':'0','fontFamily':'VW Text','fontWeight':'bold',
                               'textAlign':'center','flex':'1'}),
                html.Button("🖨️ Imprimir", id='btn-print-relacionamento', n_clicks=0,
                            style={'backgroundColor':'transparent','border':'none','color':'Black',
                                   'fontFamily':'VW Text','fontWeight':'bold','cursor':'pointer',
                                   'fontSize':'16px','padding':'0 16px','textDecoration':'none'})
            ], style={'display':'flex','alignItems':'center','justifyContent':'space-between',
                      'margin':'10px 10px 10px 10px','gap':'14px','width':'100%'}),

            # Div escondida para impressão
            html.Div(id='dummy-print-relacionamento', style={'display': 'none'}),

            # Seletor (pessoas/áreas) + gráficos
            html.Div([
                html.Div([
                    html.Label("Selecionar pessoas"),
                    dcc.Dropdown(id='dropdown-pessoas-rel', options=[], value=[], multi=True,
                                 placeholder='Busque pessoas…', style={'width': '300px'}),

                    html.Div(id='cores-pessoas-container', style={'marginTop': '8px'}),

                    html.Div([
                        html.Hr(),
                        html.Label("Adicionar evento:"),
                        html.Br(),
                        dcc.Input(id='tag-nome', type='text', placeholder='Nome do evento',
                                  style={'width':'70%','marginBottom':'6px'}),
                        dcc.Input(id='tag-cor', type='color', value="#3221cc",
                                  style={'width':'70px','marginLeft':'8px','verticalAlign':'middle'}),
                        html.Br(),
                        html.Div([
                            dcc.DatePickerSingle(
                                id='tag-data', display_format='DD/MM/YYYY',
                                placeholder='Data', clearable=True, date=None,
                                className='dp-compact', style={'width':'120px'}
                            ),
                            html.Button("Adicionar evento", id='tag-add', n_clicks=0,
                                        style={'marginLeft':'45px','height':'36px','padding':'0 10px',
                                               'whiteSpace':'nowrap'}),
                        ], style={'display':'flex','alignItems':'center','gap':'8px','flexWrap':'wrap'}),

                        dash_table.DataTable(
                            id='tag-tabela',
                            columns=[
                                {'name':'Nome','id':'nome_md','editable':False},
                                {'name':'Data','id':'data_br','editable':True},
                                {'name':'Cor','id':'cor','presentation':'markdown','editable':False},
                                {'name':'','id':'del','presentation':'markdown','editable':False},
                            ],
                            data=[],
                            markdown_options={'html': True},
                            row_deletable=False,
                            style_table={'marginTop':'8px','minWidth':'280px'},
                            style_cell={'fontFamily':'VW Text','fontSize':'14px','padding':'6px'},
                            style_header={'backgroundColor':'#f0f0f0','fontWeight':'bold','textAlign':'center'},
                            style_data={'whiteSpace':'nowrap'},
                            style_cell_conditional=[
                                {'if': {'column_id':'nome_md'}, 'textAlign':'left','width':'60%'},
                                {'if': {'column_id':'data_br'}, 'textAlign':'center','width':'20%'},
                                {'if': {'column_id':'cor'}, 'textAlign':'center','width':'15%'},
                                {'if': {'column_id':'del'}, 'textAlign':'center','width':'5%'},
                            ],
                            style_data_conditional=[
                                {'if': {'column_id': 'del'}, 'cursor': 'pointer', 'userSelect': 'none'},
                            ],
                        ),
                        dcc.Store(id='rel-tags', data=[]),
                    ], style={'marginTop':'10px'})
                ], style={'flex':'0 0 300px'}),

                html.Div([
                    dcc.Loading(
                        id='loading-rel-main',
                        children=dcc.Graph(id='grafico-relacionamento-linhas',
                                           style={'height':'420px','marginBottom':'16px'})
                    ),
                    # Botão de modo (Geral ↔ Almoço)
                    html.Div([
                        html.Button(
                            "🍽️ Almoço", id='btn-modo-rel', n_clicks=0,
                            style={'fontFamily':'VW Text','fontWeight':'bold','padding':'6px 10px',
                                   'cursor':'pointer','borderRadius':'6px','border':'1px solid #bbb',
                                   'background':'#fff','marginTop':'8px'}
                        )
                    ], style={'margin': '6px 0 8px 0', 'display':'flex', 'justifyContent':'flex-end'}),
                    dcc.Loading(
                        id='loading-rel-mini',
                        children=dcc.Graph(id='grafico-mini-media-movel',
                                           style={'height':'300px','flex':'1','minWidth':'260px'})
                    ),
                    # Observação (almoço) — começa oculta; aparece no modo almoço
                    html.Details(
                        id='rel-obs-almoco',
                        open=False,
                        children=[
                            html.Summary("O que esta página mostra?"),
                            html.Div([
                                html.P(
                                    "Entradas no refeitório e saídas da planta durante o horário de almoço "
                                    "(11:00–14:00). Para agrupar eventos próximos no mesmo encontro, "
                                    "consideramos diferenças de até 1 minuto no refeitório e 3 minutos nas "
                                    "saídas da fábrica."
                                ),
                                html.P(
                                    "O gráfico superior mostra, por pessoa, o número de dias com presença por "
                                    "semana (idêntico ao modo Geral). O gráfico inferior mostra a média móvel do "
                                    "percentual de encontros de ALMOÇO em relação à pessoa de referência "
                                    "(a primeira selecionada)."
                                )
                            ])
                        ],
                        style={
                            'display': 'none',
                            'marginTop': '8px',
                            'background': '#f8f9fa',
                            'padding': '10px 12px',
                            'borderRadius': '6px',
                            'border': '1px solid #e0e0e0'
                        }
                    ),
                ], style={'flex':'1','minWidth':'360px','paddingLeft':'24px'})
            ], style={
                'display':'flex','gap':'24px','alignItems':'flex-start','padding':'16px',
                'boxSizing':'border-box','backgroundColor':"#e0e0e0",'borderRadius':'8px',
                'boxShadow':'0px 3px 6px rgba(0,0,0,0.1)','flexWrap':'wrap'
            }),

            # GPTW
            html.Div([
                html.Div([
                    dcc.Dropdown(id='dropdown-area-gptw', options=[], value=[], multi=True,
                                 placeholder='Selecione áreas…',
                                 style={'minWidth':'260px','maxWidth':'520px','width':'100%','margin':'0'}),
                ], style={'marginBottom':'10px','display':'flex'}),
                html.Div(id='tabela-gptw-rel', style={'display':'flex','justifyContent':'center'})
            ], style={'padding':'16px','background':'#fff','borderRadius':'12px','margin':'10px'})
        ], style={'padding': '10px', 'backgroundColor': 'white'})

        return conteudo, {'display':'block'}, {'display':'none'}

    # Outras páginas
    return dash.no_update, {'display':'none'}, {
        'display':'flex','flexDirection':'column','width':'100%','height':'100%',
        'backgroundColor':"#ffffff",'padding':'16px','boxSizing':'border-box'
    }


if __name__ == '__main__':
    app.run(port=8060, debug=False)

